# 06 · Implantação

**CRISP-DM fase 6 — implantação** · Meridiano Imobiliária

---

O notebook [`05-avaliacao`](05-avaliacao.ipynb) entregou um veredito — **prosseguir para a
implantação** — e uma condição: três divergências de gravidade alta bloqueiam a publicação.
Entregou também um backlog de oito itens, cada um com o critério que o fecha, e um achado que
esta fase precisa absorver no desenho da operação: **178 imóveis estão em silhueta negativa**,
um em cada quatro do segmento do meio.

Esta fase responde à pergunta que sobra depois que o modelo já está aprovado: **o que precisa
existir para que a segmentação funcione amanhã, sem que ninguém rode um notebook.**

| Bloco | Tarefa do CRISP-DM | Pergunta | O que decide |
|:--|:---|:---|:---|
| 1 | *Plan deployment* | Como o modelo entra em operação e como o resultado chega a quem decide? | O pacote publicável, o contrato de entrada, a regra de atribuição e o plano de entrada em produção |
| 2 | *Plan monitoring and maintenance* | Como se percebe que o modelo envelheceu, se não existe gabarito? | O desvio medido em dado real, os gatilhos com limiar numérico e o responsável de cada um |
| 3 | *Produce final report* | O que a diretoria leva, e o que fica escrito? | O relatório final, o roteiro da apresentação e os insights com a evidência de cada um |
| 4 | *Review project* | O que este projeto ensina para o próximo? | A retrospectiva, com o que sustentou e o que não sustentou |

## O que "implantar" significa aqui — e o que não significa

Não significa publicar um arquivo. O modelo está implantado quando existem cinco coisas, e
nenhuma delas é o `.joblib`:

1. **um pacote com carimbo** — o artefato aprovado, identificado por versão e resumo
   criptográfico, de modo que ninguém consiga executar em produção um objeto diferente do que
   a fase 5 avaliou;
2. **um contrato de entrada** — o que a aplicação aceita, com faixa e regra de rejeição, para
   que o erro apareça na porta e não no meio do relatório da diretoria;
3. **uma regra de atribuição que admite não saber** — o imóvel de fronteira sai marcado, e a
   ação automática não dispara sobre ele;
4. **um plano de monitoramento com gatilho numérico e dono** — "acompanhar de perto" não é
   plano, é intenção;
5. **um caminho de volta** — o que se faz quando o gatilho dispara, quem decide e em quanto
   tempo.

Os quatro blocos deste notebook produzem os cinco, nessa ordem.

## Por que a fase começa pagando dívida

O bloco 0 confere o que o repositório publica hoje, antes de desenhar qualquer arquitetura. O
resultado dessa conferência é o que organiza o resto:

> **O artefato que a aplicação carrega hoje não é o modelo que a fase 5 aprovou.**

`models/pipeline_segmentacao.joblib` tem *k* = 4, foi ajustado sobre 2.925 imóveis e 21
variáveis. O modelo aprovado tem *k* = 3, 2.350 imóveis e 17 variáveis. Implantar sem tratar
isso seria publicar, no primeiro dia de operação, um resultado que nenhuma fase deste projeto
mediu — que é exatamente o erro que a fase 5 chamou de condição de publicação.

In [ ]:
# Importações
import hashlib
import json
import pickle
import pickletools
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import load
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.data import dicionario as dic
from src.eda.estatistica import psi
from src.model import particional as part
from src.model import perfilagem as perf
from src.model import personas as pers
from src.utils import io

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

io.configurar_estilo()

# As figuras vão para `docs/imagens/implantacao/`; as tabelas, para
# `reports/tables/implantacao/`. A regra do projeto continua valendo: nenhuma
# figura existe sem a tabela que a produziu.
FIGURAS = config.DOCS / "imagens" / "implantacao"
FIGURAS.mkdir(parents=True, exist_ok=True)
SECAO = "implantacao"

BARRA = config.AZUL_COBALTO
REFERENCIA = config.AMARELO
DESTAQUE = config.AZUL_CLARO
ALERTA = config.CORAL
NEUTRO = config.CINZA
APROVADO = config.VERDE


def salvar(fig, nome, dados=None):
    """Grava a figura em `docs/imagens/implantacao/` e a tabela que a originou."""
    fig.savefig(FIGURAS / f"{nome}.png", bbox_inches="tight")
    plt.close(fig)
    if dados is not None:
        io.salvar_tabela(dados, SECAO, nome, indice=False)


def ler_tabela(secao: str, nome: str) -> pd.DataFrame:
    """Lê uma tabela gravada por uma fase anterior, no formato do projeto.

    Nenhuma tabela de fase anterior é redigitada neste notebook. O que a fase 5
    mediu entra aqui pelo arquivo — se o número mudar lá, muda aqui.
    """
    caminho = config.TABLES / secao / f"{nome}.csv"
    return pd.read_csv(caminho, sep=";", decimal=",", encoding="utf-8-sig")


print(f"Projeto em {RAIZ}")
print(f"Semente do projeto: {config.SEMENTE}")

## O registro de compromissos

A fase 4 registrou *decisões*. A fase 5 registrou *julgamentos*. Esta registra **compromissos**,
e a diferença importa: um compromisso tem **dono**, **critério de conclusão** e **endereço** —
o arquivo em que ele se resolve. Sem os três, o que fica escrito é intenção — e intenção não
sobrevive ao primeiro trimestre de operação.

As duas primeiras colunas vêm da fase 5: `criterio_de_conclusao` é o mesmo nome que
`proximas-etapas.csv` usa, e `onde` é a mesma coluna do backlog. Reaproveitar os nomes não é
preciosismo — é o que permite conferir, item a item, se o compromisso desta fase fecha a tarefa
que a fase 5 escreveu, e não uma tarefa parecida.

| `acao` | Significado |
|:---|:---|
| `divergência medida` | O que o repositório publica hoje contra o que a fase 5 aprovou, com o número da diferença |
| `dívida paga` | Um item que bloqueava a publicação, fechado neste notebook, com a evidência que o fecha |
| `artefato publicado` | Um arquivo que passa a existir para a operação, com o que ele contém |
| `parâmetro calibrado` | Um número da operação medido na carteira — não escolhido por intuição |
| `gatilho definido` | Uma condição numérica, com dono e com a ação que ela dispara |
| `plano definido` | Um procedimento com responsável, prazo e critério de aceite — sem número novo |
| `pendência transferida` | O que esta fase não fecha: com quem fica, em que arquivo, e o critério que a fechará |
| `lição registrada` | O que a retrospectiva conclui, com o número que sustenta a conclusão |

A distinção que mais importa é entre `dívida paga` e `pendência transferida`. A primeira diz
que o problema acabou aqui; a segunda diz **quem** vai terminá-lo, **onde** e **como se saberá**
que terminou. Uma fase de implantação honesta produz as duas — e a lista das segundas é o que
impede que a operação comece supondo que está tudo resolvido.

In [ ]:
COMPROMISSOS: list[dict] = []

#: O vocabulário da tabela acima, em código. Existe para que a tabela seja um
#: contrato e não uma legenda: o bloco 4.5 confere o registro contra este conjunto
#: antes de publicar o manifesto, e uma ação nova exige declarar a linha primeiro.
ACOES_DECLARADAS: set[str] = {
    "divergência medida",
    "dívida paga",
    "artefato publicado",
    "parâmetro calibrado",
    "gatilho definido",
    "plano definido",
    "pendência transferida",
    "lição registrada",
}


def pt(numero) -> str:
    """Formata o número no padrão brasileiro: ponto como separador de milhar."""
    return f"{numero:,}".replace(",", ".")


def usd(valor) -> str:
    """Formata um valor monetário da base, que está em dólares."""
    return f"US$ {pt(int(round(valor)))}"


def registrar(
    bloco: str,
    compromisso: str,
    responsavel: str,
    criterio: str,
    evidencia: str,
    acao: str = "artefato publicado",
    onde: str = "—",
):
    """Anota um compromisso da fase 6, com dono, critério de conclusão, endereço e medida.

    `criterio` e `onde` usam os nomes que a fase 5 gravou em `proximas-etapas.csv`,
    de modo que os dois registros possam ser confrontados coluna a coluna.

    `evidencia` guarda o **número** ou o **arquivo** que sustenta o compromisso.
    Plano de implantação sem evidência ao lado é apresentação, não plano.
    """
    COMPROMISSOS.append(
        {
            "n": len(COMPROMISSOS) + 1,
            "bloco": bloco,
            "compromisso": compromisso,
            "responsavel": responsavel,
            "criterio_de_conclusao": criterio,
            "onde": onde,
            "evidencia": evidencia,
            "acao": acao,
        }
    )
    print(f"{len(COMPROMISSOS):>2}. [{acao:>21}] {compromisso} — {evidencia}")

# 0 · A entrada — e a licença para implantar

A fase 6 não começa com uma decisão própria: começa lendo a decisão da fase 5. Se
`manifesto_avaliacao.json` disser que o projeto não prossegue, este notebook não tem o que
fazer — e a conferência abaixo interrompe a execução em vez de seguir por inércia.

| Arquivo | Papel |
|:---|:---|
| `data/processed/manifesto_avaliacao.json` | **A licença** — o veredito da fase 5 e o que ficou pendente |
| `models/modelo-segmentacao.joblib` | O objeto que vai para produção: receita e agrupador ajustados, com o mapa canônico |
| `data/processed/base_segmentada.parquet` | A partição avaliada — o gabarito da conferência de reprodução |
| `data/processed/base_modelagem.parquet` | O espaço de entrada, em unidades de negócio |
| `data/processed/imoveis_curados.parquet` | A carteira com `preco_venda`, `bairro` e `ano_venda` |

Uma diferença de método em relação à fase 5: lá, o espaço euclidiano foi **reconstruído** a
partir da receita crua, para auditar a cadeia. Aqui ele é obtido pelo `transform` do
pré-processador **já ajustado dentro do pacote** — porque é exatamente esse o caminho que a
aplicação vai percorrer em produção. Se os dois caminhos discordassem, a silhueta recalculada
não bateria com a do manifesto da fase 4, e a conferência abaixo falharia.

O rótulo reservado entra outra vez, e o motivo é o mesmo da fase 5, agora sem novidade: os
segmentos estão congelados, nenhum número deste notebook volta para a modelagem, e sem
`preco_venda` não existe relatório de negócio nem métrica de acompanhamento.

In [ ]:
manifesto_carga = io.ler_json(config.DATA_RAW / "manifesto_carga.json")
manifesto_curadoria = io.ler_json(config.DATA_PROCESSED / "manifesto_curadoria.json")
manifesto_preparacao = io.ler_json(config.DATA_PROCESSED / "manifesto_preparacao.json")
manifesto_modelagem = io.ler_json(config.DATA_PROCESSED / "manifesto_modelagem.json")
manifesto_avaliacao = io.ler_json(config.DATA_PROCESSED / "manifesto_avaliacao.json")

MODELO = config.MODELS / "modelo-segmentacao.joblib"
BASE_MODELAGEM = config.DATA_PROCESSED / "base_modelagem.parquet"
BASE_SEGMENTADA = config.DATA_PROCESSED / "base_segmentada.parquet"

# --------------------------------------------------------------------------- #
# A licença: a fase 6 só existe se a fase 5 tiver autorizado.
# --------------------------------------------------------------------------- #
PROSSEGUE = bool(manifesto_avaliacao["prossegue_para_implantacao"])
VEREDITO_DA_FASE_5 = manifesto_avaliacao["veredito"]
BLOQUEIAM_A_PUBLICACAO = int(manifesto_avaliacao["itens_que_bloqueiam_a_publicacao"])
assert PROSSEGUE, "a fase 5 não autorizou a implantação — este notebook não deve rodar"

# O que a fase 5 mediu entra pelo manifesto. Nada disso é recalculado aqui: o achado
# que organiza o bloco 1.4 é dela, e remedi-lo permitiria que este notebook chegasse
# a um número próprio e continuasse citando a fase 5 como fonte.
AVALIACAO_DE_NEGOCIO = manifesto_avaliacao["avaliacao_de_negocio"]
IMOVEIS_EM_SILHUETA_NEGATIVA = int(AVALIACAO_DE_NEGOCIO["imoveis_em_silhueta_negativa"])
SEGMENTO_DE_FRONTEIRA = int(AVALIACAO_DE_NEGOCIO["segmento_de_fronteira"])
OCORRENCIAS_DESATUALIZADAS = int(
    manifesto_avaliacao["revisao_do_processo"]["ocorrencias_documentais_desatualizadas"]
)

# O backlog da fase 5 entra aqui, e não no bloco 4: os itens 1, 4, 5 e 8 são tarefas
# **deste** notebook, e cada um deles precisa citar o critério que a fase 5 escreveu —
# verbatim, do arquivo. Parafrasear o critério na hora de fechá-lo é a forma silenciosa
# de trocar a régua depois de ver o resultado.
backlog_da_fase_5 = ler_tabela("avaliacao", "proximas-etapas").set_index("ordem")


def tarefa_da_fase_5(ordem: int) -> pd.Series:
    """A linha do backlog da fase 5 — critério de conclusão e endereço, sem redigitar."""
    return backlog_da_fase_5.loc[ordem]

base = pd.read_parquet(BASE_MODELAGEM)
segmentada = pd.read_parquet(BASE_SEGMENTADA)
curada = pd.read_parquet(config.BASE_CURADA).set_index("pid").loc[base.pid].reset_index()
pacote = load(MODELO)

ESPACO = list(pacote["espaco"])
K = int(pacote["k"])
ALGORITMO = str(pacote["algoritmo"])
SEMENTE = int(pacote["semente"])
LINHAS = len(base)
ROTULOS = segmentada.segmento.to_numpy()
MAPA_CANONICO = {int(cru): int(canonico) for cru, canonico in pacote["mapa_canonico"].items()}

assert (curada.pid.to_numpy() == base.pid.to_numpy()).all(), "o join por pid saiu de ordem"
assert len(curada) == LINHAS, "o join por pid perdeu ou duplicou imóvel"
assert K == manifesto_avaliacao["modelo_avaliado"]["k"], "o pacote não é o modelo avaliado"

# Conferência, não recálculo: a carteira em mãos precisa concordar com o que a fase 5
# mediu. Discordar aqui significa que o arquivo e a medida se separaram entre as fases.
PCT_NEGATIVO_NA_CARTEIRA = 100 * IMOVEIS_EM_SILHUETA_NEGATIVA / LINHAS
PERCENTUAL_NEGATIVO_NO_SEGMENTO = 100 * float(
    (segmentada.loc[segmentada.segmento == SEGMENTO_DE_FRONTEIRA, "silhueta"] < 0).mean()
)
assert IMOVEIS_EM_SILHUETA_NEGATIVA == int((segmentada.silhueta < 0).sum()), (
    "a base segmentada não reproduz a contagem de silhueta negativa da fase 5"
)

# O caminho de produção: `transform` do pré-processador ajustado dentro do pacote.
preparo = pacote["pipeline"].named_steps["preparo"]
agrupador = pacote["pipeline"].named_steps["segmentacao"]
EUCLIDIANO = preparo.transform(base[ESPACO])
CENTROIDES = agrupador.cluster_centers_

crus = pacote["pipeline"].predict(base[ESPACO])
reproduzidos = np.array([MAPA_CANONICO[int(cru)] for cru in crus])
REPRODUZ = bool((reproduzidos == ROTULOS).all())
SILHUETA = round(float(silhouette_score(EUCLIDIANO, ROTULOS)), 4)

assert REPRODUZ, "o pacote não reproduz a partição aprovada — não pode ser implantado"
assert abs(SILHUETA - manifesto_modelagem["silhueta"]) < 1e-4, "o espaço de produção divergiu"

prontidao = pd.Series(
    {
        "veredito da fase 5": VEREDITO_DA_FASE_5,
        "modelo aprovado": f"{ALGORITMO}, k = {K}, silhueta {SILHUETA:.4f}",
        "o pacote reproduz a partição avaliada": "sim — 100% dos imóveis" if REPRODUZ else "NÃO",
        "carteira": f"{pt(LINHAS)} imóveis, {len(ESPACO)} variáveis de entrada",
        "itens que bloqueiam a publicação": f"{BLOQUEIAM_A_PUBLICACAO} — fase 5, bloco 4.2",
        "imóveis em silhueta negativa (fase 5)": f"{pt(IMOVEIS_EM_SILHUETA_NEGATIVA)} "
                                                 f"({PCT_NEGATIVO_NA_CARTEIRA:.1f}% da carteira)",
        "segmento de fronteira (fase 5)": f"{SEGMENTO_DE_FRONTEIRA} — "
                                          f"{PERCENTUAL_NEGATIVO_NO_SEGMENTO:.1f}% das suas "
                                          f"unidades em silhueta negativa",
    },
    name="a licença para implantar",
)
print(f"Licença da fase 5: {'CONCEDIDA' if PROSSEGUE else 'NEGADA'}\n")
prontidao.to_frame()

## 0.1 · O que o repositório publica hoje

Antes de desenhar a integração, uma pergunta que raramente se faz e que decide o resto: **se
alguém abrir a aplicação agora, qual modelo responde?**

A resposta está em três arquivos que a camada de aplicação carrega — e nenhum deles foi
produzido pela cadeia 00–05:

| Artefato publicado | O que ele é | Quem o consome |
|:---|:---|:---|
| `models/pipeline_segmentacao.joblib` | Pipeline com *k* = 4, ajustado sobre 2.925 imóveis e 21 variáveis | `src/deployment/scoring.py` |
| `models/metadados_segmentacao.json` | Versão 1.0.0, com o catálogo de 4 personas | `src/deployment/scoring.py` |
| `models/linha_de_base.json` | Distribuições de referência da carteira anterior | `src/deployment/monitoramento.py` |
| `src/model/personas.py` | Quatro personas da cadeia anterior, com os códigos S1 a S4 | `src/deployment/scoring.py` e a aplicação |

A conferência abaixo faz três coisas: identifica os artefatos, **mede** a diferença entre a
partição publicada e a aprovada sobre a mesma carteira, e conta quantos imóveis receberiam
uma **ação comercial diferente** por causa dela. A última é a única medida que a diretoria
precisa ver — as outras duas explicam por quê.

In [ ]:
ARTEFATO_LEGADO = config.MODELS / "pipeline_segmentacao.joblib"
METADADOS_LEGADO = config.MODELS / "metadados_segmentacao.json"

legado = load(ARTEFATO_LEGADO)
metadados_legado = io.ler_json(METADADOS_LEGADO)
K_LEGADO = int(metadados_legado["k"])
ESPACO_LEGADO = list(metadados_legado["espaco_de_atributos"])
TREINO_LEGADO = int(metadados_legado["registros_treino"])

# A partição publicada, aplicada à carteira que a fase 5 aprovou.
traducao_legado = {
    int(cru): int(canonico)
    for cru, canonico in metadados_legado["traducao_rotulo_bruto_para_canonico"].items()
}
rotulos_legado = np.array(
    [traducao_legado[int(cru)] for cru in legado.predict(curada[ESPACO_LEGADO])]
)
ARI_PUBLICADO_APROVADO = round(float(adjusted_rand_score(rotulos_legado, ROTULOS)), 4)

# Quantos imóveis mudam de linha de produto: para cada segmento aprovado, qual é o
# segmento publicado majoritário — e quantos imóveis caem fora dele.
contingencia = pd.crosstab(
    pd.Series(ROTULOS, name="segmento aprovado (k = 3)"),
    pd.Series(rotulos_legado, name="segmento publicado (k = 4)"),
)
correspondencia = contingencia.idxmax(axis=1)
CONCORDAM = int(sum(contingencia.loc[s, correspondencia[s]] for s in contingencia.index))
MUDAM_DE_ACAO = LINHAS - CONCORDAM

# Os nomes dos passos do pipeline também mudaram entre as cadeias: o módulo de
# scoring procura `preparacao` e `kmeans`, e o pacote aprovado tem `preparo` e
# `segmentacao`. É falha de carga, não de resultado — quebra na primeira chamada.
PASSOS_APROVADO = set(pacote["pipeline"].named_steps)
PASSOS_LEGADO = set(legado.named_steps)

# O risco menos visível da cadeia anterior não é o número de personas: é o código.
# `src/model/personas.py` já usa S1, S2 e S3 — para outros imóveis, com outras ações.
# Quem ler "S2" numa tela não tem como saber de qual cadeia veio.
CODIGOS_LEGADOS = [persona["codigo"] for persona in pers.PERSONAS.values()]

auditoria_do_publicado = pd.DataFrame(
    [
        {
            "artefato": "models/pipeline_segmentacao.joblib",
            "o_que_publica": f"k = {K_LEGADO}, {pt(TREINO_LEGADO)} imóveis, "
                             f"{len(ESPACO_LEGADO)} variáveis",
            "o_que_a_fase_5_aprovou": f"k = {K}, {pt(LINHAS)} imóveis, {len(ESPACO)} variáveis",
            "diferenca_medida": f"ARI {ARI_PUBLICADO_APROVADO:.4f} entre as duas partições",
            "gravidade": "alta",
        },
        {
            "artefato": "models/metadados_segmentacao.json",
            "o_que_publica": f"versão {metadados_legado['versao_modelo']}, "
                             f"{len(metadados_legado['personas'])} personas",
            "o_que_a_fase_5_aprovou": f"{K} segmentos, catálogo ainda inexistente",
            "diferenca_medida": f"{len(metadados_legado['personas']) - K} persona a mais "
                                f"do que o modelo tem segmentos",
            "gravidade": "alta",
        },
        {
            "artefato": "src/model/personas.py",
            "o_que_publica": f"os códigos {', '.join(CODIGOS_LEGADOS)} com "
                             f"{len(pers.PERSONAS)} nomes da cadeia anterior",
            "o_que_a_fase_5_aprovou": f"{K} linhas de produto, ainda sem código publicado",
            "diferenca_medida": f"os códigos {', '.join(sorted(CODIGOS_LEGADOS)[:K])} vão "
                                f"existir nas duas cadeias com significados diferentes",
            "gravidade": "alta",
        },
        {
            "artefato": "models/linha_de_base.json",
            "o_que_publica": f"distribuições de {pt(TREINO_LEGADO)} imóveis",
            "o_que_a_fase_5_aprovou": f"carteira curada de {pt(LINHAS)} imóveis",
            "diferenca_medida": f"{pt(TREINO_LEGADO - LINHAS)} imóveis de diferença na "
                                f"referência de monitoramento",
            "gravidade": "alta",
        },
        {
            "artefato": "src/deployment/scoring.py",
            "o_que_publica": f"procura os passos {sorted(PASSOS_LEGADO)}",
            "o_que_a_fase_5_aprovou": f"o pacote expõe {sorted(PASSOS_APROVADO)}",
            "diferenca_medida": "erro de chave na primeira chamada, não resultado errado",
            "gravidade": "média",
        },
    ]
)

registrar(
    "0 · Conferência",
    "O artefato publicado não é o modelo aprovado",
    "Área de Dados",
    "a aplicação carrega o pacote de k = 3 e o carimbo confere",
    f"ARI {ARI_PUBLICADO_APROVADO:.4f} · {pt(MUDAM_DE_ACAO)} imóveis "
    f"({100 * MUDAM_DE_ACAO / LINHAS:.1f}%) receberiam ação comercial diferente",
    acao="divergência medida",
)

io.salvar_tabela(auditoria_do_publicado, SECAO, "auditoria-do-publicado", indice=False)
auditoria_do_publicado

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(14.5, 5.2),
                          gridspec_kw={"width_ratios": [1.25, 1]}, constrained_layout=True)

sns.heatmap(contingencia, annot=True, fmt="d", cmap="Blues", linewidths=0.7,
            linecolor="white", cbar=False, ax=eixos[0])
eixos[0].set_title("Onde cada imóvel cai nas duas partições")
eixos[0].set_xlabel("Segmento publicado hoje (k = 4)")
eixos[0].set_ylabel("Segmento aprovado pela fase 5 (k = 3)")

barras = eixos[1].bar(
    ["mesma linha\nde produto", "linha de produto\ndiferente"],
    [CONCORDAM, MUDAM_DE_ACAO],
    color=[APROVADO, ALERTA], width=0.62,
)
eixos[1].bar_label(
    barras,
    labels=[f"{pt(CONCORDAM)}\n{100 * CONCORDAM / LINHAS:.1f}%",
            f"{pt(MUDAM_DE_ACAO)}\n{100 * MUDAM_DE_ACAO / LINHAS:.1f}%"],
    padding=4, fontsize=10, fontweight="bold",
)
eixos[1].set_ylabel("Imóveis da carteira")
eixos[1].set_ylim(0, LINHAS * 1.05)
eixos[1].set_title(f"O custo de publicar o artefato errado\nARI {ARI_PUBLICADO_APROVADO:.4f} "
                   f"entre as duas partições")

fig.suptitle("O que a aplicação carrega hoje contra o que a fase 5 aprovou",
             fontsize=13, fontweight="bold", color=config.AZUL_PROFUNDO)

salvar(fig, "publicado-contra-aprovado", contingencia.reset_index())
contingencia

### O que a conferência decidiu

Os dois números da figura sustentam a mesma frase, e ela vale a pena ser dita sem atenuação:

* **ARI 0,72** entre a partição publicada e a aprovada. As duas veem a mesma carteira e não
  são a mesma coisa: 0,72 é alto o bastante para que os gráficos pareçam parecidos numa
  reunião, e baixo o bastante para que as decisões sejam diferentes;
* **280 imóveis mudam de linha de produto** — 11,9% da carteira. Cada um deles é um imóvel que
  receberia teto de reforma, política de preço e canal errados, sem que nada na tela avisasse.

A tabela acrescenta um terceiro achado, que não aparece em número e é o mais traiçoeiro dos
três: **os códigos colidem.** `src/model/personas.py` já publica S1, S2 e S3 — para outros
imóveis, com outras ações e outros tetos de reforma. Um relatório antigo e um novo, lado a
lado na mesa da Diretoria, falam de "S2" e não são a mesma coisa; e nada na tela avisa.

A conclusão organiza o bloco 1: **a implantação não começa desenhando o futuro, começa
retirando de circulação o artefato antigo.** O que este notebook publica — pacote com carimbo,
contrato, catálogo, linha de base — existe para tornar impossível repetir esse erro na próxima
reciclagem, e não apenas para corrigi-lo desta vez.

# 1 · Planejar a implantação

*Plan deployment* — a primeira tarefa da fase 6. O CRISP-DM pede um plano de como o resultado
entra no ambiente do cliente e como ele chega a quem decide. As duas coisas são diferentes, e o
bloco trata as duas: **como o modelo executa** (1.1 a 1.7) e **como o resultado é entregue**
(1.8 e 1.9).

## 1.1 · A arquitetura de integração

A Meridiano não tem plataforma de *machine learning*, e propor uma seria vender um problema
novo junto com a solução. A integração usa o que já existe no projeto — o pacote serializado,
o pacote Python `src/` e a aplicação Streamlit — e acrescenta **um único ponto de entrada**
para a atribuição de segmento.

A regra que organiza o desenho é uma só:

> **A receita de preparação existe em um lugar só.** A padronização, o `log1p` e a ordem das
> colunas moram dentro do pacote serializado. Nenhuma camada de aplicação reimplementa nada
> disso — reimplementar a preparação na borda é a forma mais comum de um modelo funcionar no
> notebook e errar em produção, e é um erro que não avisa: ele devolve um número plausível.

Dois caminhos consomem o mesmo pacote:

* **lote**, a cada carga da base: a carteira inteira é reclassificada e o resultado vira
  arquivo — é o que alimenta relatórios, painel e orçamento por linha de produto;
* **sob demanda**, um imóvel por vez: o Comitê de Aquisição pergunta em que segmento cai um
  imóvel que ainda não é da carteira, e recebe a resposta com o grau de confiança.

O terceiro fluxo não é de operação, é de manutenção: fecha o ciclo de volta ao pacote, e é o
que o bloco 2 detalha.

In [ ]:
def caixa(ax, x, y, largura, altura, titulo, subtitulo, cor, cor_texto="white"):
    """Desenha um bloco do diagrama de integração."""
    ax.add_patch(
        FancyBboxPatch((x, y), largura, altura, boxstyle="round,pad=0.4,rounding_size=1.6",
                       linewidth=0, facecolor=cor, zorder=2)
    )
    ax.text(x + largura / 2, y + altura * 0.63, titulo, ha="center", va="center",
            color=cor_texto, fontsize=8.8, fontweight="bold", zorder=3)
    ax.text(x + largura / 2, y + altura * 0.25, subtitulo, ha="center", va="center",
            color=cor_texto, fontsize=7.2, alpha=0.93, zorder=3)


def seta(ax, origem, destino, cor=NEUTRO, curvatura=0.0, tracejada=False):
    """Liga dois pontos do diagrama."""
    ax.add_patch(
        FancyArrowPatch(origem, destino, arrowstyle="-|>", mutation_scale=12, linewidth=1.6,
                        color=cor, zorder=1, linestyle="dashed" if tracejada else "solid",
                        connectionstyle=f"arc3,rad={curvatura}")
    )


fig, ax = plt.subplots(figsize=(15, 7.4))
ax.set_xlim(-2, 104)
ax.set_ylim(18, 100)
ax.axis("off")
ax.grid(False)

L, A = 18.0, 13.0
COLUNA = [0.0, 21.0, 42.0, 63.0, 84.0]

ax.text(0, 94, "OPERAÇÃO — o fluxo que roda todo dia", fontsize=10, fontweight="bold",
        color=config.AZUL_PROFUNDO)
caixa(ax, COLUNA[0], 74, L, A, "Carga da base", "cadastro de imóveis · mensal", BARRA)
caixa(ax, COLUNA[0], 56, L, A, "Formulário e CRM", "um imóvel · sob demanda", BARRA)
caixa(ax, COLUNA[1], 65, L, A, "Contrato de entrada", "faixa, tipo e regra de rejeição",
      config.AZUL_PROFUNDO)
caixa(ax, COLUNA[2], 65, L, A, "Pacote publicado", "preparo + agrupador, com carimbo",
      config.AZUL_PROFUNDO)
caixa(ax, COLUNA[3], 65, L, A, "Atribuição", "segmento + faixa de confiança",
      config.AZUL_PROFUNDO)
caixa(ax, COLUNA[4], 74, L, A, "Carteira classificada", "parquet, CSV, painel e relatório",
      APROVADO, cor_texto=config.AZUL_PROFUNDO)
caixa(ax, COLUNA[4], 56, L, A, "Resposta ao consumidor", "segmento, ação, confiança",
      APROVADO, cor_texto=config.AZUL_PROFUNDO)

seta(ax, (L, 80.5), (COLUNA[1], 74.5), curvatura=-0.12)
seta(ax, (L, 62.5), (COLUNA[1], 69.0), curvatura=0.12)
seta(ax, (COLUNA[1] + L, 71.5), (COLUNA[2], 71.5))
seta(ax, (COLUNA[2] + L, 71.5), (COLUNA[3], 71.5))
seta(ax, (COLUNA[3] + L, 74.5), (COLUNA[4], 80.5), curvatura=0.12)
seta(ax, (COLUNA[3] + L, 69.0), (COLUNA[4], 62.5), curvatura=-0.12)

ax.text(0, 46, "MANUTENÇÃO — o fluxo que fecha o ciclo", fontsize=10, fontweight="bold",
        color=config.AZUL_PROFUNDO)
caixa(ax, COLUNA[4], 26, L, A, "Linha de base", "decis e medianas do treino", DESTAQUE,
      cor_texto=config.AZUL_PROFUNDO)
caixa(ax, COLUNA[3], 26, L, A, "Medição do desvio", "PSI · silhueta · composição", DESTAQUE,
      cor_texto=config.AZUL_PROFUNDO)
caixa(ax, COLUNA[2], 26, L, A, "Gatilho", "limiar numérico, com dono", REFERENCIA,
      cor_texto=config.AZUL_PROFUNDO)
caixa(ax, COLUNA[1], 26, L, A, "Reciclagem", "reajuste + ARI de aceite", ALERTA)

seta(ax, (COLUNA[3] + L / 2, 65.0), (COLUNA[3] + L / 2, 39.0), cor=DESTAQUE, tracejada=True)
seta(ax, (COLUNA[4], 32.5), (COLUNA[3] + L, 32.5), cor=NEUTRO)
seta(ax, (COLUNA[3], 32.5), (COLUNA[2] + L, 32.5), cor=NEUTRO)
seta(ax, (COLUNA[2], 32.5), (COLUNA[1] + L, 32.5), cor=NEUTRO)
seta(ax, (COLUNA[1] + L / 2, 39.0), (COLUNA[2] + L / 2, 65.0), cor=ALERTA, curvatura=-0.22)

ax.text(COLUNA[1] + L / 2 + 3, 52, "nova versão do pacote\n(major se k mudar)", fontsize=7.6,
        color=ALERTA, fontweight="bold", ha="left", va="center")
ax.text(COLUNA[3] + L / 2 + 1.5, 52, "toda atribuição alimenta\na medição do desvio",
        fontsize=7.6, color=config.AZUL_COBALTO, ha="left", va="center")

ax.set_title("A integração em produção — dois caminhos de consumo, uma única receita",
             fontsize=13, color=config.AZUL_PROFUNDO, fontweight="bold", pad=14)

arquitetura = pd.DataFrame(
    [
        {"etapa": "1 · Carga", "o_que_acontece": "O cadastro de imóveis publica a base do mês",
         "onde": "data/raw/ — job mensal", "responsavel": "Engenharia de Dados",
         "falha_possivel": "arquivo ausente, volume anômalo ou coluna faltando",
         "o_que_acontece_na_falha": "a carga é bloqueada e a classificação vigente permanece"},
        {"etapa": "2 · Contrato", "o_que_acontece": "Cada registro é validado antes de entrar",
         "onde": "models/contrato_de_entrada.json", "responsavel": "Engenharia de Dados",
         "falha_possivel": "valor fora da faixa aceita ou tipo incompatível",
         "o_que_acontece_na_falha": "o registro é rejeitado com o motivo, e não classificado "
                                    "por padrão"},
        {"etapa": "3 · Preparo", "o_que_acontece": "log1p, padronização e ordem de colunas",
         "onde": "models/modelo-segmentacao.joblib — passo `preparo`",
         "responsavel": "Área de Dados",
         "falha_possivel": "reimplementação da receita na camada de aplicação",
         "o_que_acontece_na_falha": "proibido por desenho: só existe uma receita, dentro do "
                                    "pacote"},
        {"etapa": "4 · Atribuição", "o_que_acontece": "Segmento canônico e faixa de confiança",
         "onde": "models/modelo-segmentacao.joblib — passo `segmentacao`",
         "responsavel": "Área de Dados",
         "falha_possivel": "imóvel na fronteira entre dois segmentos",
         "o_que_acontece_na_falha": "sai marcado como fronteira e vai para revisão humana"},
        {"etapa": "5 · Entrega em lote", "o_que_acontece": "Carteira classificada e relatórios",
         "onde": "data/processed/carteira_classificada.parquet",
         "responsavel": "Área de Dados",
         "falha_possivel": "consumidor lê arquivo antigo sem perceber",
         "o_que_acontece_na_falha": "o carimbo do pacote vai no arquivo; versão diferente é "
                                    "recusada pelo consumidor"},
        {"etapa": "6 · Entrega sob demanda",
         "o_que_acontece": "Resposta com segmento, ação, métrica e confiança",
         "onde": "aplicação Streamlit e, quando houver, integração com o CRM",
         "responsavel": "Diretoria Comercial",
         "falha_possivel": "ação automática disparada sobre imóvel de fronteira",
         "o_que_acontece_na_falha": "a resposta traz `confianca = fronteira` e a ação fica "
                                    "suspensa até revisão"},
    ]
)

salvar(fig, "arquitetura-de-integracao", arquitetura)
arquitetura[["etapa", "onde", "responsavel", "o_que_acontece_na_falha"]]

## 1.2 · O pacote publicável e o carimbo de integridade

O bloco 0 mostrou o que acontece quando "o modelo" é um arquivo sem identidade: convive com
outro arquivo de nome parecido, e ninguém percebe qual está sendo carregado. A correção é
barata e definitiva — **versão declarada e resumo criptográfico**.

O carimbo é um SHA-256 de cada artefato do pacote. Ele responde a uma pergunta que nenhuma
métrica responde: *o objeto que está rodando agora é bit a bit o mesmo que foi avaliado?* Se o
resumo diferir, a aplicação recusa a carga em vez de classificar com um modelo desconhecido.

A regra de versionamento é declarada agora, antes da primeira reciclagem, para que ninguém
precise decidir isso no meio de uma:

| Mudança | Parte da versão | Consequência para o consumidor |
|:---|:---|:---|
| Muda *k*, o espaço de atributos ou o algoritmo | **MAIOR** (`2.0.0` → `3.0.0`) | O catálogo de personas muda: toda ação comercial precisa ser revista com a Diretoria |
| Reajuste dos centroides com o mesmo *k* e o mesmo espaço | **MENOR** (`2.0.0` → `2.1.0`) | Imóveis podem trocar de segmento; personas e ações continuam válidas |
| Muda limiar operacional, texto de persona ou documentação | **CORREÇÃO** (`2.0.0` → `2.0.1`) | Nenhuma classificação muda |

A versão publicada aqui é a **2.0.0**: MAIOR em relação à `1.0.0` que está no repositório,
porque *k* passou de 4 para 3 e o espaço passou de 21 para 17 variáveis. Não é um ajuste da
versão anterior — é outro modelo, e a numeração precisa dizer isso.

In [ ]:
def resumo_sha256(caminho: Path, bloco: int = 1 << 20) -> str:
    """SHA-256 do arquivo, lido em blocos — o carimbo de integridade do artefato."""
    digestor = hashlib.sha256()
    with open(caminho, "rb") as arquivo:
        for pedaco in iter(lambda: arquivo.read(bloco), b""):
            digestor.update(pedaco)
    return digestor.hexdigest()


VERSAO_MODELO = "2.0.0"
VERSAO_ANTERIOR = metadados_legado["versao_modelo"]

ARTEFATOS_DO_PACOTE = {
    "modelo": MODELO,
    "base_segmentada": BASE_SEGMENTADA,
    "base_modelagem": BASE_MODELAGEM,
    "manifesto_modelagem": config.DATA_PROCESSED / "manifesto_modelagem.json",
    "manifesto_avaliacao": config.DATA_PROCESSED / "manifesto_avaliacao.json",
}

carimbo = pd.DataFrame(
    [
        {
            "artefato": nome,
            "arquivo": caminho.relative_to(RAIZ).as_posix(),
            "bytes": caminho.stat().st_size,
            "sha256": resumo_sha256(caminho),
        }
        for nome, caminho in ARTEFATOS_DO_PACOTE.items()
    ]
)

# A conferência que a aplicação fará na carga: recalcular o resumo e comparar.
CARIMBO_DO_MODELO = carimbo.loc[carimbo.artefato == "modelo", "sha256"].iloc[0]
CARIMBO_CONFERE = resumo_sha256(MODELO) == CARIMBO_DO_MODELO

registrar(
    "1 · Implantação",
    f"Publicar o pacote {VERSAO_MODELO} com carimbo de integridade",
    "Área de Dados",
    "a aplicação recusa a carga quando o SHA-256 não confere",
    f"sha256 {CARIMBO_DO_MODELO[:16]}… · {len(carimbo)} artefatos carimbados",
)

io.salvar_tabela(carimbo, SECAO, "carimbo-de-integridade", indice=False)
print(f"Versão publicada: {VERSAO_MODELO} (a anterior no repositório é {VERSAO_ANTERIOR})")
print(f"Carimbo confere na releitura: {CARIMBO_CONFERE}")
carimbo[["artefato", "arquivo", "bytes", "sha256"]]

## 1.3 · O contrato de entrada

Um modelo em produção recebe dados que ninguém revisou. A carteira de treino tinha 2.350
imóveis conferidos por três fases; a carga do mês que vem terá o que o cadastro digitar.

O contrato declara, para cada uma das 17 variáveis de entrada, **três faixas** — e a diferença
entre elas é o que separa um erro que aparece de um erro que se esconde:

| Faixa | Definição | O que a aplicação faz |
|:---|:---|:---|
| **Observada** | O que existiu na carteira de treino | Classifica normalmente |
| **Aceita** | A faixa observada, estendida em 25% no topo para as contínuas; para as ordinais, exatamente a escala do dicionário | Classifica e **marca como extrapolação** — o modelo nunca viu um imóvel assim |
| **Fora da aceita** | Qualquer valor além disso | **Rejeita o registro**, com o motivo, e não classifica |

As ordinais não ganham folga: elas são escalas mapeadas na fase 3, e um valor fora da escala
não é um imóvel incomum, é um erro de digitação. Para as contínuas a folga existe porque um
terreno 20% maior que o maior da carteira é raro, mas plausível — e recusá-lo seria transformar
o modelo em obstáculo operacional.

O padrão de cada campo é a **mediana da carteira**, e ele existe por uma razão específica: o
formulário do Comitê de Aquisição não vai ter 17 campos preenchidos. O que não vier vem da
mediana, e a resposta diz quais campos foram completados assim — suposição visível é suposição
que pode ser contestada.

In [ ]:
ORDINAIS_DE_ENTRADA = [coluna for coluna in ESPACO if coluna in dic.ORDINAIS]
FOLGA_CONTINUA = 0.25

linhas_contrato = []
for coluna in ESPACO:
    serie = base[coluna]
    ordinal = coluna in ORDINAIS_DE_ENTRADA
    minimo, maximo = float(serie.min()), float(serie.max())
    aceito_maximo = maximo if ordinal else round(maximo * (1 + FOLGA_CONTINUA), 2)
    linhas_contrato.append(
        {
            "campo": coluna,
            "rotulo": dic.rotulo(coluna),
            "tipo": "ordinal" if ordinal else "numérica",
            "obrigatorio": coluna in ("area_habitavel", "qualidade_geral", "idade_imovel"),
            "observado_minimo": minimo,
            "observado_maximo": maximo,
            "aceito_minimo": minimo if ordinal else 0.0,
            "aceito_maximo": aceito_maximo,
            "padrao": float(serie.median()),
        }
    )

contrato = pd.DataFrame(linhas_contrato)
PADROES = dict(zip(contrato.campo, contrato.padrao))
OBRIGATORIOS = contrato.loc[contrato.obrigatorio, "campo"].tolist()


def validar(atributos: dict) -> list[str]:
    """Confronta um registro com o contrato e devolve a lista de violações.

    Lista vazia significa registro aceito. A função devolve **todas** as
    violações, e não a primeira: quem preenche o formulário precisa corrigir uma
    vez, não descobrir um erro por tentativa.
    """
    violacoes = []
    desconhecidos = set(atributos) - set(contrato.campo)
    if desconhecidos:
        violacoes.append(f"campos fora do contrato: {sorted(desconhecidos)}")
    for campo in OBRIGATORIOS:
        if atributos.get(campo) is None:
            violacoes.append(f"`{campo}` é obrigatório e não pode vir da mediana")
    for _, regra in contrato.iterrows():
        valor = atributos.get(regra.campo)
        if valor is None:
            continue
        if not isinstance(valor, (int, float, np.integer, np.floating)):
            violacoes.append(f"`{regra.campo}` precisa ser numérico e veio {type(valor).__name__}")
        elif not (regra.aceito_minimo <= float(valor) <= regra.aceito_maximo):
            violacoes.append(
                f"`{regra.campo}` = {valor} está fora da faixa aceita "
                f"[{regra.aceito_minimo:g}, {regra.aceito_maximo:g}]"
            )
    return violacoes


def extrapolacoes(atributos: dict) -> list[str]:
    """Campos dentro da faixa aceita, porém fora do que a carteira de treino viu."""
    fora = []
    for _, regra in contrato.iterrows():
        valor = atributos.get(regra.campo)
        if valor is None:
            continue
        if not regra.observado_minimo <= float(valor) <= regra.observado_maximo:
            fora.append(regra.campo)
    return fora


io.salvar_json(
    {
        "versao_modelo": VERSAO_MODELO,
        "folga_das_continuas": FOLGA_CONTINUA,
        "obrigatorios": OBRIGATORIOS,
        "campos": contrato.set_index("campo").to_dict(orient="index"),
    },
    config.MODELS / "contrato_de_entrada.json",
)
io.salvar_tabela(contrato, SECAO, "contrato-de-entrada", indice=False)

registrar(
    "1 · Implantação",
    "Publicar o contrato de entrada das 17 variáveis",
    "Engenharia de Dados",
    "todo registro rejeitado sai com o motivo, e nenhum entra sem validação",
    f"{len(contrato)} campos com faixa aceita, faixa observada e padrão · "
    f"{len(OBRIGATORIOS)} obrigatórios",
)
contrato.head(8)

In [ ]:
# O contrato só serve se rejeitar de fato. Dois registros de teste: um plausível
# e um com três erros que um cadastro real produz — área em metros quadrados
# informada como pés, nota fora da escala e campo inexistente.
imovel_plausivel = {"area_habitavel": 1500.0, "qualidade_geral": 6.0, "idade_imovel": 30.0}
imovel_problematico = {
    "area_habitavel": 140.0,
    "qualidade_geral": 15.0,
    "area_terreno": 999_999.0,
    "piscina": 1,
}

teste_do_contrato = pd.DataFrame(
    [
        {"registro": "imóvel plausível", "violacoes": len(validar(imovel_plausivel)),
         "detalhe": "; ".join(validar(imovel_plausivel)) or "aceito",
         "extrapolacoes": ", ".join(extrapolacoes(imovel_plausivel)) or "nenhuma"},
        {"registro": "imóvel com erro de cadastro", "violacoes": len(validar(imovel_problematico)),
         "detalhe": "; ".join(validar(imovel_problematico)),
         "extrapolacoes": ", ".join(extrapolacoes(imovel_problematico)) or "nenhuma"},
    ]
)

io.salvar_tabela(teste_do_contrato, SECAO, "teste-do-contrato", indice=False)
for _, linha in teste_do_contrato.iterrows():
    print(f"{linha.registro}: {linha.violacoes} violação(ões)")
    print(f"    {linha.detalhe}\n")
teste_do_contrato

## 1.4 · A regra de atribuição — e o imóvel que a regra não sabe classificar

Este é o **item 4 do backlog da fase 5**, e o achado que o originou: **178 imóveis estão em
silhueta negativa** — um em cada quatro do segmento 1, ou 25,6% das unidades daquela linha. O
número é da fase 5 e entra aqui pelo manifesto, não é remedido: a conferência do bloco 0
apenas confirma que a base em mãos ainda o reproduz.

Silhueta negativa quer dizer que o imóvel está, em média, mais perto dos imóveis de outro grupo
do que dos do próprio. O K-Means atribui esses imóveis assim mesmo — ele sempre atribui, e essa
é a diferença entre o algoritmo e a operação:

> **O algoritmo não pode dizer "não sei". A regra de atribuição pode — e deve.**

A medida é a **razão de fronteira**: a distância ao centroide mais próximo dividida pela
distância ao segundo mais próximo. Perto de 0, o imóvel é núcleo do segmento; perto de 1, está
na divisa e a atribuição é sorte.

O corte precisa ser calibrado, e a regra de calibração é declarada **antes** do número
aparecer, para que não seja escolhida pelo resultado que produz:

> **Corte de fronteira** = o valor que maximiza o F1 na identificação dos imóveis de silhueta
> negativa, respeitando o teto operacional de **15% da carteira** marcada para revisão humana.

O teto vem da capacidade da Gerência de Ativos, não da estatística: marcar 30% da carteira
para revisão manual é o mesmo que não marcar nada, porque ninguém revisa. O **corte de
núcleo** é o primeiro quartil da razão — a quarta parte da carteira que está mais colada ao
próprio centro, e onde a ação automática pode disparar sem revisão.

O que esta seção **não** fecha fica dito desde já, porque a fase 5 escreveu o endereço junto
com a tarefa: o item 4 pedia a regra em `src/deployment/scoring.py`, e o que existe ao fim
deste bloco é a regra calibrada dentro do notebook e publicada como parâmetro. A reconciliação
do bloco 4.3 confere o endereço e registra a diferença — não a esconde.

In [ ]:
def razao_de_fronteira(matriz: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Distâncias aos centroides, o segmento canônico e a razão do mais próximo pelo segundo."""
    distancias = np.linalg.norm(matriz[:, None, :] - CENTROIDES[None, :, :], axis=2)
    ordem = np.argsort(distancias, axis=1)
    proximo = np.take_along_axis(distancias, ordem[:, :1], axis=1).ravel()
    segundo = np.take_along_axis(distancias, ordem[:, 1:2], axis=1).ravel()
    canonicos = np.array([MAPA_CANONICO[int(cru)] for cru in ordem[:, 0]])
    return distancias, canonicos, proximo / segundo


DISTANCIAS, CANONICOS, RAZAO = razao_de_fronteira(EUCLIDIANO)
assert (CANONICOS == ROTULOS).all(), "a regra de atribuição discorda da partição aprovada"

silhueta_negativa = (segmentada.silhueta < 0).to_numpy()
TETO_OPERACIONAL = 0.15

candidatos = []
for corte in np.round(np.arange(0.70, 0.995, 0.005), 3):
    marcado = RAZAO >= corte
    verdadeiro = int((marcado & silhueta_negativa).sum())
    precisao = verdadeiro / max(int(marcado.sum()), 1)
    revocacao = verdadeiro / max(int(silhueta_negativa.sum()), 1)
    candidatos.append(
        {
            "corte": corte,
            "pct_marcado": round(100 * float(marcado.mean()), 2),
            "precisao": round(precisao, 3),
            "revocacao": round(revocacao, 3),
            "f1": round(2 * precisao * revocacao / max(precisao + revocacao, 1e-9), 3),
        }
    )

calibracao = pd.DataFrame(candidatos)
admissiveis = calibracao[calibracao.pct_marcado <= 100 * TETO_OPERACIONAL]
escolhido = admissiveis.sort_values(["f1", "corte"], ascending=[False, True]).iloc[0]

CORTE_FRONTEIRA = float(escolhido.corte)
CORTE_NUCLEO = round(float(np.quantile(RAZAO, 0.25)), 3)
MARCADOS = int((RAZAO >= CORTE_FRONTEIRA).sum())

print(f"Corte de fronteira calibrado: {CORTE_FRONTEIRA:.3f}")
print(f"    marca {pt(MARCADOS)} imóveis ({escolhido.pct_marcado:.1f}% da carteira)")
print(f"    encontra {escolhido.revocacao:.1%} dos {IMOVEIS_EM_SILHUETA_NEGATIVA} imóveis "
      f"de silhueta negativa, com precisão de {escolhido.precisao:.1%}")
print(f"Corte de núcleo (primeiro quartil da razão): {CORTE_NUCLEO:.3f}")

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(15.5, 5.4), constrained_layout=True)

eixos[0].hist([RAZAO[~silhueta_negativa], RAZAO[silhueta_negativa]], bins=40, stacked=True,
              color=[DESTAQUE, ALERTA], label=["silhueta positiva", "silhueta negativa"])
eixos[0].axvline(CORTE_NUCLEO, color=config.AZUL_PROFUNDO, ls="dotted", lw=2)
eixos[0].axvline(CORTE_FRONTEIRA, color=REFERENCIA, ls="dashed", lw=2.4)
altura_texto = eixos[0].get_ylim()[1] * 0.9
eixos[0].text(CORTE_NUCLEO - 0.012, altura_texto, f"núcleo\n< {CORTE_NUCLEO:.3f}", ha="right",
              va="top", fontsize=8.5, color=config.AZUL_PROFUNDO, fontweight="bold")
eixos[0].text(CORTE_FRONTEIRA + 0.012, altura_texto, f"fronteira\n≥ {CORTE_FRONTEIRA:.3f}",
              ha="left", va="top", fontsize=8.5, color=config.AZUL_PROFUNDO, fontweight="bold")
eixos[0].set_xlabel("Razão de fronteira — distância ao 1º centro ÷ distância ao 2º")
eixos[0].set_ylabel("Imóveis")
eixos[0].set_title("Onde a atribuição deixa de ser confiável")
eixos[0].legend(fontsize=8.5, loc="upper left")

eixos[1].plot(calibracao.corte, calibracao.precisao, color=BARRA, lw=2, label="precisão")
eixos[1].plot(calibracao.corte, calibracao.revocacao, color=APROVADO, lw=2, label="revocação")
eixos[1].plot(calibracao.corte, calibracao.f1, color=ALERTA, lw=2.6, label="F1")
eixos[1].plot(calibracao.corte, calibracao.pct_marcado / 100, color=NEUTRO, lw=1.6, ls="dashed",
              label="fração marcada")
eixos[1].axhline(TETO_OPERACIONAL, color=REFERENCIA, ls="dotted", lw=2)
eixos[1].axvline(CORTE_FRONTEIRA, color=REFERENCIA, ls="dashed", lw=2.4)
eixos[1].annotate(f" teto operacional: {TETO_OPERACIONAL:.0%} da carteira ",
                  xy=(0.70, TETO_OPERACIONAL), va="bottom", ha="left", fontsize=8.5,
                  color=config.AZUL_PROFUNDO)
eixos[1].set_xlabel("Corte candidato")
eixos[1].set_ylabel("Valor da medida")
eixos[1].set_title(f"A calibração — o corte é medido, não escolhido\nF1 máximo dentro do teto: "
                   f"{CORTE_FRONTEIRA:.3f}")
eixos[1].legend(fontsize=8.5, ncols=2)

fig.suptitle("A regra de atribuição que admite não saber", fontsize=13, fontweight="bold",
             color=config.AZUL_PROFUNDO)

salvar(fig, "calibracao-da-fronteira", calibracao)
admissiveis.sort_values("f1", ascending=False).head(6)

In [ ]:
def faixa_de_confianca(razao: np.ndarray) -> np.ndarray:
    """Traduz a razão de fronteira nas três faixas operacionais."""
    return np.where(razao < CORTE_NUCLEO, "núcleo",
                    np.where(razao < CORTE_FRONTEIRA, "típico", "fronteira"))


FAIXAS = faixa_de_confianca(RAZAO)
faixas_por_segmento = (
    pd.crosstab(pd.Series(ROTULOS, name="segmento"), pd.Series(FAIXAS, name="faixa"),
                normalize="index")
    .reindex(columns=["núcleo", "típico", "fronteira"])
    .mul(100)
    .round(1)
)
faixas_por_segmento["imóveis na fronteira"] = (
    pd.crosstab(ROTULOS, FAIXAS).reindex(columns=["fronteira"]).fronteira
)
SEGMENTO_MAIS_FRAGIL = int(faixas_por_segmento["fronteira"].idxmax())

# A régua da fase 5 concentrou a silhueta negativa num segmento; a régua desta fase é
# a razão de fronteira, que é outra medida. As duas apontam a mesma linha de produto —
# e é isso que a conferência abaixo registra. Se um dia divergirem, é a divergência que
# importa, não a coincidência.
assert SEGMENTO_MAIS_FRAGIL == SEGMENTO_DE_FRONTEIRA, (
    "a faixa de fronteira aponta um segmento diferente do que a fase 5 mediu"
)

registrar(
    "1 · Implantação",
    "Calibrar a regra de fronteira sobre a carteira",
    "Área de Dados",
    tarefa_da_fase_5(4).criterio_de_conclusao,
    f"corte {CORTE_FRONTEIRA:.3f} · {pt(MARCADOS)} imóveis "
    f"({100 * MARCADOS / LINHAS:.1f}%) para revisão humana · "
    f"{escolhido.revocacao:.0%} dos imóveis de silhueta negativa capturados · "
    f"segmento {SEGMENTO_MAIS_FRAGIL}, o mesmo que a fase 5 apontou",
    acao="parâmetro calibrado",
    onde=tarefa_da_fase_5(4).onde,
)

io.salvar_tabela(faixas_por_segmento.reset_index(), SECAO, "faixas-de-confianca", indice=False)
print(f"O segmento {SEGMENTO_MAIS_FRAGIL} concentra a fronteira: "
      f"{faixas_por_segmento.loc[SEGMENTO_MAIS_FRAGIL, 'fronteira']:.1f}% das suas unidades")
faixas_por_segmento

## 1.5 · As três linhas de produto

Este é o **item 1 do backlog da fase 5** — o que bloqueia a publicação. O catálogo em
`src/model/personas.py` descreve quatro personas, escritas para a segmentação anterior. O
modelo aprovado tem três segmentos. Enquanto os dois números discordarem, qualquer tela
construída em cima nomeia um segmento que não existe.

O critério de conclusão é o que a fase 5 escreveu, e ele tem duas metades:

> `len(PERSONAS) == k do manifesto, travado por teste automatizado, com ação e métrica por
> segmento` — em `src/model/personas.py`.

Esta seção entrega a primeira metade e **emenda o endereço** da segunda, e as duas coisas
precisam ficar separadas. A emenda: o catálogo é publicado como **artefato de dados**
(`models/catalogo_de_personas.json`), e não como constante no código, pela mesma razão que os
critérios da fase 4 moram no manifesto — o texto de negócio precisa poder mudar com a
reciclagem sem alterar módulo nenhum, e precisa carregar a versão do modelo a que pertence.

A emenda não fecha o item sozinha. Enquanto `src/model/personas.py` continuar declarando a
constante da cadeia anterior — com os **mesmos códigos S1, S2 e S3** para outros imóveis e
outras ações, como o bloco 0 mediu —, existem dois catálogos disputando o mesmo nome, e nenhum
teste impede que a divergência volte. O bloco 4.3 confere isso e registra o que sobra, com dono.

Nada aqui é escolhido por impressão: o retrato de cada linha vem da mediana medida, e a ação
vem do que essa mediana permite fazer.

In [ ]:
perfil = perf.perfil_por_segmento(curada, ROTULOS, ESPACO)
assinatura = perf.assinatura(curada, ROTULOS, ESPACO)
valor = perf.leitura_a_posteriori(curada, ROTULOS)
silhueta_por_segmento = part.silhueta_por_grupo(EUCLIDIANO, ROTULOS)
zoneamento = perf.composicao_nominal(curada, ROTULOS, "zoneamento", top=3)

CORES_SEGMENTO = {s: config.PALETA_SEGMENTOS[s] for s in range(K)}

CATALOGO = {
    0: {
        "codigo": "S1",
        "nome": "Compacto herdado",
        "retrato": f"{perfil.loc[0, 'area_habitavel']:.0f} pé² habitáveis, "
                   f"{perfil.loc[0, 'idade_imovel']:.0f} anos de idade e "
                   f"{perfil.loc[0, 'anos_desde_reforma']:.0f} anos desde a última reforma; "
                   f"qualidade geral {perfil.loc[0, 'qualidade_geral']:.0f}, "
                   f"{perfil.loc[0, 'banheiros_total']:.1f} banheiros e sem lareira. "
                   f"{zoneamento.loc[0].get('RM', 0):.0f}% em zoneamento de média densidade.",
        "politica_de_preco": "Percentil 40 do bairro — precificação por giro, não por margem",
        "teto_de_reforma": "6% do valor de avaliação",
        "acao": "Decisão unidade a unidade em 12 meses: reforma leve padronizada (pintura, "
                "piso e louças) e venda, ou conversão em ativo de renda, ou desinvestimento. "
                "Acima do teto de 6%, o retorno não se paga neste padrão.",
        "canal": "Digital, com foco em primeira aquisição e crédito facilitado",
        "metrica": "Percentual do grupo com destino definido em 12 meses e tempo médio de giro",
        "meta": "100% com destino definido · giro de 90 para 60 dias",
    },
    1: {
        "codigo": "S2",
        "nome": "Amplo para requalificar",
        "retrato": f"{perfil.loc[1, 'area_habitavel']:.0f} pé² habitáveis — "
                   f"{100 * (perfil.loc[1, 'area_habitavel'] / perfil.loc[0, 'area_habitavel'] - 1):.0f}% "
                   f"mais espaço que o S1 — com {perfil.loc[1, 'comodos']:.0f} cômodos, "
                   f"{perfil.loc[1, 'banheiros_total']:.1f} banheiros e lareira, aos "
                   f"{perfil.loc[1, 'idade_imovel']:.0f} anos. É o **menor preço por pé² da "
                   f"carteira**: {valor.loc[1, 'preco_por_pe2']:.2f} dólares contra "
                   f"{valor.loc[2, 'preco_por_pe2']:.2f} do S3.",
        "politica_de_preco": "Comparação direta de bairro, com reavaliação após a reforma",
        "teto_de_reforma": "12% do valor de avaliação",
        "acao": "Pacote de valorização concentrado em cozinha e banheiro — os dois atributos "
                "em que o grupo fica meio nível abaixo do S3 — executado antes do anúncio. "
                "É onde a reforma tem a maior alavanca da carteira: espaço barato que a "
                "reforma reprecifica.",
        "canal": "Corretagem tradicional, com foco em família em segunda aquisição",
        "metrica": "Ganho de preço realizado sobre o custo da reforma e percentual reformado "
                   "antes do anúncio",
        "meta": "ganho ≥ 1,8× o investido · ≥ 70% reformados antes do anúncio",
    },
    2: {
        "codigo": "S3",
        "nome": "Recente de alto padrão",
        "retrato": f"{perfil.loc[2, 'idade_imovel']:.0f} anos de idade mediana, qualidade geral "
                   f"{perfil.loc[2, 'qualidade_geral']:.0f}, "
                   f"{perfil.loc[2, 'banheiros_total']:.1f} banheiros e porão de "
                   f"{perfil.loc[2, 'area_porao']:.0f} pé². É "
                   f"{valor.loc[2, 'imoveis'] / LINHAS:.0%} da carteira em unidades e "
                   f"{valor.loc[2, 'pct_receita']:.1f}% da receita transacionada.",
        "politica_de_preco": "Comparação com lançamentos do período; desconto só acima de "
                             "90 dias em estoque",
        "teto_de_reforma": "sem reforma — o imóvel já está no padrão",
        "acao": "Corretor dedicado, material de venda com fotografia profissional e vitrine "
                "premium. Nenhum orçamento de requalificação: gastar aqui é destruir margem "
                "no grupo que sustenta a receita.",
        "canal": "Corretagem dedicada e vitrine premium",
        "metrica": "Razão entre preço realizado e preço anunciado e percentual de vendas sem "
                   "desconto",
        "meta": "realizado ÷ anunciado ≥ 0,97 · ≥ 60% sem desconto",
    },
}

catalogo = (
    pd.DataFrame(CATALOGO)
    .T.rename_axis("segmento")
    .assign(
        imoveis=[int(valor.loc[s, "imoveis"]) for s in range(K)],
        pct_carteira=[round(100 * valor.loc[s, "imoveis"] / LINHAS, 1) for s in range(K)],
        pct_receita=[float(valor.loc[s, "pct_receita"]) for s in range(K)],
        preco_mediano=[float(valor.loc[s, "preco_mediano"]) for s in range(K)],
        silhueta_media=[float(silhueta_por_segmento.loc[s, "media"]) for s in range(K)],
        pct_na_fronteira=[float(faixas_por_segmento.loc[s, "fronteira"]) for s in range(K)],
        cor=[CORES_SEGMENTO[s] for s in range(K)],
    )
)

assert len(catalogo) == K, "o catálogo precisa ter exatamente uma persona por segmento"
assert catalogo.codigo.is_unique, "os códigos de linha de produto precisam ser únicos"

io.salvar_json(
    {
        "versao_modelo": VERSAO_MODELO,
        "k": K,
        "personas": {str(s): CATALOGO[s] for s in CATALOGO},
    },
    config.MODELS / "catalogo_de_personas.json",
)
io.salvar_tabela(catalogo.reset_index(), SECAO, "catalogo-de-personas", indice=False)

registrar(
    "1 · Implantação",
    f"Reescrever o catálogo de personas para {K} segmentos",
    "Área de Dados com a Diretoria Comercial",
    tarefa_da_fase_5(1).criterio_de_conclusao,
    f"{len(catalogo)} personas para k = {K} · publicado em models/catalogo_de_personas.json · "
    f"src/model/personas.py continua com {len(pers.PERSONAS)} e fica com a fase 4.3",
    acao="artefato publicado",
    onde=tarefa_da_fase_5(1).onde,
)
catalogo[["codigo", "nome", "imoveis", "pct_carteira", "pct_receita", "teto_de_reforma"]]

In [ ]:
fig = plt.figure(figsize=(16.5, 6.4), constrained_layout=True)
grade_figura = fig.add_gridspec(1, 3, width_ratios=[1.5, 1, 1])
eixo_assinatura = fig.add_subplot(grade_figura[0, 0])
eixo_peso = fig.add_subplot(grade_figura[0, 1])
eixo_preco = fig.add_subplot(grade_figura[0, 2])

sns.heatmap(
    assinatura.T.rename(index=dic.rotulo), annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    linewidths=0.6, linecolor="white", ax=eixo_assinatura,
    cbar_kws={"shrink": 0.75, "label": "desvios da mediana da carteira"},
)
eixo_assinatura.set_xticklabels([f"{catalogo.loc[s, 'codigo']}" for s in range(K)])
eixo_assinatura.set_title("O que cada linha tem de diferente da carteira")
eixo_assinatura.set_xlabel("")
eixo_assinatura.set_ylabel("")

posicoes = np.arange(K)
eixo_peso.barh(posicoes - 0.19, catalogo.pct_carteira, height=0.36, color=NEUTRO,
               label="% das unidades")
eixo_peso.barh(posicoes + 0.19, catalogo.pct_receita, height=0.36,
               color=[CORES_SEGMENTO[s] for s in range(K)],
               label="% da receita (cor da linha)")
for posicao, (unidades, receita) in enumerate(zip(catalogo.pct_carteira, catalogo.pct_receita)):
    eixo_peso.text(unidades + 0.8, posicao - 0.19, f"{unidades:.1f}%", va="center", fontsize=8.5)
    eixo_peso.text(receita + 0.8, posicao + 0.19, f"{receita:.1f}%", va="center", fontsize=8.5,
                   fontweight="bold")
eixo_peso.set_yticks(posicoes)
eixo_peso.set_yticklabels([f"{catalogo.loc[s, 'codigo']} — {catalogo.loc[s, 'nome']}"
                           for s in range(K)], fontsize=8.5)
eixo_peso.invert_yaxis()
eixo_peso.set_xlim(0, max(catalogo.pct_receita.max(), catalogo.pct_carteira.max()) * 1.28)
eixo_peso.set_xlabel("% da carteira")
eixo_peso.set_title("Unidades contra receita")
eixo_peso.legend(fontsize=8, loc="upper right")

for s in range(K):
    eixo_preco.plot([valor.loc[s, "preco_p10"], valor.loc[s, "preco_p90"]], [s, s],
                    color=CORES_SEGMENTO[s], lw=7, solid_capstyle="round", alpha=0.55)
    eixo_preco.scatter(valor.loc[s, "preco_mediano"], s, s=110, zorder=5,
                       color=CORES_SEGMENTO[s], edgecolor="white", linewidth=1.4)
    eixo_preco.text(valor.loc[s, "preco_p90"] * 1.02, s,
                    f"  mediana {usd(valor.loc[s, 'preco_mediano'])}\n"
                    f"  {valor.loc[s, 'preco_por_pe2']:.0f} US$/pé²",
                    va="center", fontsize=8.2, color=config.AZUL_PROFUNDO)
eixo_preco.set_yticks(posicoes)
eixo_preco.set_yticklabels([catalogo.loc[s, "codigo"] for s in range(K)])
eixo_preco.invert_yaxis()
eixo_preco.set_xlim(0, valor.preco_p90.max() * 1.75)
eixo_preco.set_xlabel("Preço de venda (US$) — faixa do decil 10 ao 90")
eixo_preco.set_title("O que cada linha vale")

fig.suptitle(f"As {K} linhas de produto da carteira — perfil, peso e valor", fontsize=13,
             fontweight="bold", color=config.AZUL_PROFUNDO)

salvar(fig, "linhas-de-produto", catalogo.reset_index().drop(columns=["cor"]))
perfil.T

### A leitura das três linhas, incluindo a parte incômoda

**S1 · Compacto herdado** — 39% das unidades e 27% da receita. É o estoque que a carteira
herdou: pequeno, meio século de idade, quase quatro décadas sem reforma. A ação é decisão
unidade a unidade, e o teto baixo de 6% existe para impedir o erro clássico — investir
reforma cara em imóvel de padrão baixo e não recuperar o gasto na venda.

**S2 · Amplo para requalificar** — 28% das unidades e 28% da receita. É a linha mais
interessante do ponto de vista comercial, e o número que a define é uma razão, não um valor
absoluto: tem **55% mais área que o S1 e o menor preço por pé² da carteira**. Espaço barato é
exatamente onde a reforma tem alavanca — por isso é a única linha com teto de 12%.

**S3 · Recente de alto padrão** — 33% das unidades e **44,8% da receita**. Seis anos de idade
mediana, três banheiros, acabamento superior. Nenhuma reforma: gastar aqui reduz margem no
grupo que sustenta a carteira.

E a parte incômoda, que precisa estar no catálogo e não numa nota de rodapé:

> **O S2 é a linha de fronteira.** É onde se concentram os imóveis de silhueta negativa, e a
> regra de atribuição marca uma parcela desproporcional dele para revisão humana. A leitura
> operacional é direta: o S2 é uma linha de produto **assistida** — a ação automática não
> dispara sem que alguém olhe. Não é defeito do modelo; é o modelo dizendo onde ele não sabe,
> que é a informação mais útil que ele produz para a operação.

## 1.6 · O contrato de serviço — como um imóvel novo é classificado

A função abaixo é o **ponto único de entrada** da arquitetura: a aplicação Streamlit chama
ela, e a integração com o CRM, quando existir, chama a mesma coisa por HTTP. Uma única
implementação, para que a resposta da tela e a resposta do serviço não possam divergir.

O contrato do serviço, para quando a integração for construída:

```
POST /segmentacao/v2/classificar
Content-Type: application/json

{ "area_habitavel": 1500, "qualidade_geral": 6, "idade_imovel": 30 }
```

| Resposta | Quando | Corpo |
|:---|:---|:---|
| `200` | Registro aceito | Segmento, código, persona, ação, métrica, faixa de confiança, distância ao centro e ao segundo centro, razão de fronteira, campos completados pela mediana e versão do modelo |
| `422` | Registro fora do contrato | A lista completa de violações, uma por campo — não a primeira |
| `409` | Carimbo do pacote não confere | Nenhuma classificação; a operação segue com a última carteira válida |

Três decisões de desenho ficam registradas, porque cada uma evita um erro conhecido:

1. **a resposta devolve a razão de fronteira**, e não só o segmento. Quem consome precisa poder
   decidir de forma diferente diante de um imóvel de divisa;
2. **a resposta lista os campos completados pela mediana.** Uma classificação feita com três
   campos informados e catorze presumidos não vale o mesmo que uma com dezessete informados, e
   quem lê precisa enxergar isso;
3. **a versão do modelo vai em toda resposta.** Sem ela, um registro classificado hoje e outro
   classificado depois da reciclagem parecem comparáveis, e não são.

In [ ]:
def montar_registro(atributos: dict) -> pd.DataFrame:
    """Monta a linha de entrada, completando o que faltar com a mediana da carteira."""
    registro = {campo: float(atributos.get(campo, PADROES[campo])) for campo in ESPACO}
    return pd.DataFrame([registro])[ESPACO]


def classificar(atributos: dict) -> dict:
    """Classifica um imóvel e devolve segmento, ação, confiança e proveniência.

    É a função que a aplicação e o serviço HTTP chamam. Ela valida antes de
    classificar: registro fora do contrato sai rejeitado, e não classificado com
    um valor absurdo que ninguém confere.
    """
    violacoes = validar(atributos)
    if violacoes:
        return {"status": "rejeitado", "violacoes": violacoes, "versao_modelo": VERSAO_MODELO}

    registro = montar_registro(atributos)
    matriz = preparo.transform(registro)
    distancias, canonicos, razao = razao_de_fronteira(matriz)

    segmento = int(canonicos[0])
    persona = CATALOGO[segmento]
    ordem = np.argsort(distancias[0])
    completados = [campo for campo in ESPACO if campo not in atributos]

    return {
        "status": "classificado",
        "segmento": segmento,
        "codigo": persona["codigo"],
        "persona": persona["nome"],
        "acao": persona["acao"],
        "teto_de_reforma": persona["teto_de_reforma"],
        "politica_de_preco": persona["politica_de_preco"],
        "canal": persona["canal"],
        "metrica": persona["metrica"],
        "confianca": str(faixa_de_confianca(razao)[0]),
        "razao_de_fronteira": round(float(razao[0]), 3),
        # As duas distâncias vão cruas na resposta, e não só a razão entre elas: é o
        # que o item 4 do backlog da fase 5 pede em `criterio_de_conclusao`, e é o que
        # permite a quem consome aplicar outro corte sem repetir a conta.
        "distancia_ao_centro": round(float(distancias[0][ordem[0]]), 4),
        "distancia_ao_segundo_centro": round(float(distancias[0][ordem[1]]), 4),
        "segundo_mais_proximo": CATALOGO[MAPA_CANONICO[int(ordem[1])]]["codigo"],
        "acao_automatica_liberada": bool(razao[0] < CORTE_FRONTEIRA),
        "campos_informados": sorted(set(atributos) & set(ESPACO)),
        "campos_completados_pela_mediana": completados,
        "extrapolacoes": extrapolacoes(atributos),
        "versao_modelo": VERSAO_MODELO,
        "carimbo_do_pacote": CARIMBO_DO_MODELO[:16],
    }


# Quatro ensaios. O terceiro não é inventado: é o imóvel real da carteira com a
# maior razão de fronteira — o caso em que a regra precisa dizer que não sabe.
imovel_do_comite = {
    "area_habitavel": 1900.0,
    "qualidade_geral": 8.0,
    "idade_imovel": 4.0,
    "banheiros_total": 3.0,
    "area_garagem": 620.0,
    "area_porao": 1300.0,
}
imovel_do_comite_completo = {
    **imovel_do_comite,
    "anos_desde_reforma": 4.0,
    "qualidade_cozinha": 4.0,
    "qualidade_externa": 4.0,
}
posicao_fronteirica = int(np.argmax(RAZAO))
imovel_de_fronteira = {campo: float(base.iloc[posicao_fronteirica][campo]) for campo in ESPACO}

resposta_comite = classificar(imovel_do_comite)
resposta_completa = classificar(imovel_do_comite_completo)
resposta_fronteira = classificar(imovel_de_fronteira)
resposta_rejeitada = classificar(imovel_problematico)


def resumir(titulo: str, resposta: dict) -> str:
    """Uma linha por ensaio, com o que a operação precisa ler na resposta."""
    return (
        f"{titulo}\n"
        f"    {resposta['codigo']} · confiança {resposta['confianca']} · razão "
        f"{resposta['razao_de_fronteira']} · segundo mais próximo "
        f"{resposta['segundo_mais_proximo']} · ação automática liberada: "
        f"{resposta['acao_automatica_liberada']} · "
        f"{len(resposta['campos_completados_pela_mediana'])} campos vieram da mediana"
    )


print("ENSAIO 1 — imóvel novo do Comitê de Aquisição, com 6 campos informados")
print(json.dumps(resposta_comite, indent=2, ensure_ascii=False)[:760])
print()
print(resumir("ENSAIO 2 — o MESMO imóvel, com 3 campos a mais informados",
              resposta_completa))
print(resumir(f"ENSAIO 3 — imóvel real da carteira na maior fronteira (pid "
              f"{int(base.iloc[posicao_fronteirica].pid)})", resposta_fronteira))
print(f"ENSAIO 4 — registro fora do contrato\n    {resposta_rejeitada['status']}: "
      f"{len(resposta_rejeitada['violacoes'])} violações, nenhuma classificação devolvida")

### O que os ensaios 1 e 2 mostraram — e que não estava previsto

Os dois primeiros ensaios são **o mesmo imóvel**. A diferença entre eles são três campos:
anos desde a reforma, qualidade da cozinha e qualidade externa. Com seis campos informados, o
imóvel sai na **fronteira**; com nove, sai como **típico**, e a ação automática é liberada.

O motivo é aritmético e vale entender, porque muda a instrução de preenchimento do formulário:
os campos não informados são completados pela **mediana da carteira**, que é o ponto mais
neutro possível — e um imóvel montado com metade de valores extremos e metade de valores
medianos fica, por construção, entre dois centros. A fronteira, nesse caso, **não é uma
propriedade do imóvel: é uma propriedade do preenchimento**.

Duas consequências práticas, e as duas entram no procedimento da operação:

* o formulário do Comitê de Aquisição precisa destacar os três campos obrigatórios e **pedir
  os campos de acabamento**, que são os que mais deslocam a atribuição;
* a resposta traz `campos_completados_pela_mediana` justamente para isso. Quando um imóvel cai
  na fronteira **e** tem muitos campos presumidos, a primeira providência não é escalar para
  revisão humana: é **completar o cadastro e perguntar de novo**.

Esse é o tipo de achado que só aparece quando o plano de implantação é executado uma vez, e
não apenas escrito — e é a razão de o ensaio existir neste notebook.

## 1.7 · O ensaio de produção

Um plano de implantação que não foi executado uma vez é uma hipótese. O ensaio abaixo roda o
caminho completo — contrato, preparo, atribuição, faixa de confiança — sobre a carteira
inteira, e confere três coisas que precisam ser verdade **antes** de qualquer janela de
entrada em produção:

1. **a classificação em lote reproduz a partição aprovada**, imóvel por imóvel. Não é a mesma
   conferência do bloco 0: lá o teste foi sobre o `predict` do pacote; aqui é sobre o caminho
   inteiro de produção, com a regra de fronteira no meio;
2. **o custo por imóvel cabe na operação.** Uma carga mensal de alguns milhares de registros
   não pode depender de infraestrutura que a Meridiano não tem;
3. **a saída carrega proveniência** — versão do modelo e carimbo do pacote em toda linha, para
   que ninguém precise adivinhar qual modelo produziu um arquivo encontrado meses depois.

In [ ]:
def classificar_lote(dados: pd.DataFrame) -> pd.DataFrame:
    """Classifica um conjunto inteiro — o caminho percorrido a cada carga."""
    faltando = [campo for campo in ESPACO if campo not in dados.columns]
    if faltando:
        raise ValueError(f"colunas ausentes na entrada: {faltando}")
    matriz = preparo.transform(dados[ESPACO])
    _, canonicos, razao = razao_de_fronteira(matriz)
    faixas = faixa_de_confianca(razao)
    return pd.DataFrame(
        {
            "pid": dados.pid.to_numpy(),
            "segmento": canonicos,
            "codigo_segmento": [CATALOGO[int(s)]["codigo"] for s in canonicos],
            "persona": [CATALOGO[int(s)]["nome"] for s in canonicos],
            "razao_de_fronteira": np.round(razao, 4),
            "confianca": faixas,
            "acao_automatica_liberada": razao < CORTE_FRONTEIRA,
            "versao_modelo": VERSAO_MODELO,
            "carimbo_do_pacote": CARIMBO_DO_MODELO[:16],
        }
    )


inicio = time.perf_counter()
carteira_classificada = classificar_lote(base)
DURACAO_MS = 1000 * (time.perf_counter() - inicio)

REPRODUZ_EM_LOTE = bool((carteira_classificada.segmento.to_numpy() == ROTULOS).all())
EM_FRONTEIRA = int((carteira_classificada.confianca == "fronteira").sum())
EM_NUCLEO = int((carteira_classificada.confianca == "núcleo").sum())

CARTEIRA_CLASSIFICADA = config.DATA_PROCESSED / "carteira_classificada.parquet"
carteira_classificada.to_parquet(CARTEIRA_CLASSIFICADA, index=False)

# A lista que a Gerência de Ativos recebe: só o que precisa de olho humano, com o
# preço ao lado, porque a decisão de revisão é comercial e não estatística.
revisao_humana = (
    carteira_classificada[carteira_classificada.confianca == "fronteira"]
    .merge(curada[["pid", "bairro", "preco_venda", "area_habitavel", "qualidade_geral"]], on="pid")
    .sort_values("razao_de_fronteira", ascending=False)
    .reset_index(drop=True)
)
io.salvar_tabela(revisao_humana, SECAO, "imoveis-para-revisao-humana", indice=False)

ensaio = pd.DataFrame(
    [
        {"verificacao": "A classificação em lote reproduz a partição aprovada",
         "resultado": "100% dos imóveis" if REPRODUZ_EM_LOTE else "DIVERGE",
         "situacao": "aprovado" if REPRODUZ_EM_LOTE else "reprovado"},
        {"verificacao": "Custo de processamento da carteira inteira",
         "resultado": f"{DURACAO_MS:.0f} ms para {pt(LINHAS)} imóveis "
                      f"({1000 * DURACAO_MS / LINHAS:.1f} µs por imóvel)",
         "situacao": "aprovado"},
        {"verificacao": "Imóveis liberados para ação automática",
         "resultado": f"{pt(LINHAS - EM_FRONTEIRA)} de {pt(LINHAS)} "
                      f"({100 * (LINHAS - EM_FRONTEIRA) / LINHAS:.1f}%)",
         "situacao": "aprovado"},
        {"verificacao": "Imóveis encaminhados para revisão humana",
         "resultado": f"{pt(EM_FRONTEIRA)} ({100 * EM_FRONTEIRA / LINHAS:.1f}% — teto de "
                      f"{TETO_OPERACIONAL:.0%})",
         "situacao": "aprovado" if EM_FRONTEIRA / LINHAS <= TETO_OPERACIONAL else "atenção"},
        {"verificacao": "Proveniência gravada em toda linha da saída",
         "resultado": f"versão {VERSAO_MODELO} e carimbo {CARIMBO_DO_MODELO[:16]}",
         "situacao": "aprovado"},
    ]
)

assert REPRODUZ_EM_LOTE, "o caminho de produção não reproduz a partição aprovada"

registrar(
    "1 · Implantação",
    "Executar o caminho de produção sobre a carteira inteira",
    "Área de Dados",
    "a classificação em lote reproduz a partição aprovada, imóvel por imóvel",
    f"{pt(LINHAS)} imóveis em {DURACAO_MS:.0f} ms · {pt(EM_NUCLEO)} em núcleo, "
    f"{pt(EM_FRONTEIRA)} em fronteira",
)

io.salvar_tabela(ensaio, SECAO, "ensaio-de-producao", indice=False)
ensaio

## 1.8 · Como o resultado chega ao cliente

O CRISP-DM separa *implantar o modelo* de *entregar o resultado*, e a separação é prática: o
modelo pode estar rodando e a Diretoria continuar decidindo pela planilha antiga. A entrega
tem quatro formatos, e cada um existe porque um consumidor específico não consegue usar os
outros três.

| Formato | Por que existe |
|:---|:---|
| **Arquivo** (`parquet` e `CSV`) | O sistema comercial e o orçamento consomem tabela, não tela |
| **Painel** (aplicação Streamlit) | O corretor precisa de uma pergunta por vez, com resposta imediata |
| **Serviço** (a função `classificar`, exposta por HTTP quando houver CRM) | A decisão de aquisição acontece dentro do fluxo do CRM, não fora dele |
| **Relatório** (`reports/relatorio-final.md` e o site MkDocs) | A Diretoria decide orçamento por linha de produto uma vez por trimestre, e precisa do porquê junto do quanto |

A tabela abaixo fecha cada entrega com **consumidor, meio, periodicidade, prazo e
responsável** — sem essas cinco colunas, "entregar o resultado" continua sendo uma intenção.

In [ ]:
entregas = pd.DataFrame(
    [
        {"entrega": "Carteira classificada", "consumidor": "Sistema comercial e Gerência de Ativos",
         "meio": "data/processed/carteira_classificada.parquet",
         "periodicidade": "a cada carga (mensal)", "prazo": "até D+1 da carga",
         "responsavel": "Área de Dados"},
        {"entrega": "Lista de imóveis para revisão humana",
         "consumidor": "Gerência de Ativos",
         "meio": "reports/tables/implantacao/imoveis-para-revisao-humana.csv",
         "periodicidade": "a cada carga (mensal)", "prazo": "até D+1 da carga",
         "responsavel": "Área de Dados"},
        {"entrega": "Classificação de um imóvel novo, antes da aquisição",
         "consumidor": "Comitê de Aquisição",
         "meio": "formulário da aplicação e serviço `classificar`",
         "periodicidade": "sob demanda", "prazo": "resposta imediata",
         "responsavel": "Diretoria Comercial"},
        {"entrega": "Catálogo de linhas de produto, com ação, teto e métrica",
         "consumidor": "Diretoria Comercial",
         "meio": "models/catalogo_de_personas.json e o painel",
         "periodicidade": "a cada versão MAIOR do modelo", "prazo": "antes da entrada em vigor",
         "responsavel": "Área de Dados com a Diretoria Comercial"},
        {"entrega": "Painel de monitoramento de desvio",
         "consumidor": "Área de Dados",
         "meio": "aba de monitoramento da aplicação",
         "periodicidade": "mensal", "prazo": "até o 5º dia útil",
         "responsavel": "Área de Dados"},
        {"entrega": "Relatório executivo com resultados e insights",
         "consumidor": "Diretoria e patrocinador",
         "meio": "reports/relatorio-final.md e site MkDocs",
         "periodicidade": "a cada reciclagem, no mínimo anual",
         "prazo": "até 10 dias após a reciclagem", "responsavel": "Área de Dados"},
        {"entrega": "Leitura das métricas de negócio por linha de produto",
         "consumidor": "Diretoria Comercial",
         "meio": "reports/tables/implantacao/metricas-de-negocio.csv e o painel",
         "periodicidade": "trimestral", "prazo": "até 15 dias após o fim do trimestre",
         "responsavel": "Diretoria Comercial"},
    ]
)

io.salvar_tabela(entregas, SECAO, "entregas", indice=False)

registrar(
    "1 · Implantação",
    "Definir as sete entregas com consumidor, meio, prazo e responsável",
    "Área de Dados",
    "toda entrega tem destinatário nomeado e prazo declarado",
    f"{len(entregas)} entregas · 4 formatos (arquivo, painel, serviço e relatório)",
    acao="plano definido",
)
entregas

## 1.9 · A entrada em produção, em três janelas

Ligar as três linhas de produto no mesmo dia seria trocar uma política que funciona mal por
uma que ninguém testou. O plano tem três janelas, e cada uma só abre se a anterior passar no
critério declarado.

A janela 1 é a mais importante e a mais fácil de pular: **o modelo classifica e ninguém age.**
Ela existe para responder a uma pergunta que nenhuma métrica interna responde — a segmentação
diz algo diferente do que a política de duas faixas já dizia? Se a resposta for não, o projeto
não precisa entrar em produção, precisa voltar para a fase 4.

O plano de retorno é o mesmo nas três janelas, e é deliberadamente simples: **volta a política
de duas faixas em vigor.** Não existe "voltar para o modelo anterior" — o modelo anterior é o
de *k* = 4 que o bloco 0 mostrou não corresponder a nenhuma avaliação desta cadeia.

In [ ]:
janelas = pd.DataFrame(
    [
        {"janela": "1 · Sombra", "periodo": "D+0 a D+30", "inicio": 0, "duracao": 30,
         "escopo": "As três linhas classificadas; nenhuma ação comercial muda",
         "criterio_de_aceite": f"100% da carteira classificada · fronteira ≤ "
                               f"{TETO_OPERACIONAL:.0%} · zero rejeições sem motivo registrado "
                               f"· divergência contra a política de duas faixas medida por ARI",
         "quem_aprova": "Área de Dados",
         "plano_de_retorno": "nada a desfazer — nenhuma ação foi tomada"},
        {"janela": "2 · Piloto dirigido", "periodo": "D+31 a D+90", "inicio": 30, "duracao": 60,
         "escopo": "Ação diferenciada apenas no S3, e apenas em imóveis fora da fronteira",
         "criterio_de_aceite": "métrica de realizado ÷ anunciado capturada em ≥ 90% das vendas "
                               "do período · nenhuma unidade de fronteira com ação automática",
         "quem_aprova": "Diretoria Comercial",
         "plano_de_retorno": "suspender a ação do S3 e retomar a precificação anterior; a "
                             "classificação continua sendo produzida em sombra"},
        {"janela": "3 · Operação plena", "periodo": "D+91 em diante", "inicio": 90,
         "duracao": 90,
         "escopo": "As três linhas com orçamento próprio, teto de reforma e canal definidos",
         "criterio_de_aceite": "gatilhos de monitoramento ativos · primeira leitura mensal de "
                               "PSI publicada · métricas de negócio instrumentadas no sistema "
                               "comercial",
         "quem_aprova": "Diretoria e patrocinador",
         "plano_de_retorno": "retorno à política de duas faixas, com o motivo registrado no "
                             "manifesto da próxima reciclagem"},
    ]
)

fig, ax = plt.subplots(figsize=(14, 3.9))
cores_janela = [DESTAQUE, BARRA, config.AZUL_PROFUNDO]
for posicao, (_, linha) in enumerate(janelas.iterrows()):
    ax.barh(posicao, linha.duracao, left=linha.inicio, height=0.58, color=cores_janela[posicao])
    ax.text(linha.inicio + 2, posicao, f"{linha.janela}  ·  {linha.periodo}", va="center",
            ha="left", color="white", fontsize=9.5, fontweight="bold")
    ax.scatter(linha.inicio + linha.duracao, posicao, marker="D", s=70, zorder=5,
               color=REFERENCIA, edgecolor=config.AZUL_PROFUNDO, linewidth=1.1)
    ax.text(linha.inicio + linha.duracao + 3, posicao, "go / no-go", va="center", fontsize=8.5,
            color=config.AZUL_PROFUNDO, fontweight="bold")
ax.set_yticks([])
ax.set_xlim(0, 210)
ax.set_xticks([0, 30, 60, 90, 120, 150, 180])
ax.set_xticklabels(["D+0", "D+30", "D+60", "D+90", "D+120", "D+150", "D+180"])
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
for lado in ("top", "right", "left"):
    ax.spines[lado].set_visible(False)
ax.set_title("A entrada em produção — cada janela só abre se a anterior passar no critério")

salvar(fig, "janelas-de-implantacao", janelas)

registrar(
    "1 · Implantação",
    "Entrar em produção em três janelas, com go/no-go declarado",
    "Diretoria Comercial e Área de Dados",
    "cada janela só abre com o critério da anterior atendido e registrado",
    f"{len(janelas)} janelas · sombra de 30 dias antes de qualquer ação comercial",
    acao="plano definido",
)
janelas[["janela", "periodo", "escopo", "criterio_de_aceite", "quem_aprova"]]

# 2 · Planejar o monitoramento e a manutenção

*Plan monitoring and maintenance* — a segunda tarefa da fase 6, e a que separa um projeto
entregue de um projeto que continua funcionando.

O problema específico deste projeto está na primeira linha do plano: **não existe gabarito.**
Um modelo de previsão erra e o erro aparece — bastam o valor previsto e o realizado. Um
agrupamento não tem valor realizado. Se a carteira mudar e os três segmentos deixarem de
descrever a realidade, **nada na saída avisa**: a aplicação continua devolvendo S1, S2 e S3,
com a mesma cara de sempre.

O que se acompanha, então, são três coisas que mudam **antes** de o resultado ficar errado:

| O que muda | Nome | Como se mede aqui |
|:---|:---|:---|
| A distribuição da entrada | *data drift* | PSI de cada atributo contra a linha de base gravada na implantação |
| A coesão dos grupos | *cluster drift* | Silhueta calculada no período novo, com os centroides congelados |
| A composição da carteira | mudança de mix | Participação de cada linha de produto nas entradas do período |

E há uma quarta, que não é do modelo e sim da política: **o efeito da ação diferenciada**. O
agrupamento pode estar perfeito e a ação comercial, errada — e o plano precisa distinguir os
dois casos, porque a correção é completamente diferente.

## 2.1 · A linha de base

Monitorar exige uma referência. Guardar a base de treino para isso é o caminho errado: ela
envelhece, ocupa espaço e, em contextos com dado sensível, não pode ser retida. O que se grava
é a **distribuição** — decis, mediana, desvio e proporções — que é o suficiente para calcular
PSI e não permite reconstruir registro nenhum.

O arquivo `models/linha_de_base.json` existe hoje no repositório, com as distribuições da
carteira de 2.925 imóveis da cadeia anterior. Ele é sobrescrito aqui: monitorar o modelo
aprovado contra a referência de outro experimento produziria desvio onde não há, logo na
primeira leitura.

In [ ]:
COLUNAS_NUMERICAS = [coluna for coluna in ESPACO if coluna in dic.NUMERICAS]
COLUNAS_ORDINAIS = [coluna for coluna in ESPACO if coluna in dic.ORDINAIS]

linha_de_base = {
    coluna: {
        "media": float(base[coluna].mean()),
        "mediana": float(base[coluna].median()),
        "desvio": float(base[coluna].std()),
        "minimo": float(base[coluna].min()),
        "maximo": float(base[coluna].max()),
        "decis": [float(v) for v in base[coluna].quantile(np.linspace(0, 1, 11))],
    }
    for coluna in COLUNAS_NUMERICAS + COLUNAS_ORDINAIS
}
for coluna in dic.NOMINAIS:
    if coluna in base.columns:
        linha_de_base[coluna] = {
            "proporcoes": base[coluna].value_counts(normalize=True).round(5).to_dict()
        }
linha_de_base["_referencia"] = {
    "versao_modelo": VERSAO_MODELO,
    "imoveis": LINHAS,
    "periodo": manifesto_carga["periodo_venda"],
    "carimbo_do_pacote": CARIMBO_DO_MODELO,
    "composicao_por_segmento": {
        CATALOGO[s]["codigo"]: round(100 * float((ROTULOS == s).mean()), 2) for s in range(K)
    },
    "silhueta_de_referencia": SILHUETA,
    "pct_em_fronteira": round(100 * EM_FRONTEIRA / LINHAS, 2),
}

ARQUIVO_LINHA_DE_BASE = config.MODELS / "linha_de_base.json"
io.salvar_json(linha_de_base, ARQUIVO_LINHA_DE_BASE)

resumo_da_linha_de_base = pd.DataFrame(
    [
        {
            "atributo": dic.rotulo(coluna),
            "coluna": coluna,
            "mediana": round(linha_de_base[coluna]["mediana"], 2),
            "desvio": round(linha_de_base[coluna]["desvio"], 2),
            "decil_1": round(linha_de_base[coluna]["decis"][1], 2),
            "decil_9": round(linha_de_base[coluna]["decis"][9], 2),
        }
        for coluna in COLUNAS_NUMERICAS + COLUNAS_ORDINAIS
    ]
)

registrar(
    "2 · Monitoramento",
    "Regravar a linha de base sobre a carteira aprovada",
    "Área de Dados",
    "a referência de monitoramento é a mesma carteira em que o modelo foi ajustado",
    f"{len(resumo_da_linha_de_base)} atributos com decis · referência de "
    f"{pt(TREINO_LEGADO)} para {pt(LINHAS)} imóveis",
    acao="dívida paga",
    onde="models/linha_de_base.json",
)

io.salvar_tabela(resumo_da_linha_de_base, SECAO, "linha-de-base", indice=False)
resumo_da_linha_de_base.head(8)

## 2.2 · O desvio medido em dado real

Um plano de monitoramento escrito sem nunca ter sido calculado é uma hipótese sobre o próprio
plano: ninguém sabe se o limiar escolhido dispara toda semana ou nunca.

A carteira permite um teste honesto, sem perturbação simulada: ela cobre **2006 a 2010**, e o
calendário divide o período em dois. As vendas de **2006 a 2008** são a referência; as de
**2009 e 2010** são o período de acompanhamento. É o mesmo exercício que a operação fará todo
mês, com a diferença de que aqui o resultado já é conhecido — e o intervalo cobre exatamente o
choque imobiliário americano, que é o tipo de evento que um plano de monitoramento existe para
capturar.

Três medidas, uma por linha do plano: **PSI por atributo**, **composição por linha de produto**
e **participação ano a ano**.

In [ ]:
ANO_CORTE = 2009
referencia = (curada.ano_venda < ANO_CORTE).to_numpy()
acompanhamento = ~referencia
IMOVEIS_REFERENCIA, IMOVEIS_ACOMPANHAMENTO = int(referencia.sum()), int(acompanhamento.sum())

LIMIAR_PSI_ATENCAO, LIMIAR_PSI_ACAO = 0.10, 0.25

drift = pd.DataFrame(
    [
        {
            "atributo": dic.rotulo(coluna),
            "coluna": coluna,
            "psi": round(float(psi(base.loc[referencia, coluna], base.loc[acompanhamento, coluna])), 4),
            "mediana_referencia": round(float(base.loc[referencia, coluna].median()), 2),
            "mediana_acompanhamento": round(float(base.loc[acompanhamento, coluna].median()), 2),
        }
        for coluna in COLUNAS_NUMERICAS + COLUNAS_ORDINAIS
    ]
).sort_values("psi", ascending=False)
drift["situacao"] = np.where(
    drift.psi >= LIMIAR_PSI_ACAO, "ação",
    np.where(drift.psi >= LIMIAR_PSI_ATENCAO, "atenção", "estável"),
)

PSI_MAXIMO = float(drift.psi.max())
ATRIBUTOS_EM_ACAO = int((drift.situacao == "ação").sum())
ATRIBUTOS_EM_ATENCAO = int((drift.situacao == "atenção").sum())

composicao = pd.DataFrame(
    {
        "referência 2006–2008": 100 * pd.Series(ROTULOS[referencia]).value_counts(normalize=True),
        "acompanhamento 2009–2010": 100 * pd.Series(ROTULOS[acompanhamento]).value_counts(
            normalize=True
        ),
    }
).sort_index()
composicao.index = [CATALOGO[int(s)]["codigo"] for s in composicao.index]
composicao["variação (p.p.)"] = (
    composicao["acompanhamento 2009–2010"] - composicao["referência 2006–2008"]
).round(2)
MAIOR_VARIACAO_DE_MIX = float(composicao["variação (p.p.)"].abs().max())

participacao = (
    pd.crosstab(curada.ano_venda, [CATALOGO[int(s)]["codigo"] for s in ROTULOS], normalize="index")
    .mul(100)
    .round(2)
)

print(f"Referência: {pt(IMOVEIS_REFERENCIA)} imóveis (2006–2008) · "
      f"acompanhamento: {pt(IMOVEIS_ACOMPANHAMENTO)} imóveis (2009–2010)")
print(f"PSI máximo: {PSI_MAXIMO:.4f} em `{drift.iloc[0].coluna}` "
      f"(limiar de atenção {LIMIAR_PSI_ATENCAO}, de ação {LIMIAR_PSI_ACAO})")
print(f"Maior variação de composição: {MAIOR_VARIACAO_DE_MIX:.2f} p.p.")
drift.head(6)

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(17.5, 5.6), constrained_layout=True)

cores_drift = [
    ALERTA if s == "ação" else REFERENCIA if s == "atenção" else APROVADO
    for s in drift.situacao
]
eixos[0].barh(drift.atributo[::-1], drift.psi[::-1], color=cores_drift[::-1])
eixos[0].axvline(LIMIAR_PSI_ATENCAO, ls="dashed", color=REFERENCIA, lw=2)
eixos[0].axvline(LIMIAR_PSI_ACAO, ls="dashed", color=ALERTA, lw=2)
eixos[0].text(LIMIAR_PSI_ATENCAO, len(drift) - 0.6, " atenção", fontsize=8,
              color=config.AZUL_PROFUNDO)
eixos[0].text(LIMIAR_PSI_ACAO, len(drift) - 0.6, " ação", fontsize=8, color=config.AZUL_PROFUNDO)
eixos[0].set_xlim(0, LIMIAR_PSI_ACAO * 1.15)
eixos[0].set_xlabel("PSI — referência 2006–2008 contra 2009–2010")
eixos[0].tick_params(axis="y", labelsize=8)
eixos[0].set_title("Data drift: nenhum atributo perto do limiar")

posicoes = np.arange(len(composicao))
eixos[1].bar(posicoes - 0.2, composicao["referência 2006–2008"], width=0.4, color=NEUTRO,
             label="2006–2008")
eixos[1].bar(posicoes + 0.2, composicao["acompanhamento 2009–2010"], width=0.4,
             color=[CORES_SEGMENTO[s] for s in range(K)], label="2009–2010")
for posicao, variacao in enumerate(composicao["variação (p.p.)"]):
    eixos[1].text(posicao, 2, f"{variacao:+.1f} p.p.", ha="center", fontsize=8.5,
                  color="white", fontweight="bold")
eixos[1].set_xticks(posicoes)
eixos[1].set_xticklabels(composicao.index)
eixos[1].set_ylabel("% das vendas do período")
eixos[1].set_title("Composição por linha de produto")
eixos[1].legend(fontsize=8.5)

for codigo, cor in zip(participacao.columns, [CORES_SEGMENTO[s] for s in range(K)]):
    eixos[2].plot(participacao.index, participacao[codigo], marker="o", lw=2.2, color=cor,
                  label=codigo)
eixos[2].axvline(ANO_CORTE - 0.5, ls="dashed", color=NEUTRO)
eixos[2].text(ANO_CORTE - 0.45, participacao.to_numpy().max() * 0.99, " corte de monitoramento",
              fontsize=8, color=NEUTRO, va="top")
eixos[2].set_xticks(sorted(participacao.index))
eixos[2].set_ylabel("% das vendas do ano")
eixos[2].set_title("Participação ano a ano — o mix não virou")
eixos[2].legend(fontsize=8.5, ncols=3)

fig.suptitle("Monitoramento sem gabarito: o que se acompanha é a entrada e a composição",
             fontsize=13, fontweight="bold", color=config.AZUL_PROFUNDO)

salvar(fig, "desvio-medido", drift)
io.salvar_tabela(composicao.reset_index(names="linha de produto"), SECAO,
                 "composicao-por-periodo", indice=False)
io.salvar_tabela(participacao.reset_index(), SECAO, "participacao-por-ano", indice=False)
composicao

## 2.3 · A coesão dos grupos e a reciclagem simulada

O *data drift* mede a entrada. Falta a pergunta que interessa: **os grupos continuam
descrevendo a carteira?** Um atributo pode não se mover e a estrutura, sim — basta que as
combinações mudem sem que as distribuições marginais mudem.

Duas medidas respondem:

* **silhueta no período novo, com os centroides congelados.** Se cair, os imóveis novos estão
  chegando cada vez mais longe dos centros que a fase 4 encontrou;
* **a reciclagem simulada.** Reajusta-se o agrupador apenas sobre as vendas de 2009–2010, com
  o preparo congelado, e compara-se a partição resultante com a vigente por ARI. É exatamente
  o procedimento que a manutenção vai executar — feito aqui uma vez, para saber o que ele
  devolve quando **não** há motivo para reciclar.

A segunda medida tem um controle junto: reajustar sobre a carteira inteira, com outra semente.
Se isso devolver uma partição diferente, o problema não é a carteira — é o modelo, e nenhum
gatilho de drift serviria para nada.

In [ ]:
coesao = pd.DataFrame(
    [
        {
            "período": nome,
            "imóveis": int(marcador.sum()),
            "silhueta": round(float(silhouette_score(EUCLIDIANO[marcador], ROTULOS[marcador])), 4),
            "davies_bouldin": round(
                part.metricas_internas(EUCLIDIANO[marcador], ROTULOS[marcador])["davies_bouldin"], 3
            ),
        }
        for nome, marcador in (
            ("Carteira inteira (2006–2010)", np.ones(LINHAS, dtype=bool)),
            ("Referência (2006–2008)", referencia),
            ("Acompanhamento (2009–2010)", acompanhamento),
        )
    ]
).set_index("período")

SILHUETA_ACOMPANHAMENTO = float(coesao.loc["Acompanhamento (2009–2010)", "silhueta"])
SILHUETA_REFERENCIA = float(coesao.loc["Referência (2006–2008)", "silhueta"])
PISO_SILHUETA = float(manifesto_modelagem["criterios"]["silhueta_minima"])
PISO_ARI = float(manifesto_modelagem["criterios"]["ari_minimo"])

# A reciclagem simulada: preparo congelado, agrupador reajustado na janela recente.
reciclado_na_janela = KMeans(
    n_clusters=K, init="k-means++", n_init=20, random_state=SEMENTE
).fit_predict(EUCLIDIANO[acompanhamento])
ARI_RECICLAGEM = round(
    float(adjusted_rand_score(reciclado_na_janela, ROTULOS[acompanhamento])), 4
)

# O controle: reajuste sobre a carteira inteira, com outra semente.
reciclado_na_carteira = KMeans(
    n_clusters=K, init="k-means++", n_init=20, random_state=SEMENTE + 7
).fit_predict(EUCLIDIANO)
ARI_CONTROLE = round(float(adjusted_rand_score(reciclado_na_carteira, ROTULOS)), 4)

fig, eixos = plt.subplots(1, 2, figsize=(14.5, 4.8),
                          gridspec_kw={"width_ratios": [1.15, 1]}, constrained_layout=True)

posicoes = np.arange(len(coesao))
barras = eixos[0].barh(posicoes, coesao.silhueta, color=[NEUTRO, DESTAQUE, BARRA], height=0.6)
eixos[0].axvline(PISO_SILHUETA, ls="dashed", color=REFERENCIA, lw=2.2)
eixos[0].text(PISO_SILHUETA, len(coesao) - 0.35, f" piso do Canvas: {PISO_SILHUETA:.2f}",
              fontsize=8.5, color=config.AZUL_PROFUNDO, va="center")
eixos[0].bar_label(barras, labels=[f"{v:.4f}  ({pt(n)} imóveis)"
                                   for v, n in zip(coesao.silhueta, coesao["imóveis"])],
                   padding=4, fontsize=8.5)
eixos[0].set_yticks(posicoes)
eixos[0].set_yticklabels(coesao.index, fontsize=9)
eixos[0].invert_yaxis()
eixos[0].set_xlim(0, max(coesao.silhueta.max(), PISO_SILHUETA) * 1.9)
eixos[0].set_xlabel("Silhueta com os centroides congelados")
eixos[0].set_title("Cluster drift: a coesão não caiu no período novo")

nomes_ari = ["Reciclagem na janela\n2009–2010", "Controle: reajuste na\ncarteira inteira"]
valores_ari = [ARI_RECICLAGEM, ARI_CONTROLE]
barras_ari = eixos[1].bar(nomes_ari, valores_ari, color=[BARRA, NEUTRO], width=0.55)
eixos[1].axhline(PISO_ARI, ls="dashed", color=ALERTA, lw=2.2)
eixos[1].text(-0.45, PISO_ARI + 0.015, f"piso de estabilidade do Canvas: {PISO_ARI:.2f} — "
              f"abaixo disso, a reciclagem vira novo ciclo", fontsize=8.5, color=ALERTA)
eixos[1].bar_label(barras_ari, fmt="%.4f", padding=4, fontsize=10, fontweight="bold")
eixos[1].set_ylim(0, 1.15)
eixos[1].set_ylabel("ARI contra a partição vigente")
eixos[1].set_title("A reciclagem simulada — o que o gatilho devolveria hoje")

fig.suptitle("A estrutura envelheceu? As duas medidas dizem que não",
             fontsize=13, fontweight="bold", color=config.AZUL_PROFUNDO)

salvar(fig, "coesao-e-reciclagem", coesao.reset_index())

registrar(
    "2 · Monitoramento",
    "Medir o desvio no dado real, e não em perturbação simulada",
    "Área de Dados",
    "o plano de monitoramento é publicado com a primeira leitura já executada",
    f"PSI máximo {PSI_MAXIMO:.4f} · silhueta {SILHUETA_REFERENCIA:.4f} → "
    f"{SILHUETA_ACOMPANHAMENTO:.4f} · ARI da reciclagem {ARI_RECICLAGEM:.4f}",
)
coesao

### O que a primeira leitura mostrou

Nenhum dos três indicadores dispara — e isso é informação, não ausência dela. O período de
acompanhamento cobre 2009 e 2010, o fundo do choque imobiliário americano: se houvesse um
intervalo em que a carteira de Ames mudasse de cara, seria esse.

O que **mudou** foi o preço, e o que **não mudou** foi o produto: a composição por atributo
físico ficou estável, a coesão dos grupos subiu levemente no período novo e a reciclagem
devolveu praticamente a mesma partição. É uma propriedade útil da decisão da fase 3 de
segmentar por **atributo do imóvel** e não por preço — o segmento de um imóvel não muda porque
o mercado caiu, e é por isso que a política construída sobre ele sobrevive ao ciclo.

Isso também calibra a expectativa sobre o plano abaixo: **os gatilhos vão passar a maior parte
do tempo em verde.** Um plano cujo alarme dispara todo mês é ruído; um que nunca poderia
disparar é decoração. O teste acima mostra que este fica no primeiro caso apenas se a carteira
mudar de verdade.

## 2.4 · O plano de monitoramento — o que dispara, quem responde

Seis indicadores. Cada linha tem **limiar numérico**, **periodicidade**, **ação** e
**responsável nomeado** — e a coluna que costuma faltar nos planos de monitoramento: o valor
medido **hoje**, para que a primeira leitura já exista quando a operação começar.

Um detalhe de desenho que vale explicitar: o indicador 6 não monitora o modelo, monitora a
**política**. Ele existe porque a resposta correta para "a meta de giro não foi atingida" pode
ser *revisar a ação comercial* e não *reciclar o modelo* — e sem essa linha, toda frustração
de meta viraria pedido de remodelagem.

In [ ]:
plano = pd.DataFrame(
    [
        {"indicador": "1 · Data drift dos atributos",
         "como_se_mede": "PSI de cada um dos 17 atributos contra models/linha_de_base.json",
         "periodicidade": "mensal",
         "limiar": f"PSI ≥ {LIMIAR_PSI_ACAO} em qualquer atributo, ou ≥ {LIMIAR_PSI_ATENCAO} "
                   f"em três simultaneamente",
         "valor_hoje": f"máximo {PSI_MAXIMO:.4f} ({drift.iloc[0].atributo})",
         "acao": "reciclar o agrupador e comparar com a partição vigente por ARI",
         "responsavel": "Área de Dados"},
        {"indicador": "2 · Cluster drift — coesão",
         "como_se_mede": "silhueta das entradas dos últimos 12 meses, centroides congelados",
         "periodicidade": "trimestral",
         "limiar": f"silhueta < {PISO_SILHUETA:.2f} por dois trimestres consecutivos",
         "valor_hoje": f"{SILHUETA_ACOMPANHAMENTO:.4f} no período de acompanhamento",
         "acao": "reciclagem obrigatória, com nova varredura de k",
         "responsavel": "Área de Dados"},
        {"indicador": "3 · Composição da carteira",
         "como_se_mede": "participação de cada linha de produto nas entradas do período",
         "periodicidade": "mensal",
         "limiar": "variação ≥ 10 p.p. em qualquer linha contra a linha de base",
         "valor_hoje": f"maior variação {MAIOR_VARIACAO_DE_MIX:.2f} p.p.",
         "acao": "revisar o orçamento por linha — o mix mudou e a alocação envelheceu",
         "responsavel": "Diretoria Comercial"},
        {"indicador": "4 · Imóveis de fronteira",
         "como_se_mede": "percentual das entradas com razão de fronteira ≥ "
                         f"{CORTE_FRONTEIRA:.3f}",
         "periodicidade": "mensal",
         "limiar": f"acima de {TETO_OPERACIONAL:.0%} das entradas do mês",
         "valor_hoje": f"{100 * EM_FRONTEIRA / LINHAS:.1f}% da carteira",
         "acao": "suspender a ação automática desses imóveis e escalar para avaliação humana",
         "responsavel": "Gerência de Ativos"},
        {"indicador": "5 · Volume e integridade da carga",
         "como_se_mede": "contagem de registros, rejeições pelo contrato e carimbo do pacote",
         "periodicidade": "a cada carga",
         "limiar": "queda de volume > 30%, rejeição > 5% dos registros, ou carimbo divergente",
         "valor_hoje": f"{pt(LINHAS)} imóveis, 0 rejeições, carimbo confere",
         "acao": "bloquear a publicação e acionar a origem — a classificação anterior "
                 "permanece vigente",
         "responsavel": "Engenharia de Dados"},
        {"indicador": "6 · Efeito da ação diferenciada",
         "como_se_mede": "as métricas declaradas em cada linha de produto (bloco 2.6)",
         "periodicidade": "trimestral",
         "limiar": "meta não atingida em dois trimestres consecutivos",
         "valor_hoje": "sem leitura — depende da política em operação",
         "acao": "revisar a ação, não o modelo: o agrupamento pode estar certo e a ação, errada",
         "responsavel": "Diretoria Comercial"},
    ]
)

io.salvar_tabela(plano, SECAO, "plano-de-monitoramento", indice=False)

# O painel de saúde: a margem de cada indicador até o seu limiar, na direção que
# importa. Margem positiva é folga; negativa significa gatilho disparado.
painel = pd.DataFrame(
    [
        {"indicador": "Data drift (PSI máximo)", "valor": PSI_MAXIMO, "limiar": LIMIAR_PSI_ACAO,
         "sentido": "teto"},
        {"indicador": "Atributos em atenção", "valor": float(ATRIBUTOS_EM_ATENCAO),
         "limiar": 3.0, "sentido": "teto"},
        {"indicador": "Coesão no período novo", "valor": SILHUETA_ACOMPANHAMENTO,
         "limiar": PISO_SILHUETA, "sentido": "piso"},
        {"indicador": "Variação de composição (p.p.)", "valor": MAIOR_VARIACAO_DE_MIX,
         "limiar": 10.0, "sentido": "teto"},
        {"indicador": "Imóveis em fronteira (%)", "valor": 100 * EM_FRONTEIRA / LINHAS,
         "limiar": 100 * TETO_OPERACIONAL, "sentido": "teto"},
        {"indicador": "ARI da reciclagem", "valor": ARI_RECICLAGEM, "limiar": PISO_ARI,
         "sentido": "piso"},
    ]
)
painel["margem"] = np.where(
    painel.sentido == "teto",
    1 - painel.valor / painel.limiar,
    1 - painel.limiar / painel.valor.replace(0, np.nan),
).round(3)
painel["situacao"] = np.where(painel.margem < 0, "gatilho disparado",
                              np.where(painel.margem < 0.10, "margem estreita", "folga"))

fig, ax = plt.subplots(figsize=(12.5, 4.6))
cores_painel = [
    ALERTA if s == "gatilho disparado" else REFERENCIA if s == "margem estreita" else APROVADO
    for s in painel.situacao
]
posicoes = np.arange(len(painel))
ax.barh(posicoes, painel.margem, color=cores_painel, height=0.62)
ax.axvline(0, color=config.AZUL_PROFUNDO, lw=1.6)
for posicao, linha in painel.iterrows():
    ax.text(linha.margem + 0.015, posicao,
            f"{linha.valor:.3g} contra {linha.sentido} de {linha.limiar:.3g}",
            va="center", fontsize=8.6, color=config.AZUL_PROFUNDO)
ax.set_yticks(posicoes)
ax.set_yticklabels(painel.indicador, fontsize=9)
ax.invert_yaxis()
ax.set_xlim(-0.35, 1.55)
ax.set_xlabel("Margem até o limiar — positiva é folga, negativa é gatilho disparado")
ax.set_title("Painel de saúde do modelo — a primeira leitura, publicada junto com o plano")

salvar(fig, "painel-de-saude", painel)

registrar(
    "2 · Monitoramento",
    "Publicar o plano com seis indicadores, limiar e responsável",
    "Área de Dados",
    "todo indicador tem limiar numérico, periodicidade, ação e dono nomeado",
    f"{len(plano)} indicadores · {int((painel.situacao == 'folga').sum())} em folga na "
    f"primeira leitura",
    acao="gatilho definido",
)
plano[["indicador", "periodicidade", "limiar", "valor_hoje", "responsavel"]]

## 2.5 · Quando o modelo é refeito — e o que decide o que fazer com o resultado

Este é o **item 8 do backlog da fase 5**. Um gatilho de reciclagem tem duas partes, e planos
costumam trazer só a primeira: *quando refazer* e **o que fazer com o que sair**.

A segunda parte é a que evita o pior desfecho possível — reciclar, obter uma partição
diferente e trocar as personas silenciosamente, deixando a operação com nomes que a Diretoria
nunca aprovou. O critério de aceite abaixo amarra o resultado da reciclagem a uma decisão de
negócio, medida por ARI contra a partição vigente, com o mesmo piso de estabilidade que a
fase 4 usou para aceitar o modelo.

In [ ]:
reciclagem = pd.DataFrame(
    [
        {"gatilho": "Calendário", "condicao": "a cada 12 meses, independentemente de desvio",
         "procedimento": "reexecutar a cadeia 00–06 sobre a base acumulada e comparar a nova "
                         "partição com a vigente por ARI",
         "responsavel": "Área de Dados"},
        {"gatilho": "Data drift", "condicao": f"PSI ≥ {LIMIAR_PSI_ACAO} em qualquer atributo",
         "procedimento": "reciclagem imediata, com registro do atributo que disparou",
         "responsavel": "Área de Dados"},
        {"gatilho": "Cluster drift",
         "condicao": f"silhueta < {PISO_SILHUETA:.2f} por dois trimestres consecutivos",
         "procedimento": "reciclagem com nova varredura de k, incluindo teste de estabilidade",
         "responsavel": "Área de Dados"},
        {"gatilho": "Mudança de escopo",
         "condicao": "entrada de novo tipo de ativo na carteira (comercial, terreno, multi-família)",
         "procedimento": "revisão do espaço de atributos antes de qualquer reciclagem — o "
                         "dicionário de dados muda primeiro",
         "responsavel": "Área de Dados com a Diretoria"},
        {"gatilho": "Decisão comercial",
         "condicao": "mudança de estratégia que altere o teto de reforma ou o canal de uma linha",
         "procedimento": "revisão do catálogo de personas sem reciclar o modelo — versão de "
                         "correção",
         "responsavel": "Diretoria Comercial"},
    ]
)

criterio_de_aceite = pd.DataFrame(
    [
        {"resultado_da_reciclagem": "ARI ≥ 0,80 com a partição vigente",
         "interpretacao": "mesma estrutura, centroides deslocados",
         "decisao": "versão MENOR: atualizar centroides, manter personas e ações",
         "quem_aprova": "Área de Dados"},
        {"resultado_da_reciclagem": f"ARI entre {PISO_ARI:.2f} e 0,80",
         "interpretacao": "estrutura deslocada",
         "decisao": "versão MENOR com revisão das linhas afetadas antes da publicação",
         "quem_aprova": "Diretoria Comercial"},
        {"resultado_da_reciclagem": f"ARI < {PISO_ARI:.2f}",
         "interpretacao": "estrutura diferente — o piso de estabilidade do Canvas foi rompido",
         "decisao": "versão MAIOR: novo ciclo de CRISP-DM a partir do entendimento de negócio, "
                    "com o catálogo de personas reescrito",
         "quem_aprova": "Diretoria e patrocinador"},
    ]
)

io.salvar_tabela(reciclagem, SECAO, "gatilhos-de-reciclagem", indice=False)
io.salvar_tabela(criterio_de_aceite, SECAO, "criterio-de-aceite-da-reciclagem", indice=False)

registrar(
    "2 · Manutenção",
    "Definir o gatilho de reciclagem e o critério de aceite do resultado",
    "Área de Dados",
    tarefa_da_fase_5(8).criterio_de_conclusao,
    f"{len(reciclagem)} gatilhos · 3 faixas de decisão medidas por ARI, com piso "
    f"{PISO_ARI:.2f} lido do manifesto da fase 4 · simulação de hoje devolveu "
    f"{ARI_RECICLAGEM:.4f}",
    acao="gatilho definido",
    onde=tarefa_da_fase_5(8).onde,
)
criterio_de_aceite

## 2.6 · As métricas de negócio — o que ainda não dá para medir

Este é o **item 5 do backlog da fase 5**. Cinco critérios de negócio do Canvas ficaram
pendentes na avaliação, todos pela mesma razão: **exigem a política em operação**. Nenhum
deles é medível no histórico, e fingir o contrário seria a forma mais fácil de encerrar o
projeto com um painel verde e nenhum efeito.

A tabela abaixo lê os critérios diretamente do que a fase 5 gravou — não os redigita — e
acrescenta a única coisa que faltava para que eles saiam do papel: **o campo que precisa
passar a existir no sistema comercial**, com quem responde pela captura.

É a diferença entre "instrumentar as métricas" e ter escrito que se pretende instrumentá-las:
apenas um dos cinco critérios é medível hoje. Três dependem de um dado que **a Meridiano não
coleta** — preço anunciado, data de anúncio e custo de reforma executada — e o quinto depende
de uma decisão que ainda não começou, a do destino unidade a unidade do estoque herdado.
Enquanto esses três campos não existirem, os critérios correspondentes permanecem sem leitura,
e o painel precisa mostrar isso como *sem leitura*, e não como zero.

In [ ]:
criterios_do_canvas = ler_tabela("avaliacao", "criterios-do-canvas")
pendentes = criterios_do_canvas[
    criterios_do_canvas.veredito.str.contains("pendente", case=False, na=False)
].copy()

INSTRUMENTACAO = {
    "linha de produto": {
        "metrica_operacional": "% do estoque com linha de produto atribuída",
        "campo_necessario": "nenhum — já é produzido pela carga",
        "disponivel_hoje": True,
        "linha_de_base": f"{pt(LINHAS)} de {pt(LINHAS)} imóveis (100%), "
                         f"{pt(EM_FRONTEIRA)} marcados para revisão",
        "responsavel": "Área de Dados",
        "prazo": "janela 1 (D+30)",
    },
    "reforma": {
        "metrica_operacional": "ganho de preço realizado ÷ custo da reforma, por linha",
        "campo_necessario": "`custo_reforma_executada` e `data_conclusao_reforma` no sistema "
                            "comercial",
        "disponivel_hoje": False,
        "linha_de_base": "sem leitura — a base histórica não traz custo de reforma",
        "responsavel": "Diretoria Comercial",
        "prazo": "janela 2 (D+90)",
    },
    "giro": {
        "metrica_operacional": "dias entre anúncio e venda, mediana por linha",
        "campo_necessario": "`data_anuncio` — a base traz apenas mês e ano da venda",
        "disponivel_hoje": False,
        "linha_de_base": "sem leitura — não existe data de anúncio no histórico",
        "responsavel": "Diretoria Comercial",
        "prazo": "janela 2 (D+90)",
    },
    "anunciado": {
        "metrica_operacional": "preço realizado ÷ preço anunciado, mediana por linha",
        "campo_necessario": "`preco_anunciado` no momento da publicação do imóvel",
        "disponivel_hoje": False,
        "linha_de_base": "sem leitura — o histórico só tem o preço realizado",
        "responsavel": "Diretoria Comercial",
        "prazo": "janela 2 (D+90) — é o critério que a janela 2 valida",
    },
    "destino": {
        "metrica_operacional": "% do S1 com destino definido (venda, renda ou desinvestimento)",
        "campo_necessario": "`destino_definido` e `data_da_decisao` por unidade",
        "disponivel_hoje": False,
        "linha_de_base": "0% — a decisão unidade a unidade ainda não começou",
        "responsavel": "Gerência de Ativos",
        "prazo": "janela 3 (D+91 em diante), com prazo final de 12 meses",
    },
}


def instrumentar(criterio: str) -> dict:
    """Associa cada critério pendente do Canvas ao campo que falta para medi-lo."""
    texto = criterio.lower()
    for chave in INSTRUMENTACAO:
        if chave in texto:
            return INSTRUMENTACAO[chave]
    return {"metrica_operacional": "—", "campo_necessario": "—", "disponivel_hoje": False,
            "linha_de_base": "sem leitura", "responsavel": "Área de Dados", "prazo": "—"}


metricas_de_negocio = pd.concat(
    [
        pendentes[["criterio"]].reset_index(drop=True),
        pd.DataFrame([instrumentar(c) for c in pendentes.criterio]),
    ],
    axis=1,
)
SEM_LEITURA_HOJE = int((~metricas_de_negocio.disponivel_hoje).sum())

io.salvar_tabela(metricas_de_negocio, SECAO, "metricas-de-negocio", indice=False)

registrar(
    "2 · Monitoramento",
    "Instrumentar os critérios de negócio do Canvas",
    "Diretoria Comercial",
    tarefa_da_fase_5(5).criterio_de_conclusao,
    f"{len(metricas_de_negocio)} critérios · {SEM_LEITURA_HOJE} dependem de campo que o "
    f"sistema comercial ainda não coleta",
    acao="plano definido",
    onde=tarefa_da_fase_5(5).onde,
)
metricas_de_negocio[["criterio", "metrica_operacional", "campo_necessario", "prazo"]]

## 2.7 · A manutenção do ambiente operacional

Monitorar o modelo não basta se o ambiente ao redor apodrecer. A tabela de rotinas abaixo
cobre o que mantém a cadeia executável — e uma delas é específica deste projeto e não deve ser
lida como burocracia: **a aposentadoria dos artefatos da cadeia anterior**. Enquanto
`pipeline_segmentacao.joblib` e `metadados_segmentacao.json` continuarem no diretório
`models/`, existe um caminho pelo qual a aplicação carrega o modelo errado — e o bloco 0
mediu o custo disso em 280 imóveis.

O segundo bloco é o **runbook**: o que fazer quando algo quebra, escrito antes de quebrar. A
coluna que mais importa é a última — *o que a operação faz enquanto isso* —, porque a resposta
padrão a qualquer incidente deste modelo é a mesma e precisa estar escrita: **a classificação
vigente permanece**, e nada é reclassificado com dado suspeito.

In [ ]:
manutencao = pd.DataFrame(
    [
        {"rotina": "Reexecução da cadeia 00–06", "periodicidade": "anual ou por gatilho",
         "como": "`uv run invoke notebooks`, com a semente do projeto e o ambiente do uv.lock",
         "evidencia": "manifestos regravados e ARI contra a partição vigente",
         "responsavel": "Área de Dados"},
        {"rotina": "Aposentadoria dos artefatos da cadeia anterior", "periodicidade": "uma vez",
         "como": "remover `pipeline_segmentacao.joblib` e `metadados_segmentacao.json`, e "
                 "reapontar `src/deployment/scoring.py` para o pacote carimbado",
         "evidencia": "nenhum artefato de k = 4 em models/",
         "responsavel": "Área de Dados"},
        {"rotina": "Desativar a regravação da linha de base pela cadeia anterior",
         "periodicidade": "uma vez",
         "como": "`src/pipeline/implantacao.py` grava `models/linha_de_base.json` com a "
                 "referência de k = 4; enquanto `uv run invoke implantacao` puder ser "
                 "executado, ele apaga a linha de base publicada aqui",
         "evidencia": "o SHA-256 da linha de base confere com o de metadados_implantacao.json "
                      "depois de rodar a suíte de tarefas",
         "responsavel": "Área de Dados"},
        {"rotina": "Verificação do carimbo na carga", "periodicidade": "a cada carga",
         "como": "recalcular o SHA-256 do pacote e comparar com metadados_implantacao.json",
         "evidencia": "log da carga com o resumo conferido",
         "responsavel": "Engenharia de Dados"},
        {"rotina": "Teste automatizado do catálogo", "periodicidade": "a cada commit",
         "como": "teste que compara o número de personas ao k do manifesto e valida os campos "
                 "obrigatórios de cada linha de produto",
         "evidencia": "suíte verde em `uv run invoke test`",
         "responsavel": "Área de Dados"},
        {"rotina": "Congelamento de dependências", "periodicidade": "trimestral",
         "como": "revisar `uv.lock`; atualização de scikit-learn exige reexecutar a cadeia e "
                 "conferir o ARI contra a partição vigente",
         "evidencia": "ARI ≥ 0,99 após a atualização, ou reciclagem registrada",
         "responsavel": "Área de Dados"},
        {"rotina": "Retenção de versões", "periodicidade": "a cada publicação",
         "como": "manter as duas últimas versões do pacote, com manifesto e carimbo",
         "evidencia": "diretório de versões com no mínimo o par vigente e anterior",
         "responsavel": "Engenharia de Dados"},
        {"rotina": "Publicação da documentação", "periodicidade": "a cada versão",
         "como": "`uv run invoke build` após atualizar as páginas afetadas",
         "evidencia": "site sem ocorrência do resultado anterior",
         "responsavel": "Área de Dados"},
    ]
)

runbook = pd.DataFrame(
    [
        {"incidente": "A carga não chegou ou veio com volume anômalo",
         "primeiro_passo": "comparar a contagem com a carga anterior e acionar a origem",
         "aciona": "Engenharia de Dados", "prazo": "mesmo dia",
         "enquanto_isso": "a classificação vigente permanece; nada é reclassificado"},
        {"incidente": "Mais de 5% dos registros rejeitados pelo contrato",
         "primeiro_passo": "abrir a lista de violações por campo — é erro de cadastro, não "
                           "de modelo",
         "aciona": "Engenharia de Dados", "prazo": "48 horas",
         "enquanto_isso": "os registros aceitos são classificados; os rejeitados ficam "
                          "pendentes com o motivo"},
        {"incidente": "PSI em nível de ação em algum atributo",
         "primeiro_passo": "verificar se é mudança real ou erro de origem (unidade, escala, "
                           "domínio novo)",
         "aciona": "Área de Dados", "prazo": "5 dias úteis",
         "enquanto_isso": "classificação continua, com o alerta visível no painel"},
        {"incidente": "Silhueta abaixo do piso por dois trimestres",
         "primeiro_passo": "reciclagem com nova varredura de k e teste de estabilidade",
         "aciona": "Área de Dados", "prazo": "30 dias",
         "enquanto_isso": "as ações comerciais seguem; a revisão do catálogo aguarda o ARI"},
        {"incidente": "O carimbo do pacote não confere",
         "primeiro_passo": "interromper a carga e restaurar o artefato da versão registrada",
         "aciona": "Área de Dados", "prazo": "imediato",
         "enquanto_isso": "nenhuma classificação nova é publicada — é o único incidente que "
                          "para a operação"},
        {"incidente": "A aplicação está fora do ar",
         "primeiro_passo": "reiniciar o serviço; a carteira classificada em arquivo continua "
                           "disponível",
         "aciona": "Engenharia de Dados", "prazo": "4 horas",
         "enquanto_isso": "consumo pelo arquivo `carteira_classificada.parquet`"},
    ]
)

io.salvar_tabela(manutencao, SECAO, "rotinas-de-manutencao", indice=False)
io.salvar_tabela(runbook, SECAO, "runbook-de-incidentes", indice=False)

registrar(
    "2 · Manutenção",
    "Documentar as rotinas do ambiente e o runbook de incidentes",
    "Engenharia de Dados e Área de Dados",
    "todo incidente previsto tem primeiro passo, prazo e quem aciona",
    f"{len(manutencao)} rotinas · {len(runbook)} incidentes com procedimento escrito",
    acao="plano definido",
)
runbook[["incidente", "primeiro_passo", "aciona", "prazo"]]

# 3 · Produzir o relatório final

*Produce final report* — a terceira tarefa da fase 6. O CRISP-DM pede duas coisas diferentes
aqui, e confundi-las é o erro mais comum do encerramento: o **relatório**, que é o registro
completo e permanente, e a **apresentação**, que é o recorte de trinta minutos para quem decide.

O relatório precisa sobreviver à saída de todo mundo que participou. A apresentação precisa
caber numa reunião de diretoria. São públicos e prazos diferentes — e este bloco produz os dois,
mais o arquivo que a Meridiano leva embora.

## 3.1 · A metodologia, em uma tabela

O que cada fase decidiu e com que evidência. A tabela abaixo é **lida dos registros das
próprias fases**, não redigitada: as contagens vêm dos arquivos que cada notebook gravou. Se
uma fase for reexecutada e mudar de conclusão, esta linha muda junto.

In [ ]:
registros_por_fase = {
    "1 · Entendimento do negócio": None,
    "2 · Entendimento e ajuste dos dados": ler_tabela("ajustes-dados", "registro-de-ajustes"),
    "3 · Preparação dos dados": ler_tabela("preparacao", "registro-de-decisoes"),
    "4 · Modelagem": ler_tabela("modelagem", "registro-de-decisoes"),
    "5 · Avaliação": ler_tabela("avaliacao", "registro-de-julgamentos"),
}
insights_da_eda = ler_tabela("eda-exploratoria", "insights")

metodologia = pd.DataFrame(
    [
        {"fase": "1 · Entendimento do negócio",
         "pergunta": "Que problema a segmentação resolve, e como se sabe que resolveu?",
         "o_que_decidiu": "Canvas do Problema com objetivo, critérios técnicos e de negócio, "
                          "e cinco perguntas SMART",
         "registros": len(criterios_do_canvas),
         "evidencia": "docs/pbl/canvas-do-problema.md"},
        {"fase": "2 · Entendimento e ajuste dos dados",
         "pergunta": "A base descreve a carteira que a diretoria quer segmentar?",
         "o_que_decidiu": f"{pt(manifesto_curadoria['linhas_entrada'])} registros brutos → "
                          f"{pt(manifesto_curadoria['linhas_saida'])} imóveis curados, com "
                          f"{manifesto_curadoria['registros_removidos']} removidos por regra "
                          f"declarada",
         "registros": len(registros_por_fase["2 · Entendimento e ajuste dos dados"]),
         "evidencia": "data/processed/manifesto_curadoria.json"},
        {"fase": "3 · Preparação dos dados",
         "pergunta": "Que espaço de atributos descreve o imóvel sem descrever o preço?",
         "o_que_decidiu": f"{len(ESPACO)} variáveis no espaço euclidiano; "
                          f"{len(manifesto_preparacao['reservadas_fora_do_arquivo'])} reservadas "
                          f"fisicamente fora do arquivo de modelagem",
         "registros": len(registros_por_fase["3 · Preparação dos dados"]),
         "evidencia": "data/processed/manifesto_preparacao.json"},
        {"fase": "4 · Modelagem",
         "pergunta": "Quantos grupos, com que algoritmo, e por que esses?",
         "o_que_decidiu": f"{ALGORITMO} com k = {K}, escolhido pela regra declarada antes do "
                          f"resultado; {len(manifesto_modelagem['modelos_reprovados'])} modelos "
                          f"reprovados com o motivo registrado",
         "registros": len(registros_por_fase["4 · Modelagem"]),
         "evidencia": "data/processed/manifesto_modelagem.json"},
        {"fase": "5 · Avaliação",
         "pergunta": "O modelo resolve o problema do negócio e a cadeia aguenta auditoria?",
         "o_que_decidiu": f"{manifesto_avaliacao['criterios_do_canvas']['tecnicos_atendidos']} de "
                          f"{manifesto_avaliacao['criterios_do_canvas']['tecnicos_totais']} "
                          f"critérios técnicos atendidos; "
                          f"{manifesto_avaliacao['varredura_de_k']['total']} configurações "
                          f"varridas; veredito de prosseguir",
         "registros": len(registros_por_fase["5 · Avaliação"]),
         "evidencia": "data/processed/manifesto_avaliacao.json"},
        {"fase": "6 · Implantação",
         "pergunta": "O que precisa existir para a segmentação funcionar amanhã?",
         "o_que_decidiu": f"Pacote {VERSAO_MODELO} carimbado, contrato de entrada, catálogo de "
                          f"{K} linhas, regra de fronteira e plano de monitoramento com "
                          f"{len(plano)} indicadores",
         "registros": len(COMPROMISSOS),
         "evidencia": "data/processed/manifesto_implantacao.json"},
    ]
)

funil_de_registros = ler_tabela("ajustes-dados", "funil-da-curadoria")
funil_de_colunas = ler_tabela("preparacao", "funil-da-preparacao")

fig, eixos = plt.subplots(1, 2, figsize=(16, 4.9), constrained_layout=True)

posicoes = np.arange(len(funil_de_registros))
eixos[0].barh(posicoes, funil_de_registros.registros, color=BARRA, height=0.62)
for posicao, linha in funil_de_registros.iterrows():
    eixos[0].text(60, posicao, linha.etapa, va="center", ha="left", color="white", fontsize=8.3)
    eixos[0].text(linha.registros + 25, posicao, f"{pt(int(linha.registros))}", va="center",
                  fontsize=8.6, color=config.AZUL_PROFUNDO, fontweight="bold")
eixos[0].set_yticks([])
eixos[0].invert_yaxis()
eixos[0].set_xlim(0, funil_de_registros.registros.max() * 1.16)
eixos[0].set_xlabel("Imóveis")
eixos[0].set_title(f"O funil da carteira — {pt(manifesto_curadoria['linhas_entrada'])} "
                   f"registros brutos para {pt(LINHAS)} imóveis")

posicoes = np.arange(len(funil_de_colunas))
eixos[1].barh(posicoes, funil_de_colunas.colunas, color=config.AZUL_PROFUNDO, height=0.62)
for posicao, linha in funil_de_colunas.iterrows():
    # As etapas da preparação têm nome comprido e barra curta: o rótulo vai fora
    # da barra, e o número dentro dela.
    eixos[1].text(linha.colunas - 1.2, posicao, f"{int(linha.colunas)}", va="center", ha="right",
                  color="white", fontsize=8.6, fontweight="bold")
    eixos[1].text(linha.colunas + 1.6, posicao, linha.etapa, va="center", ha="left",
                  color=config.AZUL_PROFUNDO, fontsize=8.3)
eixos[1].set_yticks([])
eixos[1].invert_yaxis()
eixos[1].set_xlim(0, funil_de_colunas.colunas.max() * 2.3)
eixos[1].set_xlabel("Colunas")
eixos[1].set_title(f"O funil do espaço — 83 colunas curadas para {len(ESPACO)} variáveis")

fig.suptitle("A cadeia, do arquivo bruto ao espaço em que os segmentos foram formados",
             fontsize=13, fontweight="bold", color=config.AZUL_PROFUNDO)

salvar(fig, "a-cadeia-em-numeros", metodologia)
print(f"{len(insights_da_eda)} insights registrados na análise exploratória")
metodologia[["fase", "o_que_decidiu", "registros"]]

## 3.2 · Os resultados

Seis números respondem ao que a diretoria contratou. Cada um vem com o critério que o Canvas
declarou antes de qualquer algoritmo rodar — porque um resultado sem a régua ao lado é uma
afirmação, e a régua deste projeto foi escrita na fase 1.

O painel abaixo é o que vai no primeiro slide da apresentação. Ele existe para responder à
única pergunta que a Diretoria realmente faz no encerramento: **isso funcionou?**

In [ ]:
avaliacao_de_negocio = manifesto_avaliacao["avaliacao_de_negocio"]
criterios = manifesto_modelagem["criterios"]

# A coluna `regua` diz de onde vem o critério de cada linha. Ela existe porque cinco
# dos seis resultados têm régua escrita na fase 1 e um não tem: o V de Cramér foi
# medido pela fase 5 como validação externa, e o Canvas nunca declarou piso para ele.
# Inventar um aqui e exibi-lo ao lado dos outros seria fabricar um critério e datá-lo
# de antes — que é a fraude metodológica que este projeto passou seis fases evitando.
resultados = pd.DataFrame(
    [
        {"resultado": "Linhas de produto", "valor": f"{K}", "regua": "Canvas §6",
         "criterio": f"entre {criterios['k_minimo']} e {criterios['k_maximo']}",
         "atende": criterios["k_minimo"] <= K <= criterios["k_maximo"]},
        {"resultado": "Coesão (silhueta)", "valor": f"{SILHUETA:.4f}", "regua": "Canvas §6",
         "criterio": f"≥ {criterios['silhueta_minima']:.2f}",
         "atende": SILHUETA >= criterios["silhueta_minima"]},
        {"resultado": "Estabilidade (ARI sob reamostragem)",
         "valor": f"{manifesto_modelagem['ari_reamostragem_medio']:.4f}", "regua": "Canvas §6",
         "criterio": f"≥ {criterios['ari_minimo']:.2f}",
         "atende": manifesto_modelagem["ari_reamostragem_medio"] >= criterios["ari_minimo"]},
        {"resultado": "Independência do preço (ARI contra quartis)",
         "valor": f"{avaliacao_de_negocio['ari_contra_quartis_de_preco']:.4f}",
         "regua": "Canvas §6",
         "criterio": f"< {avaliacao_de_negocio['teto_do_canvas']:.2f}",
         "atende": bool(avaliacao_de_negocio["criterio_pendente_atendido"])},
        {"resultado": "Receita na linha líder",
         "valor": f"{avaliacao_de_negocio['pct_receita_do_lider']:.1f}%", "regua": "Canvas §9",
         "criterio": "> 40% numa única linha (SMART 2)",
         "atende": avaliacao_de_negocio["pct_receita_do_lider"] > 40},
        {"resultado": "Validação externa (V de Cramér contra bairro)",
         "valor": f"{avaliacao_de_negocio['v_de_cramer_contra_bairro']:.3f}", "regua": "—",
         "criterio": "sem piso declarado — `bairro` nunca entrou em algoritmo nenhum",
         "atende": True},
    ]
)

COM_REGUA = resultados.regua != "—"
DENTRO_DO_CRITERIO = int(resultados.loc[COM_REGUA, "atende"].sum())

fig, ax = plt.subplots(figsize=(15, 4.6))
ax.set_xlim(0, 3)
ax.set_ylim(0, 2)
ax.axis("off")
ax.grid(False)
for posicao, linha in resultados.iterrows():
    coluna, fileira = posicao % 3, 1 - posicao // 3
    cor = APROVADO if linha.atende else ALERTA
    ax.add_patch(
        FancyBboxPatch((coluna + 0.035, fileira + 0.06), 0.93, 0.88,
                       boxstyle="round,pad=0.01,rounding_size=0.03", linewidth=0,
                       facecolor=cor, alpha=0.14)
    )
    ax.add_patch(
        FancyBboxPatch((coluna + 0.035, fileira + 0.06), 0.02, 0.88,
                       boxstyle="round,pad=0.0,rounding_size=0.01", linewidth=0, facecolor=cor)
    )
    ax.text(coluna + 0.1, fileira + 0.66, linha.valor, fontsize=21, fontweight="bold",
            color=config.AZUL_PROFUNDO, va="center")
    ax.text(coluna + 0.1, fileira + 0.40, linha.resultado, fontsize=9.2, va="center",
            color=config.AZUL_PROFUNDO, fontweight="bold")
    ax.text(coluna + 0.1, fileira + 0.22, f"critério: {linha.criterio}", fontsize=8, va="center",
            color="#5A6B77")
    if linha.regua == "—":
        marca = "medido"
    else:
        marca = "atende" if linha.atende else "não atende"
    ax.text(coluna + 0.9, fileira + 0.78, marca,
            fontsize=8.4, ha="right", color=cor, fontweight="bold")
ax.set_title(f"O que a segmentação entregou — {DENTRO_DO_CRITERIO} de "
             f"{int(COM_REGUA.sum())} resultados dentro do critério declarado na fase 1, "
             f"mais uma validação externa sem régua no Canvas",
             fontsize=13, color=config.AZUL_PROFUNDO, fontweight="bold", pad=16)

salvar(fig, "resultados-do-projeto", resultados)
resultados

## 3.3 · Os insights

Insight não é o número: é a **decisão que o número autoriza**. A tabela abaixo tem as duas
colunas juntas de propósito — cada linha traz a medida e o que fazer com ela, com o dono.

Dois deles não estavam na pergunta original e apareceram no caminho. São os que mais mudam a
operação, e valem ser lidos primeiro: o **S2 como linha assistida** e a **estabilidade da
estrutura durante um choque de mercado**.

In [ ]:
insights = pd.DataFrame(
    [
        {"n": 1,
         "insight": "A segmentação não é uma tabela de preço com outro nome",
         "evidencia": f"ARI {avaliacao_de_negocio['ari_contra_quartis_de_preco']:.4f} contra os "
                      f"quartis de preço, teto de {avaliacao_de_negocio['teto_do_canvas']:.2f}",
         "o_que_fazer": "Manter a política de preço dentro de cada linha, e não entre linhas: "
                        "o preço varia dentro do segmento, e a linha define ação, não faixa",
         "dono": "Diretoria Comercial"},
        {"n": 2,
         "insight": f"Um terço das unidades responde por "
                    f"{avaliacao_de_negocio['pct_receita_do_lider']:.1f}% da receita",
         "evidencia": f"S3 · {catalogo.loc[2, 'imoveis']} imóveis, "
                      f"{catalogo.loc[2, 'pct_carteira']:.1f}% da carteira",
         "o_que_fazer": "Concentrar o orçamento de vitrine e corretagem dedicada no S3, sem "
                        "orçamento de reforma — o imóvel já está no padrão",
         "dono": "Diretoria Comercial"},
        {"n": 3,
         "insight": "A alavanca de reforma está no S2, e não no segmento mais caro",
         "evidencia": f"S2 tem {valor.loc[1, 'preco_por_pe2']:.2f} US$/pé² contra "
                      f"{valor.loc[2, 'preco_por_pe2']:.2f} do S3, com "
                      f"{100 * (perfil.loc[1, 'area_habitavel'] / perfil.loc[0, 'area_habitavel'] - 1):.0f}% "
                      f"mais área que o S1",
         "o_que_fazer": "Teto de reforma de 12% concentrado em cozinha e banheiro, executado "
                        "antes do anúncio",
         "dono": "Gerência de Ativos"},
        {"n": 4,
         "insight": "O estoque herdado é decisão de destino, não de preço",
         "evidencia": f"S1 · {catalogo.loc[0, 'pct_carteira']:.1f}% das unidades e "
                      f"{catalogo.loc[0, 'pct_receita']:.1f}% da receita, com "
                      f"{perfil.loc[0, 'anos_desde_reforma']:.0f} anos desde a última reforma",
         "o_que_fazer": "Decisão unidade a unidade em 12 meses: renda, venda com reforma leve "
                        "(teto de 6%) ou desinvestimento",
         "dono": "Gerência de Ativos"},
        {"n": 5,
         "insight": "Os segmentos existem fora do espaço em que foram criados",
         "evidencia": f"V de Cramér {avaliacao_de_negocio['v_de_cramer_contra_bairro']:.3f} "
                      f"contra `bairro`, variável que nenhum algoritmo viu",
         "o_que_fazer": "Usar a linha de produto como chave de leitura geográfica no "
                        "planejamento de estoque por bairro",
         "dono": "Diretoria Comercial"},
        {"n": 6,
         "insight": "Uma das três linhas é assistida — o modelo diz onde não sabe",
         "evidencia": f"{catalogo.loc[SEGMENTO_MAIS_FRAGIL, 'pct_na_fronteira']:.1f}% do "
                      f"S{SEGMENTO_MAIS_FRAGIL + 1} na faixa de fronteira · "
                      f"{pt(EM_FRONTEIRA)} imóveis da carteira para revisão humana",
         "o_que_fazer": "Não disparar ação automática na faixa de fronteira; dimensionar a "
                        "revisão para até 15% das entradas do mês",
         "dono": "Gerência de Ativos"},
        {"n": 7,
         "insight": "A estrutura sobreviveu a um choque de mercado",
         "evidencia": f"PSI máximo {PSI_MAXIMO:.4f} entre 2006–2008 e 2009–2010 · silhueta "
                      f"{SILHUETA_REFERENCIA:.4f} → {SILHUETA_ACOMPANHAMENTO:.4f} · ARI "
                      f"{ARI_RECICLAGEM:.4f} na reciclagem simulada",
         "o_que_fazer": "Adotar reciclagem anual por calendário, e não trimestral: o custo de "
                        "reciclar sem motivo é a instabilidade das personas",
         "dono": "Área de Dados"},
    ]
)

io.salvar_tabela(insights, SECAO, "insights", indice=False)
insights[["n", "insight", "evidencia", "dono"]]

## 3.4 · A apresentação às partes interessadas

A apresentação de encerramento tem trinta minutos e quatro públicos na mesma sala, com
perguntas diferentes: o patrocinador quer saber se o objetivo foi atingido; a Diretoria
Comercial quer saber o que muda na operação; a Gerência de Ativos quer saber o que chega na
mesa dela; a Área de Dados quer saber o que fica sob sua responsabilidade.

O roteiro abaixo responde às quatro, nessa ordem, e **cada slide aponta a evidência que o
sustenta** — todas figuras produzidas pela cadeia, nenhuma refeita para a ocasião. O último
slide é o que transforma a reunião em decisão: as deliberações, com responsável e prazo.

In [ ]:
roteiro = pd.DataFrame(
    [
        {"slide": 1, "titulo": "O que a Meridiano pediu, e o que recebeu",
         "publico": "Patrocinador",
         "mensagem": f"{K} linhas de produto com ação, teto e métrica, sobre {pt(LINHAS)} "
                     f"imóveis classificados",
         "evidencia": "docs/imagens/implantacao/resultados-do-projeto.png", "minutos": 3},
        {"slide": 2, "titulo": "As três linhas de produto",
         "publico": "Diretoria Comercial",
         "mensagem": "Perfil, peso em unidades e peso em receita de cada linha",
         "evidencia": "docs/imagens/implantacao/linhas-de-produto.png", "minutos": 6},
        {"slide": 3, "titulo": "O que muda na operação de cada linha",
         "publico": "Diretoria Comercial e Gerência de Ativos",
         "mensagem": "Política de preço, teto de reforma, canal e métrica por linha",
         "evidencia": "reports/tables/implantacao/catalogo-de-personas.csv", "minutos": 5},
        {"slide": 4, "titulo": "A segmentação não é uma tabela de preço",
         "publico": "Patrocinador e Diretoria Comercial",
         "mensagem": f"ARI {avaliacao_de_negocio['ari_contra_quartis_de_preco']:.4f} contra os "
                     f"quartis de preço e V de Cramér "
                     f"{avaliacao_de_negocio['v_de_cramer_contra_bairro']:.3f} contra bairro",
         "evidencia": "docs/imagens/avaliacao/independencia-do-preco.png", "minutos": 4},
        {"slide": 5, "titulo": "Onde o modelo diz que não sabe",
         "publico": "Gerência de Ativos",
         "mensagem": f"{pt(EM_FRONTEIRA)} imóveis ({100 * EM_FRONTEIRA / LINHAS:.1f}%) na faixa "
                     f"de fronteira, com lista nominal para revisão",
         "evidencia": "docs/imagens/implantacao/calibracao-da-fronteira.png", "minutos": 4},
        {"slide": 6, "titulo": "Como entra em produção",
         "publico": "Todos",
         "mensagem": "Três janelas com go/no-go; a primeira é em sombra, sem mudar ação "
                     "comercial nenhuma",
         "evidencia": "docs/imagens/implantacao/janelas-de-implantacao.png", "minutos": 4},
        {"slide": 7, "titulo": "Como se saberá que o modelo envelheceu",
         "publico": "Área de Dados e patrocinador",
         "mensagem": f"{len(plano)} indicadores com limiar e dono; primeira leitura já "
                     f"publicada, sem nenhum gatilho disparado",
         "evidencia": "docs/imagens/implantacao/painel-de-saude.png", "minutos": 4},
        {"slide": 8, "titulo": "O que fica pendente e o que precisamos decidir hoje",
         "publico": "Todos",
         "mensagem": "As deliberações, com responsável e prazo — inclusive os três campos que "
                     "o sistema comercial precisa passar a coletar",
         "evidencia": "reports/tables/implantacao/deliberacoes.csv", "minutos": 4},
    ]
)

deliberacoes = pd.DataFrame(
    [
        {"deliberacao": f"Aprovar o catálogo de {K} linhas de produto como política comercial",
         "quem_decide": "Diretoria Comercial", "prazo": "na sessão",
         "consequencia_se_nao_decidir": "a janela 1 não abre — não há ação a colocar em sombra"},
        {"deliberacao": "Autorizar a janela 1 (sombra, 30 dias) sem mudança de ação comercial",
         "quem_decide": "Patrocinador", "prazo": "na sessão",
         "consequencia_se_nao_decidir": "o modelo fica publicado sem uso, e a primeira leitura "
                                        "de monitoramento não acontece"},
        {"deliberacao": "Instrumentar `preco_anunciado`, `data_anuncio` e "
                        "`custo_reforma_executada` no sistema comercial",
         "quem_decide": "Diretoria Comercial com Engenharia de Dados", "prazo": "até D+90",
         "consequencia_se_nao_decidir": f"{SEM_LEITURA_HOJE} dos critérios de negócio do Canvas "
                                        f"permanecem sem leitura por tempo indeterminado"},
        {"deliberacao": "Retirar de produção os artefatos da cadeia anterior "
                        "(`pipeline_segmentacao.joblib` e `metadados_segmentacao.json`)",
         "quem_decide": "Área de Dados", "prazo": "antes da janela 1",
         "consequencia_se_nao_decidir": f"permanece o caminho que classifica {pt(MUDAM_DE_ACAO)} "
                                        f"imóveis na linha errada"},
        {"deliberacao": "Atualizar as páginas publicadas para a cadeia atual antes de qualquer "
                        "divulgação externa",
         "quem_decide": "Área de Dados", "prazo": "antes da janela 2",
         "consequencia_se_nao_decidir": f"{manifesto_avaliacao['revisao_do_processo']['ocorrencias_documentais_desatualizadas']} "
                                        f"ocorrências continuam descrevendo outro experimento"},
        {"deliberacao": f"Decidir no Canvas §6 o piso de silhueta por faixa de k e o destino do "
                        f"candidato k = {manifesto_avaliacao['candidato_de_revisao']['k']}",
         "quem_decide": "Patrocinador e Área de Dados",
         "prazo": "antes da próxima reciclagem",
         "consequencia_se_nao_decidir": "a próxima execução repete a decisão por um piso único, "
                                        "que favorece k pequeno por construção"},
        {"deliberacao": "Decidir o destino da meta de mineração não entregue (regras de "
                        "associação e detecção de anomalia)",
         "quem_decide": "Patrocinador", "prazo": "antes da próxima reciclagem",
         "consequencia_se_nao_decidir": "o Canvas §5 continua prometendo entrega que a cadeia "
                                        "atual não produz"},
    ]
)

io.salvar_tabela(roteiro, SECAO, "roteiro-da-apresentacao", indice=False)
io.salvar_tabela(deliberacoes, SECAO, "deliberacoes", indice=False)

registrar(
    "3 · Relatório",
    "Conduzir a sessão de encerramento com as partes interessadas",
    "Área de Dados",
    "toda deliberação sai com responsável, prazo e consequência de não decidir",
    f"{len(roteiro)} slides em {int(roteiro.minutos.sum())} minutos · "
    f"{len(deliberacoes)} deliberações registradas",
    acao="plano definido",
)
deliberacoes[["deliberacao", "quem_decide", "prazo"]]

## 3.5 · O relatório final, em arquivo

A apresentação acaba quando a reunião acaba. O relatório fica.

O arquivo abaixo é montado **com os números medidos nesta execução** — nenhum valor digitado à
mão — e gravado em `reports/relatorio-final.md`. É ele que responde, meses depois, à pergunta
que sempre volta: *por que são três linhas, e não quatro?*

Ele inclui de propósito a seção que relatórios finais costumam omitir: **o que não foi
entregue**. Um relatório que só lista acertos obriga quem vier depois a redescobrir os limites
por conta própria — e a redescoberta custa mais caro do que a confissão.

In [ ]:
def tabela_markdown(dados: pd.DataFrame, colunas: dict[str, str]) -> str:
    """Converte um recorte do DataFrame em tabela Markdown, sem dependência extra."""
    cabecalho = "| " + " | ".join(colunas.values()) + " |"
    separador = "|" + "|".join([":--"] * len(colunas)) + "|"
    linhas = [
        "| " + " | ".join(str(registro[coluna]).replace("\n", " ") for coluna in colunas) + " |"
        for _, registro in dados.iterrows()
    ]
    return "\n".join([cabecalho, separador, *linhas])


perguntas_smart = ler_tabela("avaliacao", "perguntas-smart")

# Apelidos curtos para que o texto do relatório caiba na linha sem quebrar no
# meio de uma expressão — o arquivo gerado também é lido em texto puro.
# `OCORRENCIAS_DESATUALIZADAS` já veio do manifesto no bloco 0.
K_CANDIDATO = manifesto_avaliacao["candidato_de_revisao"]["k"]
ARI_DA_REAMOSTRAGEM = manifesto_modelagem["ari_reamostragem_medio"]
ARI_CONTRA_PRECO = avaliacao_de_negocio["ari_contra_quartis_de_preco"]
TETO_DO_CANVAS = avaliacao_de_negocio["teto_do_canvas"]
RECEITA_DO_LIDER = avaliacao_de_negocio["pct_receita_do_lider"]
FOLGA_DO_CANDIDATO = manifesto_avaliacao["candidato_de_revisao"]["distancia_do_piso"]
ARI_DA_REEXECUCAO = manifesto_avaliacao["modelo_avaliado"]["ari_da_reexecucao"]

relatorio = f"""# Segmentação da carteira residencial — relatório final

**Meridiano Imobiliária** · CRISP-DM, fases 1 a 6 · modelo {VERSAO_MODELO} · semente {SEMENTE}

> Documento gerado pelo notebook `notebooks/06-implantacao.ipynb`. Todos os números vêm dos
> manifestos e tabelas da cadeia — nenhum foi digitado à mão.

## 1 · Sumário executivo

* A carteira de **{pt(LINHAS)} imóveis** se organiza em **{K} linhas de produto** com perfil,
  valor e ação distintos. O modelo é {ALGORITMO} com k = {K}, silhueta {SILHUETA:.4f} e
  estabilidade ARI {ARI_DA_REAMOSTRAGEM:.4f} sob reamostragem.
* As linhas **não reproduzem uma faixa de preço**: ARI {ARI_CONTRA_PRECO:.4f} contra os
  quartis de `preco_venda`, contra o teto de {TETO_DO_CANVAS:.2f} declarado no Canvas. O preço
  ficou fora de todo artefato que alimentou algoritmo, e a verificação foi feita depois.
* A linha **S3 · {CATALOGO[2]['nome']}** concentra {RECEITA_DO_LIDER:.1f}% da receita
  transacionada com {catalogo.loc[2, 'pct_carteira']:.1f}% das unidades; a linha
  **S2 · {CATALOGO[1]['nome']}** tem o menor preço por pé² da carteira e é onde a reforma tem
  a maior alavanca.
* **{pt(EM_FRONTEIRA)} imóveis ({100 * EM_FRONTEIRA / LINHAS:.1f}%)** ficam na faixa de
  fronteira entre segmentos e vão para revisão humana em vez de ação automática.

## 2 · O problema

A Meridiano opera a carteira residencial com uma política de duas faixas de preço. A pergunta
que originou o projeto: **existem grupos de imóveis com necessidade comercial distinta que a
faixa de preço não enxerga?** O critério de sucesso, declarado antes de qualquer modelagem, é
o do Canvas do Problema: de {criterios['k_minimo']} a {criterios['k_maximo']} grupos, silhueta
≥ {criterios['silhueta_minima']:.2f}, estabilidade ARI ≥ {criterios['ari_minimo']:.2f}, menor
grupo ≥ {100 * criterios['menor_segmento_minimo']:.0f}% da carteira e ARI < {TETO_DO_CANVAS:.2f}
contra os quartis de preço.

## 3 · Metodologia

{tabela_markdown(metodologia, {"fase": "Fase", "o_que_decidiu": "O que decidiu", "evidencia": "Evidência"})}

Três decisões de método sustentam o resto:

1. **o rótulo foi reservado fisicamente.** `preco_venda`, `preco_por_pe2` e `bairro` ficaram
   fora do arquivo de modelagem, e não apenas fora do algoritmo por disciplina;
2. **os critérios de aceitação foram escritos antes** de qualquer algoritmo rodar, e lidos do
   manifesto nas fases seguintes — nunca redigitados;
3. **cada fase gravou um manifesto** com o que decidiu e com que medida, o que tornou possível
   auditar a cadeia inteira na fase 5 e reconstruí-la aqui sem consultar ninguém.

## 4 · Resultados

{tabela_markdown(resultados, {"resultado": "Resultado", "valor": "Valor", "criterio": "Critério declarado"})}

### As {K} linhas de produto

{tabela_markdown(catalogo.reset_index(), {"codigo": "Código", "nome": "Linha", "imoveis": "Imóveis", "pct_carteira": "% unidades", "pct_receita": "% receita", "teto_de_reforma": "Teto de reforma"})}

## 5 · Insights

{tabela_markdown(insights, {"n": "#", "insight": "Insight", "evidencia": "Evidência", "dono": "Dono"})}

## 6 · As perguntas SMART, respondidas

{tabela_markdown(perguntas_smart, {"pergunta": "Pergunta", "resposta_medida": "Resposta medida nesta cadeia", "situacao": "Situação"})}

## 7 · Implantação

* **Pacote {VERSAO_MODELO}**, carimbado por SHA-256 (`{CARIMBO_DO_MODELO[:16]}…`): receita e
  agrupador no mesmo objeto, com o mapa canônico dos segmentos;
* **contrato de entrada** com {len(contrato)} campos, faixa aceita, faixa observada e padrão;
* **regra de atribuição com faixa de confiança**: corte de fronteira {CORTE_FRONTEIRA:.3f},
  calibrado sobre a carteira, com teto operacional de {TETO_OPERACIONAL:.0%};
* **entrada em produção em {len(janelas)} janelas**, a primeira em sombra por 30 dias;
* **{len(entregas)} entregas** com consumidor, meio, periodicidade, prazo e responsável.

## 8 · Monitoramento e manutenção

Sem gabarito, o que se acompanha é a entrada, a coesão e a composição. A primeira leitura,
feita entre 2006–2008 e 2009–2010, está publicada:

* PSI máximo **{PSI_MAXIMO:.4f}** (limiar de ação {LIMIAR_PSI_ACAO});
* silhueta **{SILHUETA_REFERENCIA:.4f} → {SILHUETA_ACOMPANHAMENTO:.4f}** entre os dois períodos;
* maior variação de composição **{MAIOR_VARIACAO_DE_MIX:.2f} p.p.** (limiar 10 p.p.);
* reciclagem simulada com **ARI {ARI_RECICLAGEM:.4f}** contra a partição vigente (piso
  {PISO_ARI:.2f}).

{tabela_markdown(plano, {"indicador": "Indicador", "periodicidade": "Periodicidade", "limiar": "Gatilho", "responsavel": "Responsável"})}

## 9 · O que não foi entregue

* **{SEM_LEITURA_HOJE} dos {len(metricas_de_negocio)} critérios de negócio do Canvas continuam
  sem leitura**: três dependem de campos que o sistema comercial ainda não coleta — preço
  anunciado, data de anúncio e custo de reforma executada — e o quarto depende da decisão
  unidade a unidade do estoque herdado, que ainda não começou;
* **as regras de associação e a detecção de anomalia**, prometidas no Canvas §5, não fazem
  parte desta cadeia — a decisão sobre refazê-las ou revisar a meta está entre as deliberações
  da sessão de encerramento;
* **a documentação publicada descreve o experimento anterior** em {OCORRENCIAS_DESATUALIZADAS}
  ocorrências, e precisa ser reescrita antes de qualquer divulgação externa;
* **o candidato k = {K_CANDIDATO}** foi reprovado por {FOLGA_DO_CANDIDATO:.4f} de silhueta e
  fica registrado para revisão do critério, não para revisão do resultado.

## 10 · Como reproduzir

```bash
uv sync
uv run invoke notebooks
```

A semente do projeto é {SEMENTE}. A cadeia reexecutada devolve a mesma partição — a fase 5
mediu ARI {ARI_DA_REEXECUCAO:.4f} na reexecução.
"""

ARQUIVO_RELATORIO = config.REPORTS / "relatorio-final.md"
ARQUIVO_RELATORIO.parent.mkdir(parents=True, exist_ok=True)
ARQUIVO_RELATORIO.write_text(relatorio, encoding="utf-8")

registrar(
    "3 · Relatório",
    "Produzir o relatório final com metodologia, resultados e insights",
    "Área de Dados",
    "todo número do relatório vem de manifesto ou tabela da cadeia, sem digitação",
    f"reports/relatorio-final.md · {len(relatorio.splitlines())} linhas · "
    f"{len(insights)} insights e {len(resultados)} resultados",
)
print(relatorio[:1500])

# 4 · Revisar o projeto — a retrospectiva

*Review project* — a última tarefa do CRISP-DM, e a única cujo destinatário não é o cliente. A
retrospectiva não serve para a Meridiano: serve para **o próximo projeto**, que vai começar
com as mesmas armadilhas disponíveis.

A regra que este bloco segue é a mesma dos anteriores: **cada lição vem com o número que a
sustenta.** Retrospectiva sem medida vira lista de boas intenções — "comunicar melhor",
"documentar mais" —, e nenhuma delas muda o que a próxima equipe faz numa terça-feira.

## 4.1 · O que sustentou, e o que não sustentou

Dez lições, divididas pelo critério mais simples possível: o que aguentou o peso da fase
seguinte, e o que cedeu quando alguém foi conferir.

In [ ]:
licoes = pd.DataFrame(
    [
        {"licao": "Escrever o critério antes de rodar o algoritmo",
         "categoria": "sustentou", "fase": "4 · Modelagem",
         "evidencia": f"{manifesto_avaliacao['varredura_de_k']['total']} configurações varridas "
                      f"na fase 5 sem que a régua se mexesse",
         "consequencia": "A varredura virou auditoria da decisão, e não renegociação dela"},
        {"licao": "Reservar o rótulo fisicamente, e não por disciplina",
         "categoria": "sustentou", "fase": "3 · Preparação",
         "evidencia": f"ARI {avaliacao_de_negocio['ari_contra_quartis_de_preco']:.4f} contra os "
                      f"quartis de preço, medido depois da modelagem",
         "consequencia": "O critério de independência do preço virou prova, e não afirmação"},
        {"licao": "Gravar um manifesto por fase, com o que foi decidido e a medida",
         "categoria": "sustentou", "fase": "todas",
         "evidencia": "5 manifestos lidos por este notebook; nenhuma tabela redigitada",
         "consequencia": "A fase 6 reconstruiu a cadeia inteira sem consultar ninguém"},
        {"licao": "Ordenar os segmentos por um critério canônico",
         "categoria": "sustentou", "fase": "4 · Modelagem",
         "evidencia": f"mapa canônico {MAPA_CANONICO} gravado dentro do pacote",
         "consequencia": "O S1 continua sendo o mesmo imóvel na próxima execução, e o texto de "
                         "negócio não envelhece em uma rodada"},
        {"licao": "Medir o desvio no dado real antes de publicar o plano",
         "categoria": "sustentou", "fase": "6 · Implantação",
         "evidencia": f"PSI máximo {PSI_MAXIMO:.4f} e ARI {ARI_RECICLAGEM:.4f} na reciclagem "
                      f"simulada",
         "consequencia": "O plano nasce com a primeira leitura feita, e os limiares são "
                         "verificáveis em vez de plausíveis"},
        {"licao": "Deixar a documentação publicada para o fim",
         "categoria": "não sustentou", "fase": "todas",
         "evidencia": f"{manifesto_avaliacao['revisao_do_processo']['ocorrencias_documentais_desatualizadas']} "
                      f"ocorrências descrevendo o experimento anterior",
         "consequencia": "Virou a maior dívida do projeto, e bloqueia a divulgação externa"},
        {"licao": "Deixar artefatos de produção antigos convivendo com os novos",
         "categoria": "não sustentou", "fase": "6 · Implantação",
         "evidencia": f"ARI {ARI_PUBLICADO_APROVADO:.4f} e {pt(MUDAM_DE_ACAO)} imóveis com "
                      f"linha de produto diferente",
         "consequencia": "Um diretório com dois modelos é um modelo em produção e uma "
                         "armadilha; o carimbo resolve, e devia existir desde a primeira "
                         "publicação"},
        {"licao": "Amarrar constante de negócio ao modelo sem teste automatizado",
         "categoria": "não sustentou", "fase": "4 · Modelagem",
         "evidencia": f"catálogo com {len(metadados_legado['personas'])} personas para um "
                      f"modelo de {K} segmentos",
         "consequencia": "Foi pego por leitura manual na fase 5; um teste de uma linha teria "
                         "pego no commit"},
        {"licao": "Usar um piso de silhueta único para todas as faixas de k",
         "categoria": "não sustentou", "fase": "1 · Entendimento do negócio",
         "evidencia": f"k = {manifesto_avaliacao['candidato_de_revisao']['k']} reprovado por "
                      f"{manifesto_avaliacao['candidato_de_revisao']['distancia_do_piso']:.4f} "
                      f"de silhueta",
         "consequencia": "A silhueta cai com k por construção; o piso único favorece k pequeno "
                         "sem que ninguém tenha decidido isso"},
        {"licao": "Prometer entrega de mineração sem prever o esforço na cadeia",
         "categoria": "não sustentou", "fase": "1 · Entendimento do negócio",
         "evidencia": "Canvas §5 promete regras de associação e anomalia; a cadeia não tem "
                      "nenhuma das duas",
         "consequencia": "Uma das cinco perguntas SMART ficou sem resposta na cadeia atual"},
    ]
)

SUSTENTARAM = int((licoes.categoria == "sustentou").sum())
NAO_SUSTENTARAM = int((licoes.categoria == "não sustentou").sum())

fig, ax = plt.subplots(figsize=(14.5, 6.4))
ordenadas = licoes.sort_values("categoria", ascending=False).reset_index(drop=True)
cores_licao = [APROVADO if c == "sustentou" else ALERTA for c in ordenadas.categoria]
posicoes = np.arange(len(ordenadas))
ax.barh(posicoes, [1] * len(ordenadas), color=cores_licao, height=0.74)
for posicao, linha in ordenadas.iterrows():
    ax.text(0.012, posicao - 0.14, linha.licao, va="center", ha="left", color="white",
            fontsize=9.4, fontweight="bold")
    ax.text(0.012, posicao + 0.16, linha.evidencia, va="center", ha="left", color="white",
            fontsize=7.8, alpha=0.95)
    ax.text(0.988, posicao, linha.fase, va="center", ha="right", color="white", fontsize=8)
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.invert_yaxis()
ax.grid(False)
for lado in ("top", "right", "bottom", "left"):
    ax.spines[lado].set_visible(False)
ax.set_title(f"A retrospectiva — {SUSTENTARAM} práticas que sustentaram (verde) e "
             f"{NAO_SUSTENTARAM} que cederam (coral)")

salvar(fig, "licoes-aprendidas", licoes)

registrar(
    "4 · Retrospectiva",
    "Documentar as lições, com a medida de cada uma",
    "Equipe do projeto",
    "toda lição tem número, fase de origem e consequência observada",
    f"{len(licoes)} lições · {SUSTENTARAM} sustentaram, {NAO_SUSTENTARAM} não",
    acao="lição registrada",
)
licoes[["licao", "categoria", "evidencia"]]

### A lição que atravessa as outras

As cinco que cederam têm um padrão, e vale nomeá-lo: **nenhuma é sobre modelagem.** Nenhuma
fala de algoritmo errado, métrica mal escolhida ou hipótese estatística furada — a parte
técnica do projeto aguentou a auditoria da fase 5 sem ajuste.

As cinco são sobre **o que fica em volta do modelo**: documentação, artefato antigo, constante
duplicada, promessa de escopo, limiar herdado. É onde o projeto gastou menos atenção, e é de
onde vieram os três itens que bloquearam a publicação.

O corolário prático, para o próximo projeto: **o artefato e o texto que descrevem o modelo
fazem parte do modelo.** Se estiverem errados, o modelo está errado em produção, mesmo que a
silhueta esteja certa.

## 4.2 · Recomendações para os próximos projetos

Sete recomendações. Cada uma responde a uma lição que cedeu, e cada uma é acionável na
primeira semana do próximo projeto — não são princípios, são passos.

In [ ]:
recomendacoes = pd.DataFrame(
    [
        {"n": 1,
         "recomendacao": "Declarar o piso de qualidade em função do parâmetro que ele avalia",
         "responde_a": "Piso de silhueta único para todas as faixas de k",
         "como_aplicar": "No Canvas, escrever o piso por faixa de k (ex.: 0,15 para k ≤ 3; "
                         "0,13 para k de 4 a 5), decidido antes da primeira varredura",
         "quando": "Fase 1, antes de qualquer modelagem"},
        {"n": 2,
         "recomendacao": "Travar por teste automatizado toda constante derivada do modelo",
         "responde_a": "Catálogo com mais personas do que o modelo tem segmentos",
         "como_aplicar": "Um teste que leia o manifesto e compare `len(catálogo) == k`, e que "
                         "falhe o commit — não uma revisão manual",
         "quando": "Fase 4, junto com a primeira gravação do modelo"},
        {"n": 3,
         "recomendacao": "Carimbar o artefato desde a primeira publicação",
         "responde_a": "Dois modelos convivendo no mesmo diretório",
         "como_aplicar": "SHA-256 e versão semântica no manifesto; a aplicação recusa a carga "
                         "quando o resumo não confere",
         "quando": "Fase 4, na primeira serialização"},
        {"n": 4,
         "recomendacao": "Escrever o contrato de entrada junto com o dicionário de dados",
         "responde_a": "Contrato definido só na fase 6, quando a operação já esperava a entrega",
         "como_aplicar": "Faixa aceita, faixa observada e padrão por campo, gravados como "
                         "artefato na preparação",
         "quando": "Fase 3, junto com a matriz de modelagem"},
        {"n": 5,
         "recomendacao": "Listar na fase 1 os campos que a operação precisa passar a coletar",
         "responde_a": f"{SEM_LEITURA_HOJE} critérios de negócio sem leitura por falta de campo",
         "como_aplicar": "Para cada critério de sucesso, perguntar 'com que dado isso será "
                         "medido?' e abrir a demanda de captura no mesmo dia",
         "quando": "Fase 1, na definição dos critérios de sucesso"},
        {"n": 6,
         "recomendacao": "Tratar a documentação publicada como parte da definição de pronto",
         "responde_a": "Páginas descrevendo outro experimento ao fim do projeto",
         "como_aplicar": "Nenhuma fase é encerrada com a página correspondente desatualizada; "
                         "a varredura documental entra na suíte de testes",
         "quando": "Todas as fases"},
        {"n": 7,
         "recomendacao": "Reconciliar o escopo prometido ao fim de cada fase",
         "responde_a": "Meta de mineração prometida no Canvas e não entregue",
         "como_aplicar": "Ao fechar a fase, confrontar o Canvas item a item: entregue, "
                         "revisado ou explicitamente adiado — o silêncio não é opção",
         "quando": "Todas as fases"},
    ]
)

io.salvar_tabela(recomendacoes, SECAO, "recomendacoes", indice=False)

registrar(
    "4 · Retrospectiva",
    "Registrar as recomendações para os próximos projetos",
    "Equipe do projeto",
    "cada recomendação responde a uma lição que cedeu e diz em que fase se aplica",
    f"{len(recomendacoes)} recomendações · {int((recomendacoes.quando.str.startswith('Fase 1')).sum())} "
    f"aplicáveis já na fase 1",
    acao="lição registrada",
)
recomendacoes[["n", "recomendacao", "como_aplicar", "quando"]]

## 4.3 · O backlog da fase 5, reconciliado

A fase 5 entregou oito itens, três deles bloqueando a publicação. Fechar a fase 6 sem dizer o
que aconteceu com cada um seria repetir exatamente o erro que a retrospectiva acabou de
apontar.

O backlog entra aqui **pelo arquivo** — `reports/tables/avaliacao/proximas-etapas.csv` —, com
as duas colunas que a fase 5 usou para transformar intenção em tarefa: `criterio_de_conclusao`
e `onde`. A segunda é a que costuma ser esquecida numa reconciliação, e é a que impede o desfecho
mais confortável possível: declarar fechado um item porque *alguma coisa foi feita*, quando a
coisa foi feita em outro lugar.

Por isso o estado de cada item **não é declarado aqui, é medido**. Para cada linha existe uma
conferência que abre o arquivo apontado em `onde` e verifica se ele carrega o que a fase 5
pediu. O estado sai dessa conferência:

| Estado | Como se chega nele |
|:---|:---|
| `fechado` | A conferência passou: o critério da fase 5 está atendido **no arquivo que ela nomeou** |
| `parcial` | A parte que dependia de medida foi feita neste notebook; a parte que mora em `src/` ou em `docs/` continua aberta, com dono |
| `transferido` | Não é trabalho desta fase: é decisão de negócio, com dono nomeado |

A reconciliação acrescenta uma coluna que a fase 5 não tinha, e ela é o que sustenta a decisão
sobre a janela 1: a **natureza** do item — o que ele move.

| Natureza | O que o item move | Efeito sobre a entrada em produção |
|:---|:---|:---|
| `modelo` | O que a operação carrega, ou como ela classifica | Impede a janela 1 |
| `código` | Um módulo do repositório que hoje serve à cadeia anterior | Pré-requisito da janela 3 |
| `texto` | Documentação publicada e Canvas | Pré-requisito da janela 2 |
| `operação` | Um campo que o sistema comercial precisa passar a coletar | Pré-requisito da janela 2 |
| `decisão` | Uma deliberação de negócio | Não impede janela nenhuma; tem prazo |

Nenhum item some, e nenhum fecha por parecer fechado.

In [ ]:
backlog = backlog_da_fase_5.reset_index()


def endereco_cumpre(caminho: str, *termos: str) -> bool:
    """O arquivo que a fase 5 nomeou em `onde` já carrega o que ela pediu?

    É conferência de **endereço**, não de mérito: procura no arquivo os termos que
    provam que a mudança chegou lá. Um `False` aqui não julga a qualidade de nada —
    diz apenas que o item continua morando no notebook, e não no lugar onde a fase 5
    escreveu que ele deveria morar.
    """
    arquivo = RAIZ / caminho
    if not arquivo.exists():
        return False
    texto = arquivo.read_text(encoding="utf-8")
    return all(termo in texto for termo in termos)


TEXTO_DOS_TESTES = "\n".join(
    arquivo.read_text(encoding="utf-8") for arquivo in sorted((RAIZ / "tests").glob("test_*.py"))
)

PERSONAS_NO_CODIGO = len(pers.PERSONAS)
TESTE_TRAVA_O_CATALOGO = "PERSONAS" in TEXTO_DOS_TESTES and "manifesto" in TEXTO_DOS_TESTES
SCORING_REAPONTADO = endereco_cumpre(
    "src/deployment/scoring.py", "modelo-segmentacao.joblib", "razao_de_fronteira"
)
MONITORAMENTO_COM_RECICLAGEM = endereco_cumpre(
    "src/deployment/monitoramento.py", "adjusted_rand_score"
)
CANVAS_REVISADO = endereco_cumpre("docs/pbl/canvas-do-problema.md", f"{K} segmentos")

# --------------------------------------------------------------------------- #
# A reconciliação. `conferencia` é medida — o estado sai dela, e não do contrário.
# `natureza` diz o que o item move: o modelo, o código, o texto, a operação ou uma
# decisão de negócio. É essa coluna, e não a contagem de itens, que decide a janela 1.
# --------------------------------------------------------------------------- #
reconciliacao = {
    1: {"natureza": "código",
        "dono_do_que_falta": "Área de Dados",
        "conferencia": bool(len(CATALOGO) == K and PERSONAS_NO_CODIGO == K
                            and TESTE_TRAVA_O_CATALOGO),
        "o_que_esta_fase_fez": f"Catálogo de {K} linhas publicado em "
                               f"models/catalogo_de_personas.json e dentro do pacote da "
                               f"aplicação, com ação, teto, canal e métrica por linha",
        "o_que_falta": f"src/model/personas.py declara {PERSONAS_NO_CODIGO} personas e nenhum "
                       f"teste compara len(PERSONAS) ao k do manifesto"},
    2: {"natureza": "texto",
        "dono_do_que_falta": "Área de Dados",
        "conferencia": bool(OCORRENCIAS_DESATUALIZADAS == 0),
        "o_que_esta_fase_fez": "Relatório final gerado com os números desta cadeia em "
                               "reports/relatorio-final.md",
        "o_que_falta": f"{OCORRENCIAS_DESATUALIZADAS} ocorrências desatualizadas continuam em "
                       f"docs/, README.md e mkdocs.yml"},
    3: {"natureza": "texto",
        "dono_do_que_falta": "Patrocinador",
        "conferencia": bool(CANVAS_REVISADO),
        "o_que_esta_fase_fez": "As cinco respostas SMART medidas nesta cadeia entram no "
                               "relatório final",
        "o_que_falta": "registrar a revisão no Canvas §9 e decidir o destino da pergunta 3"},
    4: {"natureza": "código",
        "dono_do_que_falta": "Área de Dados",
        "conferencia": bool(SCORING_REAPONTADO),
        "o_que_esta_fase_fez": f"Regra de fronteira calibrada (corte {CORTE_FRONTEIRA:.3f}); a "
                               f"saída traz razão, faixa e segundo mais próximo, e o pacote da "
                               f"aplicação leva os dois cortes",
        "o_que_falta": "src/deployment/scoring.py ainda carrega o artefato da cadeia anterior "
                       "e não devolve a razão de fronteira"},
    5: {"natureza": "operação",
        "dono_do_que_falta": "Diretoria Comercial",
        "conferencia": bool(SEM_LEITURA_HOJE == 0),
        "o_que_esta_fase_fez": "Cada critério de negócio ganhou métrica, campo de origem, "
                               "responsável e prazo",
        "o_que_falta": f"{SEM_LEITURA_HOJE} critérios seguem sem leitura: faltam preço "
                       f"anunciado, data de anúncio e custo de reforma executada"},
    6: {"natureza": "decisão",
        "dono_do_que_falta": "Patrocinador e Área de Dados",
        "conferencia": False,
        "o_que_esta_fase_fez": "Levado à sessão de encerramento como deliberação formal",
        "o_que_falta": "decidir o piso por faixa de k antes da próxima execução"},
    7: {"natureza": "decisão",
        "dono_do_que_falta": "Patrocinador",
        "conferencia": False,
        "o_que_esta_fase_fez": "Levado à sessão de encerramento como deliberação formal",
        "o_que_falta": "refazer a etapa de regras sobre a base curada, ou revisar a meta de "
                       "mineração no Canvas §5"},
    8: {"natureza": "código",
        "dono_do_que_falta": "Área de Dados",
        "conferencia": bool(MONITORAMENTO_COM_RECICLAGEM),
        "o_que_esta_fase_fez": f"Cinco gatilhos e três faixas de aceite por ARI, com piso "
                               f"{PISO_ARI:.2f}; a simulação de hoje devolveu "
                               f"{ARI_RECICLAGEM:.4f}",
        "o_que_falta": "src/deployment/monitoramento.py não conhece ARI nem reciclagem: a "
                       "regra vive em CSV e no manifesto, não no módulo que a executaria"},
}

backlog = backlog.join(pd.DataFrame(reconciliacao).T.rename_axis("ordem"), on="ordem")
backlog["estado"] = np.where(
    backlog.conferencia.astype(bool), "fechado",
    np.where(backlog.natureza == "decisão", "transferido", "parcial"),
)
FECHADOS = int((backlog.estado == "fechado").sum())
PARCIAIS = int((backlog.estado == "parcial").sum())
TRANSFERIDOS = int((backlog.estado == "transferido").sum())
ABERTOS = backlog[backlog.estado != "fechado"]
BLOQUEIOS_ABERTOS = int(ABERTOS.bloqueia_publicacao.astype(bool).sum())

# A regra da janela 1, declarada antes da contagem. A fase 5 não escreveu regra para
# janela nenhuma: escreveu que três itens bloqueiam a **publicação**. A janela 1 não
# publica — classifica em sombra, sem mudar ação comercial e sem publicar página. A
# fase 6 declara então uma regra própria, e o que a impede não é a quantidade de itens
# abertos: é um item aberto que mude **o que a operação carrega ou como ela
# classifica**. Essa é a natureza `modelo`, e só ela.
#
# Isso é uma revisão do critério da fase 5, e não uma leitura dele. Fica registrada
# como tal, com dono, logo abaixo — abrir a janela por conta própria seria repetir em
# uma linha de código o que este notebook passou quatro blocos condenando.
NATUREZAS_QUE_BLOQUEIAM_A_JANELA_1 = ("modelo",)
ABERTOS_QUE_BLOQUEIAM_A_JANELA_1 = int(
    ABERTOS.natureza.isin(NATUREZAS_QUE_BLOQUEIAM_A_JANELA_1).sum()
)
JANELA_1_PELO_CRITERIO_DA_FASE_5 = bool(BLOQUEIOS_ABERTOS == 0 and REPRODUZ_EM_LOTE)
JANELA_1_LIBERADA = bool(REPRODUZ_EM_LOTE and ABERTOS_QUE_BLOQUEIAM_A_JANELA_1 == 0)

cores_estado = {"fechado": APROVADO, "parcial": REFERENCIA, "transferido": NEUTRO}

fig, ax = plt.subplots(figsize=(14, 5.4))
posicoes = np.arange(len(backlog))
ax.barh(posicoes, [1] * len(backlog), color=[cores_estado[e] for e in backlog.estado],
        height=0.74)
for posicao, linha in backlog.iterrows():
    cor_texto = "white" if linha.estado != "parcial" else config.AZUL_PROFUNDO
    ax.text(0.012, posicao - 0.15, f"{linha.ordem}. {linha.etapa}", va="center", ha="left",
            color=cor_texto, fontsize=9.3, fontweight="bold")
    resumo = linha.o_que_esta_fase_fez if linha.estado == "fechado" else linha.o_que_falta
    prefixo = "→" if linha.estado == "fechado" else "falta:"
    ax.text(0.012, posicao + 0.17, f"{prefixo} {resumo}", va="center", ha="left",
            color=cor_texto, fontsize=7.7, alpha=0.95)
    marca = linha.natureza + (" · bloqueava a publicação" if linha.bloqueia_publicacao else "")
    ax.text(0.988, posicao - 0.15, linha.estado.upper(), va="center", ha="right",
            color=cor_texto, fontsize=8.6, fontweight="bold")
    ax.text(0.988, posicao + 0.17, marca, va="center", ha="right", color=cor_texto, fontsize=7.4)
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.invert_yaxis()
ax.grid(False)
for lado in ("top", "right", "bottom", "left"):
    ax.spines[lado].set_visible(False)
veredito_da_janela = ("nenhum item aberto de natureza `modelo` — a janela 1 abre"
                      if JANELA_1_LIBERADA else "a janela 1 está bloqueada")
ax.set_title(f"O backlog da fase 5 — {FECHADOS} fechados, {PARCIAIS} parciais e "
             f"{TRANSFERIDOS} transferidos · {veredito_da_janela}")

salvar(fig, "backlog-reconciliado", backlog)

for _, linha in ABERTOS.iterrows():
    registrar(
        "4 · Retrospectiva",
        linha.etapa,
        linha.dono_do_que_falta,
        linha.criterio_de_conclusao,
        f"estado {linha.estado} ({linha.natureza}) — falta: {linha.o_que_falta}",
        acao="pendência transferida",
        onde=linha.onde,
    )

if JANELA_1_LIBERADA and not JANELA_1_PELO_CRITERIO_DA_FASE_5:
    registrar(
        "4 · Retrospectiva",
        "Revisar o critério de publicação da fase 5 para admitir a janela de sombra",
        "Patrocinador",
        "a janela 1 abre com itens de texto e de código abertos, desde que nenhum item "
        "aberto mude o que a operação carrega ou como ela classifica",
        f"pelo critério da fase 5 nenhuma janela abriria ({BLOQUEIOS_ABERTOS} bloqueios "
        f"abertos); pela regra declarada acima, a janela 1 abre e as 2 e 3 não",
        acao="pendência transferida",
        onde="docs/pbl/canvas-do-problema.md §6",
    )

print(f"\nItens que bloqueavam a publicação e continuam abertos: {BLOQUEIOS_ABERTOS}")
print(f"Itens abertos de natureza `modelo`: {ABERTOS_QUE_BLOQUEIAM_A_JANELA_1}")
print(f"Janela 1 liberada pela regra declarada: {JANELA_1_LIBERADA}")
backlog[["ordem", "etapa", "natureza", "estado", "o_que_falta", "onde"]]

### O que a reconciliação decide

O resultado é desconfortável e é o mais útil que esta fase produz: **nenhum dos oito itens
fecha pelo critério literal da fase 5.** Seis ficam parciais e dois são transferidos como
deliberação de negócio.

Vale separar o que isso significa do que não significa. Os itens 1, 4 e 8 tiveram a **parte que
dependia de medida** entregue aqui, e entregue inteira: o catálogo tem uma linha de produto por
segmento, com ação, teto, canal e métrica; a regra de fronteira está calibrada sobre a carteira
e cabe no teto operacional; os gatilhos de reciclagem têm limiar numérico e faixa de aceite por
ARI. O que não foi feito é a outra metade, e é sempre a mesma:

> **A fase 5 não escreveu apenas o critério — escreveu o endereço.** Os três itens pediam a
> mudança em `src/model/personas.py`, `src/deployment/scoring.py` e
> `src/deployment/monitoramento.py`, e os três módulos continuam servindo à cadeia anterior.
> Um notebook não reescreve o pacote do projeto; o que ele pode fazer é publicar o artefato,
> medir a distância e nomear quem fecha.

Declarar esses três como "fechados" era possível — bastava a paráfrase certa no critério. É
exatamente o movimento que a fase 5 passou o bloco 3 inteiro auditando em outras fases, e seria
irônico cometê-lo na página seguinte.

A leitura, para a sessão de encerramento:

> **A janela 1 pode abrir.** Nenhum item aberto é de natureza `modelo`: nenhum deles muda o que
> a operação carrega nem como ela classifica. A janela 2 não abre enquanto a documentação
> publicada e o Canvas descreverem outro experimento; a janela 3 não abre enquanto os três
> módulos de `src/` servirem à cadeia anterior.

Isso é uma **revisão** do critério da fase 5, e não uma leitura dele — ela bloqueou a
*publicação*, e a janela 1 não publica. A revisão está registrada como pendência com dono no
`registro-de-compromissos`, para ser aprovada por quem escreveu o critério original, e não
resolvida por uma linha de código deste notebook.

## 4.4 · O pacote da aplicação — um arquivo, e nenhuma dependência do repositório

Os artefatos das seções anteriores resolvem a operação **de dentro** do repositório: quem quiser
classificar um imóvel precisa carregar o `.joblib`, ler quatro JSON, dois `parquet` e uma dúzia
de CSV, e precisa saber a ordem. É aceitável para um *job* de carga que roda ao lado do código;
é inviável para a aplicação Streamlit, que sobe em outra máquina e não deve depender da árvore
de diretórios do projeto — nem do pacote `src`.

Esta seção grava **um arquivo só**, `models/aplicacao-segmentacao.pkl`, com tudo o que a
aplicação precisa: classificar um imóvel, nomear a linha de produto, dizer o grau de confiança,
mostrar a carteira e apontar a evidência de cada número.

Três regras de desenho, e cada uma evita um erro conhecido de empacotamento por `pickle`:

1. **nenhuma função entra no arquivo.** Um `pickle` não guarda o código de uma função: guarda o
   par *(módulo, nome)* e procura os dois na hora de abrir. Uma função definida em notebook é
   gravada como `__main__.classificar`, e a aplicação quebra na abertura. O que entra são
   **dados e parâmetros** — os cortes, as faixas, o catálogo, o contrato — mais o `Pipeline` do
   scikit-learn, que é objeto de biblioteca e sobrevive à travessia. A aplicação **reconstrói o
   comportamento** a partir dos parâmetros; ela não recebe o comportamento pronto;
2. **nada de `src`.** Nenhum objeto do pacote do projeto entra. A conferência abaixo lê o fluxo
   de bytes do arquivo gravado e falha se encontrar referência a `__main__` ou a qualquer módulo
   sob `src` — porque o `src` está instalado neste ambiente e um teste feito aqui daria falso
   negativo, enquanto a máquina da aplicação não o terá;
3. **a receita continua existindo em um lugar só.** O passo `preparo` que vai no arquivo é o
   **mesmo objeto ajustado** que a fase 4 gravou, lido do pacote carimbado — não uma cópia
   reajustada aqui. O SHA-256 do `.joblib` de origem viaja junto, para que a aplicação possa
   provar de qual pacote ela veio.

A conferência que fecha a seção é a mesma exigência do bloco 1.7, agora sobre o arquivo: ele é
reaberto, a carteira é reclassificada **usando somente o que está dentro dele** — pipeline,
centroides, mapa canônico e cortes — e o resultado é confrontado imóvel por imóvel com a
partição aprovada e com as faixas de confiança calibradas no bloco 1.4.

In [ ]:
PACOTE_DA_APLICACAO = config.MODELS / "aplicacao-segmentacao.pkl"
FORMATO_DO_PACOTE = "meridiano.segmentacao/1"

LEIA_ME = (
    "Pacote autocontido da segmentação da Meridiano. Para classificar um imóvel: "
    "1) complete os campos ausentes com contrato['padroes']; "
    "2) monte um DataFrame de uma linha com as colunas de modelo['espaco'], nessa ordem; "
    "3) matriz = modelo['pipeline'].named_steps['preparo'].transform(registro); "
    "4) calcule as distâncias euclidianas até modelo['centroides'] — a razão entre a menor e a "
    "segunda menor é a razão de fronteira; "
    "5) o rótulo cru do centroide mais próximo vira segmento canônico por modelo['mapa_canonico']; "
    "6) a razão vira faixa de confiança pelos cortes em atribuicao['corte_de_nucleo'] e "
    "['corte_de_fronteira'] — na faixa 'fronteira' a ação automática não dispara."
)

aplicacao = {
    "formato": FORMATO_DO_PACOTE,
    "leia_me": LEIA_ME,
    "versao_modelo": VERSAO_MODELO,
    "carimbo_de_origem": {linha.artefato: linha.sha256 for _, linha in carimbo.iterrows()},
    "modelo": {
        "algoritmo": ALGORITMO,
        "k": K,
        "semente": SEMENTE,
        "espaco": ESPACO,
        "rotulos_das_colunas": {coluna: dic.rotulo(coluna) for coluna in ESPACO},
        "pipeline": pacote["pipeline"],
        "centroides": CENTROIDES,
        "mapa_canonico": MAPA_CANONICO,
        "silhueta": SILHUETA,
        "imoveis_de_treino": LINHAS,
        "periodo_de_treino": manifesto_carga["periodo_venda"],
    },
    "contrato": {
        "campos": contrato,
        "obrigatorios": OBRIGATORIOS,
        "padroes": PADROES,
        "folga_das_continuas": FOLGA_CONTINUA,
    },
    "atribuicao": {
        "corte_de_nucleo": CORTE_NUCLEO,
        "corte_de_fronteira": CORTE_FRONTEIRA,
        "teto_operacional": TETO_OPERACIONAL,
        "faixas": ["núcleo", "típico", "fronteira"],
        "calibracao": calibracao,
        "faixas_por_segmento": faixas_por_segmento,
    },
    "catalogo": {
        "personas": CATALOGO,
        "linhas": catalogo,
        "cores": CORES_SEGMENTO,
        "perfil_por_linha": perfil,
        "assinatura": assinatura,
    },
    "carteira": carteira_classificada.merge(
        curada[["pid", "bairro", "preco_venda", "area_habitavel", "qualidade_geral"]], on="pid"
    ),
    "monitoramento": {
        "linha_de_base": linha_de_base,
        "plano": plano,
        "painel": painel,
        "gatilhos_de_reciclagem": reciclagem,
        "criterio_de_aceite": criterio_de_aceite,
        "limiares": {
            "psi_atencao": LIMIAR_PSI_ATENCAO,
            "psi_acao": LIMIAR_PSI_ACAO,
            "piso_de_silhueta": PISO_SILHUETA,
            "piso_de_ari": PISO_ARI,
        },
    },
    "evidencias": {
        "resultados": resultados,
        "insights": insights,
        "entregas": entregas,
        "janelas": janelas,
        "deliberacoes": deliberacoes,
        "backlog_da_fase_5": backlog,
        "auditoria_do_publicado": auditoria_do_publicado,
    },
}

with open(PACOTE_DA_APLICACAO, "wb") as arquivo:
    pickle.dump(aplicacao, arquivo, protocol=5)

TAMANHO_DO_PACOTE = PACOTE_DA_APLICACAO.stat().st_size
CARIMBO_DA_APLICACAO = resumo_sha256(PACOTE_DA_APLICACAO)

print(f"{PACOTE_DA_APLICACAO.relative_to(RAIZ).as_posix()} gravado")
print(f"    {TAMANHO_DO_PACOTE / 1024:.0f} KB · sha256 {CARIMBO_DA_APLICACAO[:16]}…")
for chave, conteudo in aplicacao.items():
    if isinstance(conteudo, dict):
        print(f"    {chave:<18} {len(conteudo)} chaves: {', '.join(list(conteudo)[:6])}")

In [ ]:
TEXTO_NO_FLUXO = ("SHORT_BINUNICODE", "BINUNICODE", "BINUNICODE8", "UNICODE")


def modulos_referenciados(bytes_do_pacote: bytes) -> set[str]:
    """Os módulos que o arquivo referencia, lidos do próprio fluxo de bytes.

    A conferência olha o fluxo, e não o dicionário em memória, porque o que quebra
    na outra máquina é a **referência**: o `pickle` guarda o par (módulo, nome) e
    procura os dois na abertura. Neste ambiente o `src` está instalado e o arquivo
    abriria sem erro — é exatamente por isso que o teste não pode ser "tentar abrir".
    """
    modulos, recentes = set(), []
    for opcode, argumento, _ in pickletools.genops(bytes_do_pacote):
        if opcode.name in TEXTO_NO_FLUXO:
            recentes.append(argumento)
        elif opcode.name == "STACK_GLOBAL":
            if len(recentes) >= 2:
                modulos.add(recentes[-2])
            recentes = []
        elif opcode.name == "GLOBAL":
            modulos.add(str(argumento).split(" ", 1)[0])
            recentes = []
        elif opcode.name != "MEMOIZE":
            recentes = []
    return modulos


MODULOS_DO_PACOTE = modulos_referenciados(PACOTE_DA_APLICACAO.read_bytes())
MODULOS_PROIBIDOS = sorted(
    modulo for modulo in MODULOS_DO_PACOTE
    if modulo == "__main__" or modulo.split(".")[0] == "src"
)
BIBLIOTECAS_EXIGIDAS = sorted({modulo.split(".")[0] for modulo in MODULOS_DO_PACOTE})

# --------------------------------------------------------------------------- #
# A releitura. A partir daqui nada usa `preparo`, `CENTROIDES`, `MAPA_CANONICO`
# ou os cortes do notebook: tudo vem de dentro do arquivo recém-gravado.
# --------------------------------------------------------------------------- #
with open(PACOTE_DA_APLICACAO, "rb") as arquivo:
    lido = pickle.load(arquivo)

modelo_lido, cortes_lidos = lido["modelo"], lido["atribuicao"]
matriz_lida = modelo_lido["pipeline"].named_steps["preparo"].transform(base[modelo_lido["espaco"]])
distancias_lidas = np.linalg.norm(
    matriz_lida[:, None, :] - modelo_lido["centroides"][None, :, :], axis=2
)
ordem_lida = np.argsort(distancias_lidas, axis=1)
razao_lida = (
    np.take_along_axis(distancias_lidas, ordem_lida[:, :1], axis=1).ravel()
    / np.take_along_axis(distancias_lidas, ordem_lida[:, 1:2], axis=1).ravel()
)
segmentos_lidos = np.array([modelo_lido["mapa_canonico"][int(cru)] for cru in ordem_lida[:, 0]])
faixas_lidas = np.where(
    razao_lida < cortes_lidos["corte_de_nucleo"], "núcleo",
    np.where(razao_lida < cortes_lidos["corte_de_fronteira"], "típico", "fronteira"),
)

REPRODUZ_PELO_ARQUIVO = bool((segmentos_lidos == ROTULOS).all())
FAIXAS_CONFEREM = bool((faixas_lidas == FAIXAS).all())
CATALOGO_COMPLETO = bool(len(lido["catalogo"]["personas"]) == lido["modelo"]["k"])

conferencia_do_pacote = pd.DataFrame(
    [
        {"verificacao": "O arquivo não referencia `__main__` nem `src`",
         "resultado": "nenhuma referência" if not MODULOS_PROIBIDOS
                      else ", ".join(MODULOS_PROIBIDOS),
         "situacao": "aprovado" if not MODULOS_PROIBIDOS else "reprovado"},
        {"verificacao": "Bibliotecas que a máquina da aplicação precisa ter",
         "resultado": ", ".join(BIBLIOTECAS_EXIGIDAS),
         "situacao": "aprovado"},
        {"verificacao": "A releitura reproduz a partição aprovada",
         "resultado": f"{pt(int((segmentos_lidos == ROTULOS).sum()))} de {pt(LINHAS)} imóveis",
         "situacao": "aprovado" if REPRODUZ_PELO_ARQUIVO else "reprovado"},
        {"verificacao": "A releitura reproduz as faixas de confiança calibradas",
         "resultado": f"cortes {cortes_lidos['corte_de_nucleo']:.3f} e "
                      f"{cortes_lidos['corte_de_fronteira']:.3f}"
                      if FAIXAS_CONFEREM else "DIVERGE",
         "situacao": "aprovado" if FAIXAS_CONFEREM else "reprovado"},
        {"verificacao": "O catálogo tem uma linha de produto por segmento",
         "resultado": f"{len(lido['catalogo']['personas'])} personas para k = "
                      f"{lido['modelo']['k']}",
         "situacao": "aprovado" if CATALOGO_COMPLETO else "reprovado"},
        {"verificacao": "Tamanho, versão e formato do arquivo",
         "resultado": f"{TAMANHO_DO_PACOTE / 1024:.0f} KB · versão {lido['versao_modelo']} · "
                      f"formato {lido['formato']}",
         "situacao": "aprovado"},
    ]
)

assert not MODULOS_PROIBIDOS, f"o pacote referencia o que a aplicação não terá: {MODULOS_PROIBIDOS}"
assert REPRODUZ_PELO_ARQUIVO, "o pacote da aplicação não reproduz a partição aprovada"
assert FAIXAS_CONFEREM, "o pacote da aplicação não reproduz as faixas de confiança"
assert CATALOGO_COMPLETO, "o catálogo dentro do pacote não tem uma entrada por segmento"

registrar(
    "4 · Encerramento",
    "Publicar o pacote autocontido para a aplicação",
    "Área de Dados",
    "a aplicação classifica a partir de um único arquivo, sem importar `src` e sem "
    "reimplementar a receita de preparação",
    f"{PACOTE_DA_APLICACAO.name} · {TAMANHO_DO_PACOTE / 1024:.0f} KB · "
    f"sha256 {CARIMBO_DA_APLICACAO[:16]}… · a releitura reclassifica {pt(LINHAS)} imóveis "
    f"sem divergência",
    onde="models/aplicacao-segmentacao.pkl",
)

io.salvar_tabela(conferencia_do_pacote, SECAO, "conferencia-do-pacote", indice=False)
conferencia_do_pacote

## 4.5 · O que a operação recebe

Nenhum modelo novo é ajustado aqui — a fase 6 **publica**, não modela. O que este notebook
acrescenta são os artefatos que faltavam entre um modelo aprovado e uma segmentação em
operação:

| Artefato | Conteúdo |
|:---|:---|
| `models/aplicacao-segmentacao.pkl` | **O pacote da aplicação**: modelo, catálogo, contrato, cortes, linha de base, carteira e evidências em um arquivo, sem dependência de `src` |
| `models/metadados_implantacao.json` | O **recorte operacional** do manifesto: origem, versão, carimbo SHA-256, regra de versionamento, cortes e contrato de entrada |
| `models/catalogo_de_personas.json` | As três linhas de produto, com ação, teto, canal e métrica |
| `models/contrato_de_entrada.json` | As 17 variáveis com faixa aceita, faixa observada e padrão |
| `models/linha_de_base.json` | As distribuições de referência do monitoramento, sobre a carteira aprovada |
| `data/processed/carteira_classificada.parquet` | A carteira com linha de produto, razão de fronteira e proveniência |
| `data/processed/manifesto_implantacao.json` | O que esta fase decidiu, com que medida, e o que ficou pendente |
| `reports/relatorio-final.md` | O relatório final, montado com os números desta execução |
| `reports/tables/implantacao/*.csv` | Toda tabela que originou figura, e as que não têm figura |
| `docs/imagens/implantacao/*.png` | As figuras da fase, para o MkDocs |

Três arquivos guardam a mesma decisão em recortes diferentes, e a divisão tem regra:

* o `.pkl` é para **quem executa** — a aplicação, que precisa de tudo junto e não tem o
  repositório nem o pacote `src`;
* `models/metadados_implantacao.json` é para **quem opera**: o recorte que a carga confere
  antes de rodar, legível sem abrir um binário;
* `data/processed/manifesto_implantacao.json` é para **quem audita**, e fica ao lado dos
  manifestos das outras cinco fases, com a chave `origem` no mesmo formato.

Os três carregam a mesma versão e o mesmo carimbo. Se um dia discordarem, o carimbo denuncia.

In [ ]:
config.garantir_diretorios()

metadados_implantacao = {
    "origem": {
        "manifesto_avaliacao": (config.DATA_PROCESSED / "manifesto_avaliacao.json")
        .relative_to(RAIZ).as_posix(),
        "modelo": MODELO.relative_to(RAIZ).as_posix(),
        "base_modelagem": BASE_MODELAGEM.relative_to(RAIZ).as_posix(),
        "base_segmentada": BASE_SEGMENTADA.relative_to(RAIZ).as_posix(),
        "rotulo_reservado_aberto_de": config.BASE_CURADA.relative_to(RAIZ).as_posix(),
        "backlog_da_fase_5": "reports/tables/avaliacao/proximas-etapas.csv",
    },
    "versao_modelo": VERSAO_MODELO,
    "versao_anterior_no_repositorio": VERSAO_ANTERIOR,
    "regra_de_versionamento": {
        "maior": "muda k, o espaço de atributos ou o algoritmo",
        "menor": "reajuste de centroides com o mesmo k e o mesmo espaço",
        "correcao": "muda limiar operacional, texto de persona ou documentação",
    },
    "modelo": {
        "algoritmo": ALGORITMO,
        "k": K,
        "espaco": ESPACO,
        "mapa_canonico": MAPA_CANONICO,
        "silhueta": SILHUETA,
        "semente": SEMENTE,
        "imoveis_de_treino": LINHAS,
        "periodo_de_treino": manifesto_carga["periodo_venda"],
    },
    "carimbo": {
        linha.artefato: {"arquivo": linha.arquivo, "sha256": linha.sha256, "bytes": int(linha.bytes)}
        for _, linha in carimbo.iterrows()
    },
    "regra_de_atribuicao": {
        "corte_de_nucleo": CORTE_NUCLEO,
        "corte_de_fronteira": CORTE_FRONTEIRA,
        "teto_operacional": TETO_OPERACIONAL,
        "calibrado_por": "F1 na identificação da silhueta negativa, sob o teto operacional",
        "imoveis_em_fronteira": EM_FRONTEIRA,
        "pct_em_fronteira": round(100 * EM_FRONTEIRA / LINHAS, 2),
        "revocacao_da_silhueta_negativa": float(escolhido.revocacao),
    },
    "contrato_de_entrada": {
        "campos": len(contrato),
        "obrigatorios": OBRIGATORIOS,
        "folga_das_continuas": FOLGA_CONTINUA,
        "arquivo": "models/contrato_de_entrada.json",
    },
    "catalogo": {
        CATALOGO[s]["codigo"]: {
            "nome": CATALOGO[s]["nome"],
            "imoveis": int(valor.loc[s, "imoveis"]),
            "pct_receita": float(valor.loc[s, "pct_receita"]),
            "teto_de_reforma": CATALOGO[s]["teto_de_reforma"],
        }
        for s in range(K)
    },
    "divergencia_com_o_publicado": {
        "artefato_legado": ARTEFATO_LEGADO.name,
        "k_legado": K_LEGADO,
        "ari_contra_o_aprovado": ARI_PUBLICADO_APROVADO,
        "imoveis_que_mudam_de_linha": MUDAM_DE_ACAO,
        "acao": "aposentar os artefatos da cadeia anterior antes da janela 1",
    },
    "monitoramento": {
        "indicadores": len(plano),
        "psi_maximo_na_primeira_leitura": PSI_MAXIMO,
        "atributos_em_acao": ATRIBUTOS_EM_ACAO,
        "atributos_em_atencao": ATRIBUTOS_EM_ATENCAO,
        "silhueta_referencia": SILHUETA_REFERENCIA,
        "silhueta_acompanhamento": SILHUETA_ACOMPANHAMENTO,
        "maior_variacao_de_composicao_pp": MAIOR_VARIACAO_DE_MIX,
        "ari_da_reciclagem_simulada": ARI_RECICLAGEM,
        "ari_do_controle": ARI_CONTROLE,
        "piso_de_ari": PISO_ARI,
        "piso_de_silhueta": PISO_SILHUETA,
    },
    "entrega": {
        "entregas": len(entregas),
        "janelas": len(janelas),
        "criterios_de_negocio_sem_leitura": SEM_LEITURA_HOJE,
        "relatorio_final": "reports/relatorio-final.md",
    },
    "retrospectiva": {
        "licoes": len(licoes),
        "sustentaram": SUSTENTARAM,
        "nao_sustentaram": NAO_SUSTENTARAM,
        "recomendacoes": len(recomendacoes),
    },
    "pacote_da_aplicacao": {
        "arquivo": PACOTE_DA_APLICACAO.relative_to(RAIZ).as_posix(),
        "formato": FORMATO_DO_PACOTE,
        "bytes": TAMANHO_DO_PACOTE,
        "sha256": CARIMBO_DA_APLICACAO,
        "bibliotecas_exigidas": BIBLIOTECAS_EXIGIDAS,
        "referencia_a_src_ou_main": bool(MODULOS_PROIBIDOS),
        "releitura_reproduz_a_particao": REPRODUZ_PELO_ARQUIVO,
        "releitura_reproduz_as_faixas": FAIXAS_CONFEREM,
    },
    "backlog_da_fase_5": {
        "itens": int(len(backlog)),
        "fechados": FECHADOS,
        "parciais": PARCIAIS,
        "transferidos": TRANSFERIDOS,
        "bloqueios_abertos": BLOQUEIOS_ABERTOS,
        "abertos_por_natureza": ABERTOS.natureza.value_counts().to_dict(),
        "estado_medido_contra": "a coluna `onde` de proximas-etapas.csv",
    },
    "janela_1_liberada": JANELA_1_LIBERADA,
    "janela_1_pelo_criterio_literal_da_fase_5": JANELA_1_PELO_CRITERIO_DA_FASE_5,
    "regra_declarada_para_a_janela_1": "abre quando o caminho de produção reproduz a "
                                       "partição aprovada e nenhum item aberto é de "
                                       "natureza `modelo`",
    "condicao_para_a_janela_2": "documentação publicada e Canvas reescritos sobre a cadeia "
                                "00–06, e os três campos do sistema comercial capturados",
    "condicao_para_a_janela_3": "src/model/personas.py, src/deployment/scoring.py e "
                                "src/deployment/monitoramento.py consumindo os artefatos "
                                "publicados aqui",
    "compromissos": len(COMPROMISSOS),
}

registro = pd.DataFrame(COMPROMISSOS).set_index("n")

# O vocabulário do bloco de abertura é contrato, não legenda: o manifesto não pode
# publicar uma ação que a tabela não declarou.
ACOES_NAO_DECLARADAS = sorted(set(registro.acao) - ACOES_DECLARADAS)
assert not ACOES_NAO_DECLARADAS, f"ações fora do vocabulário declarado: {ACOES_NAO_DECLARADAS}"

io.salvar_tabela(registro.reset_index(), SECAO, "registro-de-compromissos", indice=False)
metadados_implantacao["compromissos_por_acao"] = registro.acao.value_counts().to_dict()

# O manifesto inteiro fica em `data/processed/`, ao lado dos das outras cinco fases —
# é o artefato de auditoria. Em `models/` vai o **recorte operacional**: o que a carga
# e a aplicação conferem na hora de rodar. Gravar o mesmo dicionário nos dois lugares
# fazia parecer que existiam dois artefatos, quando existia um arquivo copiado.
CHAVES_OPERACIONAIS = (
    "origem", "versao_modelo", "versao_anterior_no_repositorio", "regra_de_versionamento",
    "modelo", "carimbo", "regra_de_atribuicao", "contrato_de_entrada", "catalogo",
    "pacote_da_aplicacao",
)
io.salvar_json(metadados_implantacao, config.DATA_PROCESSED / "manifesto_implantacao.json")
io.salvar_json(
    {chave: metadados_implantacao[chave] for chave in CHAVES_OPERACIONAIS},
    config.MODELS / "metadados_implantacao.json",
)

print("Pacote de implantação publicado:")
for arquivo in ("aplicacao-segmentacao.pkl", "metadados_implantacao.json",
                "catalogo_de_personas.json", "contrato_de_entrada.json",
                "linha_de_base.json"):
    caminho = config.MODELS / arquivo
    print(f"    models/{arquivo:<32} {caminho.stat().st_size:>7} bytes")
print(f"    {ARQUIVO_RELATORIO.relative_to(RAIZ).as_posix():<39} "
      f"{ARQUIVO_RELATORIO.stat().st_size:>7} bytes")
print(f"\n{len(registro)} compromissos registrados")
print(f"Janela 1 liberada: {metadados_implantacao['janela_1_liberada']}")

In [ ]:
registro

# Próximo passo

A fase 6 termina com a segmentação **implantável** — e a distinção em relação a "implantada" é
o que este notebook não pode fechar sozinho: a janela 1 depende de uma autorização que é da
Diretoria, e a janela 2 depende de páginas que ainda descrevem outro experimento.

O que fica estabelecido, e que a operação pode citar sem remedir:

* **o que a aplicação carrega passa a ser verificável.** O pacote tem versão e carimbo, e a
  carga é recusada quando o resumo não confere — o caminho que colocaria 280 imóveis na linha
  errada deixa de existir por construção, e não por atenção de quem opera;
* **as três linhas de produto têm ação, teto, canal e métrica.** O catálogo é artefato de
  dados, carrega a versão do modelo a que pertence e tem exatamente uma entrada por segmento.
  É metade do critério que a fase 5 escreveu para fechar o item — a outra metade,
  `src/model/personas.py` travado por teste, continua aberta e está registrada como tal;
* **o modelo diz onde não sabe.** A faixa de fronteira é calibrada sobre a carteira, cabe no
  teto operacional de revisão humana e encontra a maior parte dos imóveis de silhueta negativa
  que a fase 5 apontou;
* **o plano de monitoramento nasce com a primeira leitura feita.** Os seis indicadores foram
  medidos entre 2006–2008 e 2009–2010 — o período do choque imobiliário — e nenhum gatilho
  disparou; o único com margem estreita é a faixa de fronteira, calibrada de propósito logo
  abaixo do teto operacional. A estrutura da carteira sobreviveu ao ciclo porque a segmentação
  é de **produto**, e não de preço;
* **a aplicação recebe um arquivo, e não uma árvore de diretórios.**
  `models/aplicacao-segmentacao.pkl` leva modelo, catálogo, contrato, cortes, linha de base,
  carteira e evidências, sem referência a `__main__` nem ao pacote `src` — a conferência lê o
  fluxo de bytes do arquivo gravado e a releitura reclassifica a carteira inteira sem
  divergência;
* **o que não foi entregue está escrito**, com dono, endereço e prazo: nenhum dos oito itens do
  backlog da fase 5 fecha pelo critério literal dela, porque três deles pediam mudança em
  módulos de `src/` que continuam servindo à cadeia anterior. A reconciliação mede isso contra
  a coluna `onde` do contrato, e não contra a lembrança do que esta fase fez.

## O que volta para o código

Três mudanças, todas medidas neste notebook e nenhuma delas improviso de execução. As três são
os itens 1, 4 e 8 do backlog da fase 5, e é por causa delas que a janela 3 não abre:

* **`src/deployment/scoring.py` aponta para o artefato da cadeia anterior** e procura passos
  que o pacote aprovado não tem. Precisa ser reapontado para `modelo-segmentacao.joblib`,
  passar a ler o catálogo e o contrato publicados aqui e devolver a razão de fronteira na
  resposta;
* **`src/model/personas.py` continua com quatro personas.** O catálogo canônico agora é
  `models/catalogo_de_personas.json`; o módulo deve lê-lo em vez de declarar constante, e um
  teste precisa comparar o número de entradas ao *k* do manifesto;
* **`src/pipeline/implantacao.py` e `src/deployment/monitoramento.py`** foram escritos para a
  cadeia anterior. O plano, os limiares e a linha de base desta fase estão publicados como
  artefato — os módulos passam a consumi-los, em vez de trazerem os números no corpo do
  código.

A ordem importa: enquanto os artefatos de *k* = 4 estiverem em `models/`, existe um caminho
pelo qual a operação carrega o modelo errado. **Aposentá-los é a primeira tarefa da janela 1**,
e o custo de não fazê-lo está medido: 280 imóveis com teto de reforma, política de preço e
canal trocados.